In [70]:
from pathlib import Path
import json
import os
import random
import sys
import time
import wave
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("/workspace/E-DAIC/original_data")
SPLIT_DIR = Path("/workspace/E-DAIC/splits")
LABEL_DIR = Path("/workspace/E-DAIC/others")

TRAIN_SPLIT_PATH = SPLIT_DIR / "train_split.csv"
VAL_SPLIT_PATH = SPLIT_DIR / "dev_split.csv"
TEST_SPLIT_PATH = SPLIT_DIR / "test_split.csv"
DETAILED_LABEL_PATH = LABEL_DIR / "Detailed_PHQ8_Labels.csv"

AUDIO_CACHE_ROOT = Path("/workspace/E-DAIC/cache/phq8_audio_egemaps")
AUDIO_CACHE_VERSION = "egemaps23_turnmean_100hz_turns120_v1"
AUDIO_CACHE_DIR = AUDIO_CACHE_ROOT / AUDIO_CACHE_VERSION
CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_audio_paperlike")

PHQ8_QUESTION_COLUMNS = ["PHQ_8NoInterest", "PHQ_8Depressed", "PHQ_8Sleep", "PHQ_8Tired", "PHQ_8Appetite", "PHQ_8Failure", "PHQ_8Concentrating", "PHQ_8Moving"]

FEATURE_RATE = 100
AUDIO_FEATURE_DIM = 23
MAX_TURNS = 120
LSTM_HIDDEN_SIZE = 50
ATTENTION_HEADS = 4
NUM_CLASSES = 4

BATCH_SIZE = 10
NUM_EPOCHS = 50
LEARNING_RATE = 5e-4
ADAM_EPSILON = 1e-8
WEIGHT_DECAY = 1e-3
MAX_GRAD_NORM = 1.0

ALPHA = 1.0
BETA = 0.5
LOSS_EPSILON = 1e-12
SEEDS = [42, 100, 1234]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

AUDIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("Python：", sys.version.split()[0])
print("PyTorch：", torch.__version__)
print("CUDA可用：", torch.cuda.is_available())
print("计算设备：", device)
print("语音特征频率：", FEATURE_RATE)
print("语音特征维度：", AUDIO_FEATURE_DIM)
print("最大对话轮数：", MAX_TURNS)
print("训练轮数：", NUM_EPOCHS)
print("随机种子：", SEEDS)
print("语音缓存目录：", AUDIO_CACHE_DIR)
print("Checkpoint目录：", CHECKPOINT_ROOT)

Python： 3.12.3
PyTorch： 2.9.0
CUDA可用： True
计算设备： cuda:0
语音特征频率： 100
语音特征维度： 23
最大对话轮数： 120
训练轮数： 50
随机种子： [42, 100, 1234]
语音缓存目录： /workspace/E-DAIC/cache/phq8_audio_egemaps/egemaps23_turnmean_100hz_turns120_v1
Checkpoint目录： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike


In [37]:
train_split_df=pd.read_csv(TRAIN_SPLIT_PATH)
test_split_df=pd.read_csv(TEST_SPLIT_PATH)
val_split_df=pd.read_csv(VAL_SPLIT_PATH)
detailed_label_df=pd.read_csv(DETAILED_LABEL_PATH)

train_metadata_df=train_split_df.merge(detailed_label_df[["Participant_ID"]+PHQ8_QUESTION_COLUMNS], on="Participant_ID",how="left",validate="one_to_one")
val_metadata_df=val_split_df.merge(detailed_label_df[["Participant_ID"]+PHQ8_QUESTION_COLUMNS], on="Participant_ID",how="left",validate="one_to_one")
test_metadata_df=test_split_df.copy()

if train_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise ValueError("训练集存在缺失的PHQ-8题目标签")

if val_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise ValueError("验证集存在缺失的PHQ-8题目标签")
train_participant_ids = set(train_metadata_df["Participant_ID"].astype(int).tolist())
val_participant_ids = set(val_metadata_df["Participant_ID"].astype(int).tolist())
test_participant_ids = set(test_metadata_df["Participant_ID"].astype(int).tolist())
if train_participant_ids & val_participant_ids:
    raise ValueError("训练集与验证集存在参与者重叠")

if train_participant_ids & test_participant_ids:
    raise ValueError("训练集与测试集存在参与者重叠")

if val_participant_ids & test_participant_ids:
    raise ValueError("验证集与测试集存在参与者重叠")

print("训练参与者：", len(train_metadata_df))
print("验证参与者：", len(val_metadata_df))
print("测试参与者：", len(test_metadata_df))
print("训练标签形状：", train_metadata_df[PHQ8_QUESTION_COLUMNS].shape)
print("验证标签形状：", val_metadata_df[PHQ8_QUESTION_COLUMNS].shape)
print("全部题目标签范围：", sorted(pd.unique(train_metadata_df[PHQ8_QUESTION_COLUMNS].to_numpy().reshape(-1)).tolist()))
print("训练、验证、测试参与者没有重叠")

训练参与者： 163
验证参与者： 55
测试参与者： 56
训练标签形状： (163, 8)
验证标签形状： (55, 8)
全部题目标签范围： [0, 1, 2, 3]
训练、验证、测试参与者没有重叠


In [51]:
all_participants_ids=sorted(train_participant_ids|val_participant_ids|test_participant_ids)
missing_wav_ids=[]
missing_transcript_ids=[]
missing_egemaps_ids=[]

for participant_id in all_participants_ids:
    participant_dir=DATA_DIR/f"{participant_id}_P"
    wav_path=participant_dir/f"{participant_id}_AUDIO.wav"
    transcript_path=participant_dir/f"{participant_id}_Transcript.csv"
    egemaps_path=participant_dir/"features"/f"{participant_id}_OpenSMILE2.3.0_egemaps.csv"

    if not wav_path.exists():
        missing_wav_ids.append(participant_id)

    if not transcript_path.exists():
        missing_transcript_ids.append(participant_id)

    if not egemaps_path.exists():
        missing_egemaps_ids.append(participant_id)

print("全部参与者数量：", len(all_participants_ids))
print("缺失WAV数量：", len(missing_wav_ids))
print("缺失转录数量：", len(missing_transcript_ids))
print("缺失eGeMAPS数量：", len(missing_egemaps_ids))
print("缺失WAV参与者：", missing_wav_ids)
print("缺失转录参与者：", missing_transcript_ids)
print("缺失eGeMAPS参与者：", missing_egemaps_ids)

if missing_wav_ids or missing_transcript_ids or missing_egemaps_ids:
    raise FileNotFoundError("语音正式输入文件不完整")

print("全部语音输入文件检查通过")

全部参与者数量： 274
缺失WAV数量： 0
缺失转录数量： 0
缺失eGeMAPS数量： 0
缺失WAV参与者： []
缺失转录参与者： []
缺失eGeMAPS参与者： []
全部语音输入文件检查通过


In [76]:
def get_audio_input_paths(participant_id):
    participant_id=int(participant_id)
    participant_dir=DATA_DIR/f"{participant_id}_P"
    wav_path=participant_dir/f"{participant_id}_AUDIO.wav"
    transcript_path=participant_dir/f"{participant_id}_Transcript.csv"
    egemaps_path=participant_dir/"features"/f"{participant_id}_OpenSMILE2.3.0_egemaps.csv"
    return wav_path,transcript_path,egemaps_path

def get_file_signature(file_path):
    file_stat=file_path.stat()
    signature={}
    signature["size"]=file_stat.st_size
    signature["mtime_ns"]=file_stat.st_mtime_ns
    return signature

def clean_audio_turn_boundaries(transcript_df,audio_duration_seconds,feature_frame_count):
    start_times=(transcript_df["Start_Time"].to_numpy(dtype=np.float64)*FEATURE_RATE).tolist()
    end_times=(transcript_df["End_Time"].to_numpy(dtype=np.float64)*FEATURE_RATE).tolist()
    maximum_valid_frame=min(audio_duration_seconds*FEATURE_RATE,feature_frame_count)
    cleaned_start_times=[]
    cleaned_end_times=[]
    previous_start_time=-float("inf")
    previous_end_time=-float("inf")

    for start_time,end_time in zip(start_times,end_times):
        if not np.isfinite(start_time) or not np.isfinite(end_time):
            continue
        if start_time < previous_start_time or end_time < previous_end_time:
            continue
        if start_time < 0 or end_time > maximum_valid_frame:
            continue

        rounded_start_time=round(start_time)
        rounded_end_time=round(end_time)

        if rounded_start_time>=rounded_end_time:
            continue
        cleaned_start_times.append(rounded_start_time)
        cleaned_end_times.append(rounded_end_time)
        previous_start_time=rounded_start_time
        previous_end_time=rounded_end_time
    if len(cleaned_start_times)==0:
        raise ValueError("清理时间边界后没有有效对话伦茨")
    return cleaned_start_times,cleaned_end_times

def get_wav_duration_seconds(wav_path):
    with wave.open(str(wav_path),"rb") as wav_file:

        frame_count = wav_file.getnframes()
        sample_rate = wav_file.getframerate()

    if sample_rate <= 0:
        raise ValueError(f"WAV采样率不正确：{wav_path}")

    duration_seconds = frame_count / sample_rate
    return float(duration_seconds)
def compute_audio_features(pparticipant_id):
    participant_id=int(pparticipant_id)
    wav_path,transcript_path,egemaps_path=get_audio_input_paths(participant_id)
    if not wav_path.exists():
        raise FileNotFoundError(f"WAV文件不存在：{wav_path}")

    if not transcript_path.exists():
        raise FileNotFoundError(f"转录文件不存在：{transcript_path}")

    if not egemaps_path.exists():
        raise FileNotFoundError(f"eGeMAPS文件不存在：{egemaps_path}")

    transcript_df=pd.read_csv(transcript_path)
    egemaps_df=pd.read_csv(egemaps_path,sep=";")
    required_transcript_columns={"Start_Time","End_Time"}
    if not required_transcript_columns.issubset(transcript_df.columns):
        raise ValueError(f"参与者{participant_id}的转录文件缺少时间列")

    if egemaps_df.shape[1] < 3:
        raise ValueError(f"参与者{participant_id}的eGeMAPS列数不足")

    audio_feature_values=egemaps_df.iloc[:,2:].to_numpy(dtype=np.float32)
    if audio_feature_values.shape[1] != AUDIO_FEATURE_DIM:
        raise ValueError(f"参与者{participant_id}的语音特征维度为{audio_feature_values.shape[1]}，预期为{AUDIO_FEATURE_DIM}")

    if not np.isfinite(audio_feature_values).all():
        raise ValueError(f"参与者{participant_id}的语音特征包含NaN或Inf")

    audio_duration_seconds=get_wav_duration_seconds(wav_path)
    start_times,end_times=clean_audio_turn_boundaries(transcript_df,audio_duration_seconds,audio_feature_values.shape[0])
    audio_feature_tensor=torch.from_numpy(audio_feature_values)
    turn_vector_list=[]

    for start_time,end_time in zip(start_times,end_times):
        turn_frames=audio_feature_tensor[start_time:end_time]
        if turn_frames.shape[0]==0:
            continue
        turn_vector=turn_frames.mean(dim=0)
        turn_vector_list.append(turn_vector)
    if len(turn_vector_list)==0:
        raise ValueError(f"参与者{participant_id}没有可用语音轮次")

    turn_vectors=torch.stack(turn_vector_list, dim=0)
    real_turn_count=min(turn_vectors.shape[0], MAX_TURNS)
    fixed_audio_features=torch.zeros((MAX_TURNS,AUDIO_FEATURE_DIM),dtype=torch.float32)
    padding_mask=torch.ones(MAX_TURNS,dtype=torch.bool)
    fixed_audio_features[:real_turn_count]=turn_vectors[:real_turn_count]
    padding_mask[:real_turn_count]=False

    if fixed_audio_features.shape != (MAX_TURNS, AUDIO_FEATURE_DIM):
        raise RuntimeError("固定长度语音特征形状错误")

    if padding_mask.shape != (MAX_TURNS,):
        raise RuntimeError("语音Padding Mask形状错误")

    if not torch.isfinite(fixed_audio_features).all():
        raise RuntimeError(f"参与者{participant_id}处理后的语音特征包含NaN或Inf")

    metadata={}
    metadata["participant_id"] = participant_id
    metadata["cache_version"] = AUDIO_CACHE_VERSION
    metadata["feature_rate"] = FEATURE_RATE
    metadata["feature_dim"] = AUDIO_FEATURE_DIM
    metadata["max_turns"] = MAX_TURNS
    metadata["original_transcript_turns"] = len(transcript_df)
    metadata["cleaned_turns"] = len(turn_vector_list)
    metadata["real_turn_count"] = real_turn_count
    metadata["audio_duration_seconds"] = audio_duration_seconds
    metadata["wav_signature"] = get_file_signature(wav_path)
    metadata["transcript_signature"] = get_file_signature(transcript_path)
    metadata["egemaps_signature"] = get_file_signature(egemaps_path)

    return fixed_audio_features, padding_mask, real_turn_count, metadata

def get_audio_cache_path(participant_id):
    participant_id=int(participant_id)
    return AUDIO_CACHE_DIR/f"{participant_id}.pt"

def audio_cache_is_valid(cache_record,participant_id):
    participant_id=int(participant_id)
    wav_path,transcript_path,egemaps_path=get_audio_input_paths(participant_id)

    if cache_record.get("participant_id") != participant_id:
        return False

    if cache_record.get("cache_version") != AUDIO_CACHE_VERSION:
        return False

    if cache_record.get("feature_rate") != FEATURE_RATE:
        return False

    if cache_record.get("feature_dim") != AUDIO_FEATURE_DIM:
        return False

    if cache_record.get("max_turns") != MAX_TURNS:
        return False

    if cache_record.get("wav_signature") != get_file_signature(wav_path):
        return False

    if cache_record.get("transcript_signature") != get_file_signature(transcript_path):
        return False

    if cache_record.get("egemaps_signature") != get_file_signature(egemaps_path):
        return False

    if cache_record["audio_features"].shape != (MAX_TURNS, AUDIO_FEATURE_DIM):
        return False

    if cache_record["padding_mask"].shape != (MAX_TURNS,):
        return False

    return True

def get_participant_audio_features(participant_id,force_recompute=False):
    participant_id=int(participant_id)
    cache_path=get_audio_cache_path(participant_id)

    if cache_path.exists() and not force_recompute:
        cache_record=torch.load(cache_path,map_location="cpu",weights_only=False)

        if audio_cache_is_valid(cache_record,participant_id):
            return cache_record["audio_features"],cache_record["padding_mask"],cache_record["real_turn_count"],True

    audio_features,padding_mask,real_turn_count,metadata=compute_audio_features(participant_id)
    cache_record=dict(metadata)
    cache_record["audio_features"]=audio_features
    cache_record["padding_mask"]=padding_mask
    temporary_cache_path=cache_path.with_suffix(".tmp")
    torch.save(cache_record,temporary_cache_path)
    temporary_cache_path.replace(cache_path)

    return audio_features,padding_mask,real_turn_count,False

In [77]:
audio_cache_rows=[]
audio_cache_hit_count=0
audio_cache_miss_count=0

for index,participant_id in enumerate(all_participants_ids):
    audio_features,padding_mask,real_turn_count,cache_hit=get_participant_audio_features(participant_id)
    if cache_hit:
        audio_cache_hit_count += 1
    else:
        audio_cache_miss_count += 1

    row = {}
    row["Participant_ID"] = participant_id
    row["Real_Turn_Count"] = real_turn_count
    row["Padding_Turn_Count"] = int(padding_mask.sum().item())
    row["Cache_Hit"] = cache_hit
    audio_cache_rows.append(row)
    audio_cache_summary_df = pd.DataFrame(audio_cache_rows)

audio_cache_summary_df = pd.DataFrame(audio_cache_rows)
print("全部参与者数量：", len(audio_cache_summary_df))
print("缓存命中数量：", audio_cache_hit_count)
print("新建缓存数量：", audio_cache_miss_count)
print("最少真实轮数：", audio_cache_summary_df["Real_Turn_Count"].min())
print("最多真实轮数：", audio_cache_summary_df["Real_Turn_Count"].max())
print("缓存文件数量：", len(list(AUDIO_CACHE_DIR.glob("*.pt"))))

display(audio_cache_summary_df.head())

全部参与者数量： 274
缓存命中数量： 274
新建缓存数量： 0
最少真实轮数： 42
最多真实轮数： 120
缓存文件数量： 274


,Participant_ID,Real_Turn_Count,Padding_Turn_Count,Cache_Hit
0,300,76,44,True
1,301,71,49,True
2,302,98,22,True
3,303,93,27,True
4,304,79,41,True


In [78]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [83]:
class CachedAudioPHQ8Dataset(Dataset):
    def __init__(self,metadata_df, split_name):
        super().__init__()
        if split_name not in {"train","val"}:
            raise ValueError("CachedAudioPHQ8Dataset只支持train或val")

        self.split_name=split_name
        self.metadata_df=metadata_df.reset_index(drop=True).copy()
        self.participant_ids=self.metadata_df["Participant_ID"].astype(int).tolist()
        self.labels=torch.tensor(self.metadata_df[PHQ8_QUESTION_COLUMNS].to_numpy(),dtype=torch.long)
        audio_feature_list=[]
        padding_mask_list=[]
        real_turn_count_list=[]
        cache_hit_count=0

        for participant_id in self.participant_ids:
            audio_features,padding_mask,real_turn_count,cache_hit=get_participant_audio_features(participant_id)
            audio_feature_list.append(audio_features)
            padding_mask_list.append(padding_mask)
            real_turn_count_list.append(real_turn_count)

            if cache_hit:
                cache_hit_count+=1

        self.audio_features=torch.stack(audio_feature_list,dim=0)
        self.padding_masks=torch.stack(padding_mask_list,dim=0)
        self.real_turn_counts=torch.tensor(real_turn_count_list,dtype=torch.long)
        self.cache_hit_count=cache_hit_count
        self.cache_miss_count=len(self.participant_ids)-cache_hit_count

        if self.audio_features.shape != (len(self.participant_ids), MAX_TURNS, AUDIO_FEATURE_DIM):
            raise RuntimeError(f"{split_name}语音特征整体形状错误：{self.audio_features.shape}")

        if self.padding_masks.shape != (len(self.participant_ids), MAX_TURNS):
            raise RuntimeError(f"{split_name} Mask整体形状错误：{self.padding_masks.shape}")

        if self.labels.shape != (len(self.participant_ids), len(PHQ8_QUESTION_COLUMNS)):
            raise RuntimeError(f"{split_name}标签整体形状错误：{self.labels.shape}")

        if not torch.isfinite(self.audio_features).all():
            raise RuntimeError(f"{split_name}语音特征存在NaN或Inf")

    def __len__(self):
        return len(self.participant_ids)
    def __getitem__(self, index):
        audio_features=self.audio_features[index]
        padding_mask=self.padding_masks[index]
        labels=self.labels[index]
        return audio_features,padding_mask,labels


In [84]:
train_audio_dataset = CachedAudioPHQ8Dataset(train_metadata_df, split_name="train")
val_audio_dataset = CachedAudioPHQ8Dataset(val_metadata_df, split_name="val")

print("训练样本数：", len(train_audio_dataset))
print("验证样本数：", len(val_audio_dataset))
print("训练语音特征：", train_audio_dataset.audio_features.shape)
print("训练Mask：", train_audio_dataset.padding_masks.shape)
print("训练全部标签：", train_audio_dataset.labels.shape)
print("验证语音特征：", val_audio_dataset.audio_features.shape)
print("验证Mask：", val_audio_dataset.padding_masks.shape)
print("验证全部标签：", val_audio_dataset.labels.shape)
print("训练缓存命中：", train_audio_dataset.cache_hit_count)
print("验证缓存命中：", val_audio_dataset.cache_hit_count)

训练样本数： 163
验证样本数： 55
训练语音特征： torch.Size([163, 120, 23])
训练Mask： torch.Size([163, 120])
训练全部标签： torch.Size([163, 8])
验证语音特征： torch.Size([55, 120, 23])
验证Mask： torch.Size([55, 120])
验证全部标签： torch.Size([55, 8])
训练缓存命中： 163
验证缓存命中： 55


In [85]:
def create_audio_dataloaders(seed):
    train_generator = torch.Generator()
    train_generator.manual_seed(seed)
    train_loader = DataLoader(train_audio_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False, generator=train_generator)
    val_loader = DataLoader(val_audio_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    return train_loader, val_loader

In [86]:
set_seed(42)
train_audio_loader, val_audio_loader = create_audio_dataloaders(seed=42)
batch_audio_features, batch_audio_masks, batch_all_labels = next(iter(train_audio_loader))

print("训练批次数：", len(train_audio_loader))
print("验证批次数：", len(val_audio_loader))
print("Batch语音特征：", batch_audio_features.shape)
print("Batch Mask：", batch_audio_masks.shape)
print("Batch全部标签：", batch_all_labels.shape)
print("当前Batch的Q1标签：", batch_all_labels[:, 0])
print("语音特征类型：", batch_audio_features.dtype)
print("Mask类型：", batch_audio_masks.dtype)
print("标签类型：", batch_all_labels.dtype)

训练批次数： 17
验证批次数： 6
Batch语音特征： torch.Size([10, 120, 23])
Batch Mask： torch.Size([10, 120])
Batch全部标签： torch.Size([10, 8])
当前Batch的Q1标签： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1])
语音特征类型： torch.float32
Mask类型： torch.bool
标签类型： torch.int64


In [88]:
class AudioLSTMAttentionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=AUDIO_FEATURE_DIM, hidden_size=LSTM_HIDDEN_SIZE, batch_first=True, bidirectional=True)
        self.attention1 = nn.MultiheadAttention(embed_dim=LSTM_HIDDEN_SIZE * 2, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.attention2 = nn.MultiheadAttention(embed_dim=LSTM_HIDDEN_SIZE * 2, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(MAX_TURNS * LSTM_HIDDEN_SIZE * 2, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, NUM_CLASSES))

    def encode(self, audio_features, padding_mask):
        lstm_output, _ = self.lstm(audio_features)
        attention1_output, _ = self.attention1(lstm_output, lstm_output, lstm_output, key_padding_mask=padding_mask, need_weights=False)
        attention2_output, _ = self.attention2(attention1_output, attention1_output, attention1_output, key_padding_mask=padding_mask, need_weights=False)
        return attention2_output

    def forward(self, audio_features, padding_mask):
        encoded_audio = self.encode(audio_features, padding_mask)
        logits = self.mlp(encoded_audio)
        return logits

In [89]:
set_seed(42)
audio_model = AudioLSTMAttentionClassifier().to(device)

audio_lstm_parameters = sum(parameter.numel() for parameter in audio_model.lstm.parameters() if parameter.requires_grad)
audio_attention1_parameters = sum(parameter.numel() for parameter in audio_model.attention1.parameters() if parameter.requires_grad)
audio_attention2_parameters = sum(parameter.numel() for parameter in audio_model.attention2.parameters() if parameter.requires_grad)
audio_mlp_parameters = sum(parameter.numel() for parameter in audio_model.mlp.parameters() if parameter.requires_grad)
audio_total_parameters = sum(parameter.numel() for parameter in audio_model.parameters() if parameter.requires_grad)

print("模型设备：", next(audio_model.parameters()).device)
print("LSTM参数：", f"{audio_lstm_parameters:,}")
print("Attention1参数：", f"{audio_attention1_parameters:,}")
print("Attention2参数：", f"{audio_attention2_parameters:,}")
print("MLP参数：", f"{audio_mlp_parameters:,}")
print("总可训练参数：", f"{audio_total_parameters:,}")

模型设备： cuda:0
LSTM参数： 30,000
Attention1参数： 40,400
Attention2参数： 40,400
MLP参数： 3,073,284
总可训练参数： 3,184,084


In [90]:
batch_audio_features_on_device = batch_audio_features.to(device)
batch_audio_masks_on_device = batch_audio_masks.to(device)

audio_model.eval()

with torch.no_grad():
    batch_encoded_audio = audio_model.encode(batch_audio_features_on_device, batch_audio_masks_on_device)
    batch_audio_logits = audio_model(batch_audio_features_on_device, batch_audio_masks_on_device)
    batch_audio_probabilities = torch.softmax(batch_audio_logits, dim=1)
    batch_audio_predictions = torch.argmax(batch_audio_logits, dim=1)

print("模型输入形状：", batch_audio_features_on_device.shape)
print("Audio Encoder输出：", batch_encoded_audio.shape)
print("模型Logits形状：", batch_audio_logits.shape)
print("第一个样本Logits：", batch_audio_logits[0])
print("第一个样本概率：", batch_audio_probabilities[0])
print("概率之和：", batch_audio_probabilities[0].sum().item())
print("当前Batch预测：", batch_audio_predictions)
print("当前Batch真实Q1标签：", batch_all_labels[:, 0])

模型输入形状： torch.Size([10, 120, 23])
Audio Encoder输出： torch.Size([10, 120, 100])
模型Logits形状： torch.Size([10, 4])
第一个样本Logits： tensor([0.0470, 0.0119, 0.0338, 0.0695], device='cuda:0')
第一个样本概率： tensor([0.2516, 0.2429, 0.2483, 0.2573], device='cuda:0')
概率之和： 1.0
当前Batch预测： tensor([3, 3, 3, 3, 3, 3, 3, 3, 3, 3], device='cuda:0')
当前Batch真实Q1标签： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1])


In [91]:
def compute_audio_question_class_weights(labels_matrix, question_index, beta):
    question_labels = labels_matrix[:, question_index]
    class_counts = torch.bincount(question_labels, minlength=NUM_CLASSES).float()

    if torch.any(class_counts == 0):
        raise ValueError(f"第{question_index + 1}题存在训练样本数量为0的类别")

    total_count = class_counts.sum()
    class_weights = (total_count / class_counts).pow(beta)
    return class_counts, class_weights

In [92]:
audio_question_class_counts = []
audio_question_class_weights = []

for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
    class_counts, class_weights = compute_audio_question_class_weights(train_audio_dataset.labels, question_index, BETA)
    audio_question_class_counts.append(class_counts)
    audio_question_class_weights.append(class_weights)

audio_question_class_counts = torch.stack(audio_question_class_counts, dim=0)
audio_question_class_weights = torch.stack(audio_question_class_weights, dim=0)

print("类别数量形状：", audio_question_class_counts.shape)
print("类别权重形状：", audio_question_class_weights.shape)
print("Q1类别数量：", audio_question_class_counts[0])
print("Q1类别权重：", audio_question_class_weights[0])

类别数量形状： torch.Size([8, 4])
类别权重形状： torch.Size([8, 4])
Q1类别数量： tensor([81., 57., 20.,  5.])
Q1类别权重： tensor([1.4186, 1.6910, 2.8548, 5.7096])


In [93]:
def imbalanced_ordinal_loss(logits, labels, class_weights, alpha=1.0, epsilon=1e-12):
    probabilities = torch.softmax(logits, dim=1)
    class_ids = torch.arange(NUM_CLASSES, device=logits.device)
    distances = torch.abs(labels.unsqueeze(1) - class_ids.unsqueeze(0)).float()
    true_class_weights = class_weights[labels].unsqueeze(1)
    weighted_distances = distances * true_class_weights
    probability_penalties = -torch.log(torch.clamp(1.0 - probabilities, min=epsilon))
    sample_losses = torch.sum(probability_penalties * weighted_distances.pow(alpha), dim=1)
    loss = sample_losses.mean()
    return loss

In [94]:
set_seed(42)
audio_step_model = AudioLSTMAttentionClassifier().to(device)
audio_step_optimizer = torch.optim.AdamW(audio_step_model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)

q1_labels = batch_all_labels[:, 0].to(device)
q1_class_weights = audio_question_class_weights[0].to(device)
batch_audio_features_on_device = batch_audio_features.to(device)
batch_audio_masks_on_device = batch_audio_masks.to(device)

audio_step_model.train()
audio_step_optimizer.zero_grad(set_to_none=True)

q1_logits = audio_step_model(batch_audio_features_on_device, batch_audio_masks_on_device)
q1_loss = imbalanced_ordinal_loss(q1_logits, q1_labels, q1_class_weights, alpha=ALPHA, epsilon=LOSS_EPSILON)

final_linear_weight_before = audio_step_model.mlp[-1].weight.detach().clone()

q1_loss.backward()
gradient_norm_before_clipping = torch.nn.utils.clip_grad_norm_(audio_step_model.parameters(), MAX_GRAD_NORM)
audio_step_optimizer.step()

final_linear_weight_after = audio_step_model.mlp[-1].weight.detach().clone()
weight_change = torch.abs(final_linear_weight_after - final_linear_weight_before)
q1_predictions = torch.argmax(q1_logits.detach(), dim=1)
q1_accuracy = (q1_predictions == q1_labels).float().mean()

print("Logits形状：", q1_logits.shape)
print("Q1标签形状：", q1_labels.shape)
print("Q1类别权重：", q1_class_weights)
print("ImbOLL形状：", q1_loss.shape)
print("ImbOLL数值：", q1_loss.item())
print("损失是否有限：", torch.isfinite(q1_loss).item())
print("裁剪前梯度长度：", gradient_norm_before_clipping.item())
print("最终Linear权重平均变化：", weight_change.mean().item())
print("最终Linear权重最大变化：", weight_change.max().item())
print("当前Batch预测：", q1_predictions)
print("当前Batch真实标签：", q1_labels)
print("当前Batch Accuracy：", q1_accuracy.item())

Logits形状： torch.Size([10, 4])
Q1标签形状： torch.Size([10])
Q1类别权重： tensor([1.4186, 1.6910, 2.8548, 5.7096], device='cuda:0')
ImbOLL形状： torch.Size([])
ImbOLL数值： 2.327353000640869
损失是否有限： True
裁剪前梯度长度： 4.088662147521973
最终Linear权重平均变化： 0.00035151789779774845
最终Linear权重最大变化： 0.0005000308156013489
当前Batch预测： tensor([3, 0, 3, 3, 3, 3, 3, 3, 3, 3], device='cuda:0')
当前Batch真实标签： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1], device='cuda:0')
当前Batch Accuracy： 0.10000000149011612


In [95]:
def compute_ccc(predictions, labels, epsilon=1e-12):
    predictions = predictions.float()
    labels = labels.float()
    prediction_mean = predictions.mean()
    label_mean = labels.mean()
    prediction_variance = predictions.var(correction=1)
    label_variance = labels.var(correction=1)
    centered_predictions = predictions - prediction_mean
    centered_labels = labels - label_mean
    prediction_length = torch.sqrt(torch.sum(centered_predictions ** 2))
    label_length = torch.sqrt(torch.sum(centered_labels ** 2))

    if prediction_length.item() < epsilon or label_length.item() < epsilon:
        return 0.0

    correlation = torch.sum(centered_predictions * centered_labels) / (prediction_length * label_length)
    numerator = 2.0 * correlation * torch.sqrt(prediction_variance) * torch.sqrt(label_variance)
    denominator = prediction_variance + label_variance + (prediction_mean - label_mean) ** 2
    ccc = numerator / torch.clamp(denominator, min=epsilon)
    return float(ccc.item())


def compute_audio_metrics(predictions, labels):
    predictions = predictions.cpu().long()
    labels = labels.cpu().long()
    confusion_matrix = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

    for true_label, predicted_label in zip(labels, predictions):
        confusion_matrix[true_label, predicted_label] += 1

    true_positive = confusion_matrix.diag().float()
    predicted_count = confusion_matrix.sum(dim=0).float()
    actual_count = confusion_matrix.sum(dim=1).float()
    precision = true_positive / torch.clamp(predicted_count, min=1.0)
    recall = true_positive / torch.clamp(actual_count, min=1.0)
    per_class_f1 = 2.0 * precision * recall / torch.clamp(precision + recall, min=1e-12)
    accuracy = (predictions == labels).float().mean().item()
    macro_f1 = per_class_f1.mean().item()
    weighted_f1 = (per_class_f1 * actual_count).sum().item() / actual_count.sum().item()
    ccc = compute_ccc(predictions, labels)
    rmse = torch.sqrt(torch.mean((predictions.float() - labels.float()) ** 2)).item()
    mae = torch.mean(torch.abs(predictions.float() - labels.float())).item()

    metrics = {}
    metrics["accuracy"] = accuracy
    metrics["micro_f1"] = accuracy
    metrics["macro_f1"] = macro_f1
    metrics["weighted_f1"] = weighted_f1
    metrics["ccc"] = ccc
    metrics["rmse"] = rmse
    metrics["mae"] = mae
    metrics["per_class_f1"] = per_class_f1
    metrics["confusion_matrix"] = confusion_matrix
    return metrics

In [96]:
def train_audio_one_epoch(model, dataloader, optimizer, question_index, class_weights, device, max_grad_norm):
    model.train()
    total_loss = 0.0
    total_sample_count = 0
    prediction_list = []
    label_list = []
    gradient_norm_list = []

    for batch_audio_features, batch_padding_masks, batch_all_labels in dataloader:
        batch_audio_features = batch_audio_features.to(device)
        batch_padding_masks = batch_padding_masks.to(device)
        batch_labels = batch_all_labels[:, question_index].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_audio_features, batch_padding_masks)
        loss = imbalanced_ordinal_loss(logits, batch_labels, class_weights, alpha=ALPHA, epsilon=LOSS_EPSILON)

        if not torch.isfinite(loss):
            raise RuntimeError(f"训练损失出现NaN或Inf，第{question_index + 1}题")

        loss.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        predictions = torch.argmax(logits.detach(), dim=1)
        batch_size = batch_labels.shape[0]
        total_loss += loss.item() * batch_size
        total_sample_count += batch_size
        prediction_list.append(predictions.cpu())
        label_list.append(batch_labels.cpu())
        gradient_norm_list.append(float(gradient_norm.item()))

    all_predictions = torch.cat(prediction_list, dim=0)
    all_labels = torch.cat(label_list, dim=0)
    metrics = compute_audio_metrics(all_predictions, all_labels)
    metrics["loss"] = total_loss / total_sample_count
    metrics["mean_gradient_norm"] = float(np.mean(gradient_norm_list))
    return metrics

In [97]:
def evaluate_audio_question(model, dataloader, question_index, class_weights, device):
    model.eval()
    total_loss = 0.0
    total_sample_count = 0
    prediction_list = []
    label_list = []

    with torch.no_grad():
        for batch_audio_features, batch_padding_masks, batch_all_labels in dataloader:
            batch_audio_features = batch_audio_features.to(device)
            batch_padding_masks = batch_padding_masks.to(device)
            batch_labels = batch_all_labels[:, question_index].to(device)
            logits = model(batch_audio_features, batch_padding_masks)
            loss = imbalanced_ordinal_loss(logits, batch_labels, class_weights, alpha=ALPHA, epsilon=LOSS_EPSILON)

            if not torch.isfinite(loss):
                raise RuntimeError(f"验证损失出现NaN或Inf，第{question_index + 1}题")

            predictions = torch.argmax(logits, dim=1)
            batch_size = batch_labels.shape[0]
            total_loss += loss.item() * batch_size
            total_sample_count += batch_size
            prediction_list.append(predictions.cpu())
            label_list.append(batch_labels.cpu())

    all_predictions = torch.cat(prediction_list, dim=0)
    all_labels = torch.cat(label_list, dim=0)
    metrics = compute_audio_metrics(all_predictions, all_labels)
    metrics["loss"] = total_loss / total_sample_count
    metrics["predictions"] = all_predictions
    metrics["labels"] = all_labels
    return metrics

In [98]:
def print_audio_epoch_result(epoch, train_metrics, val_metrics):
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss={train_metrics['loss']:.6f} | Val Loss={val_metrics['loss']:.6f} | Val Acc={val_metrics['accuracy']:.4f} | Val MacroF1={val_metrics['macro_f1']:.4f} | Val CCC={val_metrics['ccc']:.4f}")

In [99]:
set_seed(42)
audio_function_test_model = AudioLSTMAttentionClassifier().to(device)
audio_function_test_optimizer = torch.optim.AdamW(audio_function_test_model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
function_test_train_loader, function_test_val_loader = create_audio_dataloaders(seed=42)
function_test_class_weights = audio_question_class_weights[0].to(device)

function_test_train_metrics = train_audio_one_epoch(audio_function_test_model, function_test_train_loader, audio_function_test_optimizer, question_index=0, class_weights=function_test_class_weights, device=device, max_grad_norm=MAX_GRAD_NORM)
function_test_val_metrics = evaluate_audio_question(audio_function_test_model, function_test_val_loader, question_index=0, class_weights=function_test_class_weights, device=device)

print_audio_epoch_result(1, function_test_train_metrics, function_test_val_metrics)
print("训练平均梯度长度：", function_test_train_metrics["mean_gradient_norm"])
print("验证每类F1：", function_test_val_metrics["per_class_f1"])
print("验证混淆矩阵：")
print(function_test_val_metrics["confusion_matrix"])

Epoch 01/50 | Train Loss=2.296797 | Val Loss=2.603723 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
训练平均梯度长度： 5.266138651791741
验证每类F1： tensor([0.0000, 0.5714, 0.0000, 0.0000])
验证混淆矩阵：
tensor([[ 0, 25,  0,  0],
        [ 0, 22,  0,  0],
        [ 0,  4,  0,  0],
        [ 0,  4,  0,  0]])


In [100]:
def get_audio_experiment_paths(question_index, seed):
    question_number = question_index + 1
    experiment_dir = CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_number}"
    experiment_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    paths["experiment_dir"] = experiment_dir
    paths["best_loss"] = experiment_dir / "best_loss.pt"
    paths["best_ccc"] = experiment_dir / "best_ccc.pt"
    paths["last"] = experiment_dir / "last.pt"
    paths["history"] = experiment_dir / "history.csv"
    return paths

In [101]:
def copy_state_dict_to_cpu(state_dict):
    return {name: tensor.detach().cpu() for name, tensor in state_dict.items()}


def metrics_for_checkpoint(metrics):
    saved_metrics = {}

    for key, value in metrics.items():
        if key in {"predictions", "labels"}:
            continue

        if isinstance(value, torch.Tensor):
            saved_metrics[key] = value.detach().cpu()
        else:
            saved_metrics[key] = value

    return saved_metrics


def save_audio_checkpoint(path, checkpoint_type, model, optimizer, epoch, question_index, seed, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, include_optimizer):
    checkpoint = {}
    checkpoint["checkpoint_type"] = checkpoint_type
    checkpoint["epoch"] = epoch
    checkpoint["question_index"] = question_index
    checkpoint["question_number"] = question_index + 1
    checkpoint["question"] = PHQ8_QUESTION_COLUMNS[question_index]
    checkpoint["seed"] = seed
    checkpoint["model_state_dict"] = copy_state_dict_to_cpu(model.state_dict())
    checkpoint["optimizer_state_dict"] = optimizer.state_dict() if include_optimizer else None
    checkpoint["train_metrics"] = metrics_for_checkpoint(train_metrics)
    checkpoint["val_metrics"] = metrics_for_checkpoint(val_metrics)
    checkpoint["class_counts"] = class_counts.detach().cpu()
    checkpoint["class_weights"] = class_weights.detach().cpu()
    checkpoint["best_val_loss"] = best_val_loss
    checkpoint["best_val_ccc"] = best_val_ccc
    checkpoint["best_loss_epoch"] = best_loss_epoch
    checkpoint["best_ccc_epoch"] = best_ccc_epoch
    checkpoint["config"] = {"audio_feature_dim": AUDIO_FEATURE_DIM, "max_turns": MAX_TURNS, "lstm_hidden_size": LSTM_HIDDEN_SIZE, "attention_heads": ATTENTION_HEADS, "num_classes": NUM_CLASSES, "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE, "adam_epsilon": ADAM_EPSILON, "weight_decay": WEIGHT_DECAY, "max_grad_norm": MAX_GRAD_NORM, "alpha": ALPHA, "beta": BETA, "audio_cache_version": AUDIO_CACHE_VERSION}
    torch.save(checkpoint, path)

In [102]:
def train_single_audio_question(question_index, seed, num_epochs=NUM_EPOCHS):
    set_seed(seed)
    paths = get_audio_experiment_paths(question_index, seed)
    train_loader, val_loader = create_audio_dataloaders(seed)
    model = AudioLSTMAttentionClassifier().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    class_counts = audio_question_class_counts[question_index].clone()
    class_weights_cpu = audio_question_class_weights[question_index].clone()
    class_weights = class_weights_cpu.to(device)
    best_val_loss = float("inf")
    best_val_ccc = -float("inf")
    best_loss_epoch = 0
    best_ccc_epoch = 0
    history_rows = []
    training_start_time = time.time()

    print("=" * 70)
    print(f"开始正式训练：Q{question_index + 1} {PHQ8_QUESTION_COLUMNS[question_index]}，Seed={seed}")
    print("=" * 70)

    for epoch in range(1, num_epochs + 1):
        epoch_start_time = time.time()
        train_metrics = train_audio_one_epoch(model, train_loader, optimizer, question_index, class_weights, device, MAX_GRAD_NORM)
        val_metrics = evaluate_audio_question(model, val_loader, question_index, class_weights, device)
        epoch_seconds = time.time() - epoch_start_time

        history_row = {}
        history_row["Epoch"] = epoch
        history_row["Train_Loss"] = train_metrics["loss"]
        history_row["Train_Accuracy"] = train_metrics["accuracy"]
        history_row["Train_Macro_F1"] = train_metrics["macro_f1"]
        history_row["Train_CCC"] = train_metrics["ccc"]
        history_row["Mean_Gradient_Norm"] = train_metrics["mean_gradient_norm"]
        history_row["Val_Loss"] = val_metrics["loss"]
        history_row["Val_Accuracy"] = val_metrics["accuracy"]
        history_row["Val_Macro_F1"] = val_metrics["macro_f1"]
        history_row["Val_Weighted_F1"] = val_metrics["weighted_f1"]
        history_row["Val_CCC"] = val_metrics["ccc"]
        history_row["Val_RMSE"] = val_metrics["rmse"]
        history_row["Val_MAE"] = val_metrics["mae"]
        history_row["Epoch_Seconds"] = epoch_seconds
        history_rows.append(history_row)

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_loss_epoch = epoch
            save_audio_checkpoint(paths["best_loss"], "best_loss", model, optimizer, epoch, question_index, seed, train_metrics, val_metrics, class_counts, class_weights_cpu, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, include_optimizer=False)

        if val_metrics["ccc"] > best_val_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
            save_audio_checkpoint(paths["best_ccc"], "best_ccc", model, optimizer, epoch, question_index, seed, train_metrics, val_metrics, class_counts, class_weights_cpu, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, include_optimizer=False)

        print_audio_epoch_result(epoch, train_metrics, val_metrics)

    save_audio_checkpoint(paths["last"], "last", model, optimizer, num_epochs, question_index, seed, train_metrics, val_metrics, class_counts, class_weights_cpu, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, include_optimizer=True)

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(paths["history"], index=False)
    elapsed_seconds = time.time() - training_start_time

    result = {}
    result["question_index"] = question_index
    result["question_number"] = question_index + 1
    result["question"] = PHQ8_QUESTION_COLUMNS[question_index]
    result["seed"] = seed
    result["best_val_loss"] = best_val_loss
    result["best_loss_epoch"] = best_loss_epoch
    result["best_val_ccc"] = best_val_ccc
    result["best_ccc_epoch"] = best_ccc_epoch
    result["best_loss_path"] = paths["best_loss"]
    result["best_ccc_path"] = paths["best_ccc"]
    result["last_path"] = paths["last"]
    result["history_path"] = paths["history"]
    result["history_df"] = history_df
    result["elapsed_seconds"] = elapsed_seconds

    print("=" * 70)
    print(f"Q{question_index + 1} Seed={seed}训练结束")
    print(f"最佳验证Loss：{best_val_loss:.6f}，Epoch {best_loss_epoch}")
    print(f"最佳验证CCC：{best_val_ccc:.6f}，Epoch {best_ccc_epoch}")
    print(f"总耗时：{elapsed_seconds:.2f}秒")
    print(f"Best Loss：{paths['best_loss']}")
    print(f"Best CCC：{paths['best_ccc']}")
    print(f"Last：{paths['last']}")
    print("=" * 70)

    return result

In [103]:
audio_q1_seed42_result = train_single_audio_question(question_index=0, seed=42)

开始正式训练：Q1 PHQ_8NoInterest，Seed=42
Epoch 01/50 | Train Loss=2.296797 | Val Loss=2.603723 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 02/50 | Train Loss=2.160058 | Val Loss=2.566039 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 03/50 | Train Loss=2.158615 | Val Loss=2.549071 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 04/50 | Train Loss=2.180788 | Val Loss=2.565554 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 05/50 | Train Loss=2.115467 | Val Loss=2.579630 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 06/50 | Train Loss=2.180418 | Val Loss=2.620130 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 07/50 | Train Loss=2.138356 | Val Loss=2.553385 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 08/50 | Train Loss=2.133307 | Val Loss=2.552493 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 09/50 | Train Loss=2.146451 | Val Loss=2.675855 | Val Acc=0.4727 | Val MacroF1=0.1800 | 

In [104]:
print("Best Loss存在：", audio_q1_seed42_result["best_loss_path"].exists())
print("Best CCC存在：", audio_q1_seed42_result["best_ccc_path"].exists())
print("Last存在：", audio_q1_seed42_result["last_path"].exists())
print("历史记录轮数：", len(audio_q1_seed42_result["history_df"]))
display(audio_q1_seed42_result["history_df"].tail())

Best Loss存在： True
Best CCC存在： True
Last存在： True
历史记录轮数： 50


,Epoch,Train_Loss,Train_Accuracy,Train_Macro_F1,Train_CCC,Mean_Gradient_Norm,Val_Loss,Val_Accuracy,Val_Macro_F1,Val_Weighted_F1,Val_CCC,Val_RMSE,Val_MAE,Epoch_Seconds
45,46,2.031444,0.435583,0.275022,0.140930,39.993144,2.647437,0.418182,0.200181,0.333926,0.023204,0.943880,0.672727,0.364249
46,47,1.939445,0.398773,0.265614,0.191613,47.821966,2.678876,0.290909,0.119403,0.191045,0.045796,1.018019,0.818182,0.357451
47,48,1.892571,0.484663,0.365984,0.335373,28.553006,2.734561,0.527273,0.280172,0.482132,0.134901,0.990867,0.618182,0.357039
48,49,2.176455,0.521472,0.271317,0.198464,29.361758,2.652452,0.363636,0.133333,0.213333,0.019750,0.924416,0.709091,0.374563
49,50,2.011734,0.368098,0.232432,0.168344,41.008999,2.603335,0.363636,0.133333,0.213333,0.019750,0.924416,0.709091,0.385962


In [105]:
def load_audio_question_model(checkpoint_path, expected_question_index=None, expected_seed=None):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    if expected_question_index is not None and checkpoint["question_index"] != expected_question_index:
        raise RuntimeError(f"Checkpoint题目不一致：实际{checkpoint['question_index']}，预期{expected_question_index}")

    if expected_seed is not None and checkpoint["seed"] != expected_seed:
        raise RuntimeError(f"Checkpoint Seed不一致：实际{checkpoint['seed']}，预期{expected_seed}")

    model = AudioLSTMAttentionClassifier()
    load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model = model.to(device)
    model.eval()

    print("已加载Checkpoint：", checkpoint_path)
    print("Checkpoint类型：", checkpoint["checkpoint_type"])
    print("保存Epoch：", checkpoint["epoch"])
    print("题目：", checkpoint["question"])
    print("Seed：", checkpoint["seed"])
    print("缺少参数：", load_result.missing_keys)
    print("多余参数：", load_result.unexpected_keys)

    return model, checkpoint

In [106]:
audio_q1_best_loss_model, audio_q1_best_loss_checkpoint = load_audio_question_model(audio_q1_seed42_result["best_loss_path"], expected_question_index=0, expected_seed=42)
audio_q1_class_weights_device = audio_q1_best_loss_checkpoint["class_weights"].to(device)
audio_q1_best_loss_metrics = evaluate_audio_question(audio_q1_best_loss_model, val_audio_loader, question_index=0, class_weights=audio_q1_class_weights_device, device=device)
audio_q1_saved_best_loss = float(audio_q1_best_loss_checkpoint["val_metrics"]["loss"])
audio_q1_restored_best_loss = float(audio_q1_best_loss_metrics["loss"])
audio_q1_best_loss_difference = abs(audio_q1_saved_best_loss - audio_q1_restored_best_loss)

print("Checkpoint记录的Loss：", audio_q1_saved_best_loss)
print("重新计算的Loss：", audio_q1_restored_best_loss)
print("Loss差值：", audio_q1_best_loss_difference)
print("Checkpoint记录的CCC：", audio_q1_best_loss_checkpoint["val_metrics"]["ccc"])
print("重新计算的CCC：", audio_q1_best_loss_metrics["ccc"])
print("混淆矩阵：")
print(audio_q1_best_loss_metrics["confusion_matrix"])

已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
Checkpoint类型： best_loss
保存Epoch： 10
题目： PHQ_8NoInterest
Seed： 42
缺少参数： []
多余参数： []
Checkpoint记录的Loss： 2.548061175779863
重新计算的Loss： 2.548061175779863
Loss差值： 0.0
Checkpoint记录的CCC： 0.01018634531646967
重新计算的CCC： 0.01018634531646967
混淆矩阵：
tensor([[ 0, 25,  0,  0],
        [ 0, 21,  1,  0],
        [ 0,  4,  0,  0],
        [ 0,  4,  0,  0]])


In [107]:
audio_q1_best_ccc_model, audio_q1_best_ccc_checkpoint = load_audio_question_model(audio_q1_seed42_result["best_ccc_path"], expected_question_index=0, expected_seed=42)
audio_q1_class_weights_device = audio_q1_best_ccc_checkpoint["class_weights"].to(device)
audio_q1_best_ccc_metrics = evaluate_audio_question(audio_q1_best_ccc_model, val_audio_loader, question_index=0, class_weights=audio_q1_class_weights_device, device=device)
audio_q1_saved_best_ccc = float(audio_q1_best_ccc_checkpoint["val_metrics"]["ccc"])
audio_q1_restored_best_ccc = float(audio_q1_best_ccc_metrics["ccc"])
audio_q1_best_ccc_difference = abs(audio_q1_saved_best_ccc - audio_q1_restored_best_ccc)

print("Checkpoint记录的CCC：", audio_q1_saved_best_ccc)
print("重新计算的CCC：", audio_q1_restored_best_ccc)
print("CCC差值：", audio_q1_best_ccc_difference)
print("Checkpoint记录的Loss：", audio_q1_best_ccc_checkpoint["val_metrics"]["loss"])
print("重新计算的Loss：", audio_q1_best_ccc_metrics["loss"])
print("Accuracy：", audio_q1_best_ccc_metrics["accuracy"])
print("Macro F1：", audio_q1_best_ccc_metrics["macro_f1"])
print("每类F1：", audio_q1_best_ccc_metrics["per_class_f1"])
print("混淆矩阵：")
print(audio_q1_best_ccc_metrics["confusion_matrix"])

已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 14
题目： PHQ_8NoInterest
Seed： 42
缺少参数： []
多余参数： []
Checkpoint记录的CCC： 0.1517808735370636
重新计算的CCC： 0.1517808735370636
CCC差值： 0.0
Checkpoint记录的Loss： 2.5651531436226587
重新计算的Loss： 2.5651531436226587
Accuracy： 0.38181817531585693
Macro F1： 0.251932829618454
每类F1： tensor([0.5106, 0.3721, 0.1250, 0.0000])
混淆矩阵：
tensor([[12, 10,  3,  0],
        [ 7,  8,  7,  0],
        [ 2,  1,  1,  0],
        [ 1,  2,  1,  0]])


In [108]:
def load_completed_audio_experiment(question_index, seed):
    paths = get_audio_experiment_paths(question_index, seed)
    required_paths = [paths["best_loss"], paths["best_ccc"], paths["last"], paths["history"]]

    if not all(path.exists() for path in required_paths):
        return None

    best_loss_checkpoint = torch.load(paths["best_loss"], map_location="cpu", weights_only=False)
    best_ccc_checkpoint = torch.load(paths["best_ccc"], map_location="cpu", weights_only=False)
    last_checkpoint = torch.load(paths["last"], map_location="cpu", weights_only=False)
    history_df = pd.read_csv(paths["history"])

    if best_loss_checkpoint["question_index"] != question_index:
        return None

    if best_ccc_checkpoint["question_index"] != question_index:
        return None

    if last_checkpoint["question_index"] != question_index:
        return None

    if best_loss_checkpoint["seed"] != seed:
        return None

    if best_ccc_checkpoint["seed"] != seed:
        return None

    if last_checkpoint["seed"] != seed:
        return None

    if last_checkpoint["epoch"] != NUM_EPOCHS:
        return None

    if len(history_df) != NUM_EPOCHS:
        return None

    result = {}
    result["question_index"] = question_index
    result["question_number"] = question_index + 1
    result["question"] = PHQ8_QUESTION_COLUMNS[question_index]
    result["seed"] = seed
    result["best_val_loss"] = float(best_loss_checkpoint["val_metrics"]["loss"])
    result["best_loss_epoch"] = int(best_loss_checkpoint["epoch"])
    result["best_val_ccc"] = float(best_ccc_checkpoint["val_metrics"]["ccc"])
    result["best_ccc_epoch"] = int(best_ccc_checkpoint["epoch"])
    result["best_loss_path"] = paths["best_loss"]
    result["best_ccc_path"] = paths["best_ccc"]
    result["last_path"] = paths["last"]
    result["history_path"] = paths["history"]
    result["history_df"] = history_df
    result["elapsed_seconds"] = float(history_df["Epoch_Seconds"].sum())
    result["loaded_from_disk"] = True
    return result

In [109]:
def load_completed_audio_experiment(question_index, seed):
    paths = get_audio_experiment_paths(question_index, seed)
    required_paths = [paths["best_loss"], paths["best_ccc"], paths["last"], paths["history"]]

    if not all(path.exists() for path in required_paths):
        return None

    best_loss_checkpoint = torch.load(paths["best_loss"], map_location="cpu", weights_only=False)
    best_ccc_checkpoint = torch.load(paths["best_ccc"], map_location="cpu", weights_only=False)
    last_checkpoint = torch.load(paths["last"], map_location="cpu", weights_only=False)
    history_df = pd.read_csv(paths["history"])

    if best_loss_checkpoint["question_index"] != question_index:
        return None

    if best_ccc_checkpoint["question_index"] != question_index:
        return None

    if last_checkpoint["question_index"] != question_index:
        return None

    if best_loss_checkpoint["seed"] != seed:
        return None

    if best_ccc_checkpoint["seed"] != seed:
        return None

    if last_checkpoint["seed"] != seed:
        return None

    if last_checkpoint["epoch"] != NUM_EPOCHS:
        return None

    if len(history_df) != NUM_EPOCHS:
        return None

    result = {}
    result["question_index"] = question_index
    result["question_number"] = question_index + 1
    result["question"] = PHQ8_QUESTION_COLUMNS[question_index]
    result["seed"] = seed
    result["best_val_loss"] = float(best_loss_checkpoint["val_metrics"]["loss"])
    result["best_loss_epoch"] = int(best_loss_checkpoint["epoch"])
    result["best_val_ccc"] = float(best_ccc_checkpoint["val_metrics"]["ccc"])
    result["best_ccc_epoch"] = int(best_ccc_checkpoint["epoch"])
    result["best_loss_path"] = paths["best_loss"]
    result["best_ccc_path"] = paths["best_ccc"]
    result["last_path"] = paths["last"]
    result["history_path"] = paths["history"]
    result["history_df"] = history_df
    result["elapsed_seconds"] = float(history_df["Epoch_Seconds"].sum())
    result["loaded_from_disk"] = True
    return result

In [110]:
def train_or_load_audio_question(question_index, seed):
    completed_result = load_completed_audio_experiment(question_index, seed)

    if completed_result is not None:
        print(f"Q{question_index + 1} Seed={seed}已经完成，直接读取磁盘结果")
        return completed_result

    print(f"Q{question_index + 1} Seed={seed}尚未完成，开始正式训练")
    result = train_single_audio_question(question_index=question_index, seed=seed, num_epochs=NUM_EPOCHS)
    result["loaded_from_disk"] = False
    return result

In [111]:
def train_all_audio_questions_for_seed(seed):
    seed_results = []
    seed_start_time = time.time()

    print()
    print("#" * 80)
    print(f"开始处理Audio-only全部8道题，Seed={seed}")
    print("#" * 80)

    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        question_result = train_or_load_audio_question(question_index, seed)
        seed_results.append(question_result)

    summary_rows = []

    for result in seed_results:
        row = {}
        row["Question_Number"] = result["question_number"]
        row["Question"] = result["question"]
        row["Seed"] = result["seed"]
        row["Best_Loss"] = result["best_val_loss"]
        row["Best_Loss_Epoch"] = result["best_loss_epoch"]
        row["Best_CCC"] = result["best_val_ccc"]
        row["Best_CCC_Epoch"] = result["best_ccc_epoch"]
        row["Loaded_From_Disk"] = result["loaded_from_disk"]
        row["Elapsed_Seconds"] = result["elapsed_seconds"]
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_path = CHECKPOINT_ROOT / f"seed_{seed}_question_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    seed_elapsed_seconds = time.time() - seed_start_time

    print()
    print("#" * 80)
    print(f"Audio-only Seed={seed}的8道题全部完成")
    print(f"本次运行耗时：{seed_elapsed_seconds:.2f}秒")
    print(f"8道题平均最佳验证CCC：{summary_df['Best_CCC'].mean():.6f}")
    print(f"汇总文件：{summary_path}")
    print("#" * 80)

    return seed_results, summary_df

In [112]:
del audio_q1_best_loss_model
del audio_q1_best_ccc_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [113]:
audio_seed42_results, audio_seed42_summary_df = train_all_audio_questions_for_seed(seed=42)
display(audio_seed42_summary_df)


################################################################################
开始处理Audio-only全部8道题，Seed=42
################################################################################
Q1 Seed=42已经完成，直接读取磁盘结果
Q2 Seed=42尚未完成，开始正式训练
开始正式训练：Q2 PHQ_8Depressed，Seed=42
Epoch 01/50 | Train Loss=2.520217 | Val Loss=2.414230 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
Epoch 02/50 | Train Loss=2.568020 | Val Loss=2.365933 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
Epoch 03/50 | Train Loss=2.437933 | Val Loss=2.373367 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
Epoch 04/50 | Train Loss=2.434529 | Val Loss=2.331725 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
Epoch 05/50 | Train Loss=2.356787 | Val Loss=2.343744 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
Epoch 06/50 | Train Loss=2.526985 | Val Loss=2.315916 | Val Acc=0.3455 | Val MacroF1=0.1284 | Val CCC=0.0000
Epoch 07/50 | Train Loss=2.354275 | Val Loss=2.408296 | Val Acc=0.2545 | Val

,Question_Number,Question,Seed,Best_Loss,Best_Loss_Epoch,Best_CCC,Best_CCC_Epoch,Loaded_From_Disk,Elapsed_Seconds
0,1,PHQ_8NoInterest,42,2.548061,10,0.151781,14,True,18.595483
1,2,PHQ_8Depressed,42,2.305021,11,0.166761,15,False,18.382560
2,3,PHQ_8Sleep,42,2.463115,14,0.131503,9,False,18.087579
3,4,PHQ_8Tired,42,2.348140,28,0.240907,39,False,18.072135
4,5,PHQ_8Appetite,42,2.206802,32,0.114119,18,False,17.958508
5,6,PHQ_8Failure,42,2.249691,45,0.238375,19,False,18.162375
6,7,PHQ_8Concentrating,42,2.265645,3,0.136929,39,False,17.664063
7,8,PHQ_8Moving,42,1.781414,12,0.126379,44,False,17.733304


In [114]:
print("结果数量：", len(audio_seed42_results))
print("汇总行数：", len(audio_seed42_summary_df))
print("Best Loss文件全部存在：", all(result["best_loss_path"].exists() for result in audio_seed42_results))
print("Best CCC文件全部存在：", all(result["best_ccc_path"].exists() for result in audio_seed42_results))
print("Last文件全部存在：", all(result["last_path"].exists() for result in audio_seed42_results))

结果数量： 8
汇总行数： 8
Best Loss文件全部存在： True
Best CCC文件全部存在： True
Last文件全部存在： True


In [115]:
class CachedAudioTestDataset(Dataset):
    def __init__(self, metadata_df):
        super().__init__()
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.total_scores = torch.tensor(self.metadata_df["PHQ_Score"].to_numpy(), dtype=torch.long)
        audio_feature_list = []
        padding_mask_list = []
        real_turn_count_list = []
        cache_hit_count = 0

        for participant_id in self.participant_ids:
            audio_features, padding_mask, real_turn_count, cache_hit = get_participant_audio_features(participant_id)
            audio_feature_list.append(audio_features)
            padding_mask_list.append(padding_mask)
            real_turn_count_list.append(real_turn_count)

            if cache_hit:
                cache_hit_count += 1

        self.audio_features = torch.stack(audio_feature_list, dim=0)
        self.padding_masks = torch.stack(padding_mask_list, dim=0)
        self.real_turn_counts = torch.tensor(real_turn_count_list, dtype=torch.long)
        self.cache_hit_count = cache_hit_count
        self.cache_miss_count = len(self.participant_ids) - cache_hit_count

        if self.audio_features.shape != (len(self.participant_ids), MAX_TURNS, AUDIO_FEATURE_DIM):
            raise RuntimeError(f"测试集语音特征形状错误：{self.audio_features.shape}")

        if self.padding_masks.shape != (len(self.participant_ids), MAX_TURNS):
            raise RuntimeError(f"测试集Mask形状错误：{self.padding_masks.shape}")

        if self.total_scores.shape != (len(self.participant_ids),):
            raise RuntimeError(f"测试集总分形状错误：{self.total_scores.shape}")

        if not torch.isfinite(self.audio_features).all():
            raise RuntimeError("测试集语音特征包含NaN或Inf")

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        audio_features = self.audio_features[index]
        padding_mask = self.padding_masks[index]
        total_score = self.total_scores[index]
        participant_id = self.participant_ids[index]
        return audio_features, padding_mask, total_score, participant_id

In [116]:
test_audio_dataset = CachedAudioTestDataset(test_metadata_df)
test_audio_loader = DataLoader(test_audio_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

test_batch_audio, test_batch_masks, test_batch_scores, test_batch_participant_ids = next(iter(test_audio_loader))

print("测试样本数：", len(test_audio_dataset))
print("测试批次数：", len(test_audio_loader))
print("测试语音特征：", test_audio_dataset.audio_features.shape)
print("测试Mask：", test_audio_dataset.padding_masks.shape)
print("测试总分：", test_audio_dataset.total_scores.shape)
print("第一批语音特征：", test_batch_audio.shape)
print("第一批Mask：", test_batch_masks.shape)
print("第一批总分：", test_batch_scores.shape)
print("第一批参与者编号：", test_batch_participant_ids)
print("缓存命中数量：", test_audio_dataset.cache_hit_count)
print("缓存未命中数量：", test_audio_dataset.cache_miss_count)

测试样本数： 56
测试批次数： 6
测试语音特征： torch.Size([56, 120, 23])
测试Mask： torch.Size([56, 120])
测试总分： torch.Size([56])
第一批语音特征： torch.Size([10, 120, 23])
第一批Mask： torch.Size([10, 120])
第一批总分： torch.Size([10])
第一批参与者编号： tensor([600, 602, 604, 605, 606, 607, 609, 615, 618, 619])
缓存命中数量： 56
缓存未命中数量： 0


In [117]:
def compute_audio_total_score_metrics(predictions, labels):
    predictions = predictions.cpu().float()
    labels = labels.cpu().float()
    ccc = compute_ccc(predictions, labels)
    rmse = torch.sqrt(torch.mean((predictions - labels) ** 2)).item()
    mae = torch.mean(torch.abs(predictions - labels)).item()
    exact_accuracy = torch.mean((predictions == labels).float()).item()
    metrics = {}
    metrics["ccc"] = ccc
    metrics["rmse"] = rmse
    metrics["mae"] = mae
    metrics["exact_accuracy"] = exact_accuracy
    return metrics

In [118]:
def evaluate_audio_seed_on_test(seed, seed_results, test_loader):
    question_count = len(PHQ8_QUESTION_COLUMNS)
    sample_count = len(test_loader.dataset)
    expected_participant_ids = torch.tensor(test_loader.dataset.participant_ids, dtype=torch.long)
    true_total_scores = test_loader.dataset.total_scores.clone()
    question_predictions = torch.empty((sample_count, question_count), dtype=torch.long)
    checkpoint_rows = []

    for question_index in range(question_count):
        question_result = seed_results[question_index]
        checkpoint_path = question_result["best_ccc_path"]
        model, checkpoint = load_audio_question_model(checkpoint_path, expected_question_index=question_index, expected_seed=seed)
        batch_prediction_list = []
        batch_participant_id_list = []
        model.eval()

        with torch.no_grad():
            for batch_audio_features, batch_padding_masks, batch_total_scores, batch_participant_ids in test_loader:
                batch_audio_features = batch_audio_features.to(device)
                batch_padding_masks = batch_padding_masks.to(device)
                logits = model(batch_audio_features, batch_padding_masks)
                batch_predictions = torch.argmax(logits, dim=1)
                batch_prediction_list.append(batch_predictions.cpu())
                batch_participant_id_list.append(batch_participant_ids.cpu())

        question_prediction = torch.cat(batch_prediction_list, dim=0)
        observed_participant_ids = torch.cat(batch_participant_id_list, dim=0)

        if not torch.equal(observed_participant_ids, expected_participant_ids):
            raise RuntimeError("测试参与者顺序不一致")

        if question_prediction.shape != (sample_count,):
            raise RuntimeError(f"Q{question_index + 1}测试预测形状错误：{question_prediction.shape}")

        question_predictions[:, question_index] = question_prediction

        checkpoint_row = {}
        checkpoint_row["Question_Number"] = question_index + 1
        checkpoint_row["Question"] = PHQ8_QUESTION_COLUMNS[question_index]
        checkpoint_row["Seed"] = seed
        checkpoint_row["Selected_Epoch"] = checkpoint["epoch"]
        checkpoint_row["Validation_CCC"] = checkpoint["val_metrics"]["ccc"]
        checkpoint_row["Checkpoint_Path"] = str(checkpoint_path)
        checkpoint_rows.append(checkpoint_row)

        print(f"Seed {seed}，Q{question_index + 1}/8测试完成")

        del model
        del checkpoint
        del logits

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    predicted_total_scores = question_predictions.sum(dim=1)
    total_metrics = compute_audio_total_score_metrics(predicted_total_scores, true_total_scores)
    prediction_df = pd.DataFrame({"Participant_ID": expected_participant_ids.numpy()})

    for question_index in range(question_count):
        prediction_df[f"Pred_Q{question_index + 1}"] = question_predictions[:, question_index].numpy()

    prediction_df["True_Total"] = true_total_scores.numpy()
    prediction_df["Predicted_Total"] = predicted_total_scores.numpy()
    prediction_df["Total_Error"] = prediction_df["Predicted_Total"] - prediction_df["True_Total"]
    prediction_df["Absolute_Total_Error"] = prediction_df["Total_Error"].abs()

    prediction_path = CHECKPOINT_ROOT / f"seed_{seed}_test_predictions_56.csv"
    checkpoint_summary_path = CHECKPOINT_ROOT / f"seed_{seed}_test_selected_checkpoints.csv"
    prediction_df.to_csv(prediction_path, index=False)
    checkpoint_df = pd.DataFrame(checkpoint_rows)
    checkpoint_df.to_csv(checkpoint_summary_path, index=False)

    result = {}
    result["seed"] = seed
    result["sample_count"] = sample_count
    result["ccc"] = total_metrics["ccc"]
    result["rmse"] = total_metrics["rmse"]
    result["mae"] = total_metrics["mae"]
    result["exact_accuracy"] = total_metrics["exact_accuracy"]
    result["question_predictions"] = question_predictions
    result["predicted_total"] = predicted_total_scores
    result["true_total"] = true_total_scores
    result["participant_ids"] = expected_participant_ids
    result["prediction_df"] = prediction_df
    result["checkpoint_df"] = checkpoint_df
    result["prediction_path"] = prediction_path
    result["checkpoint_summary_path"] = checkpoint_summary_path

    print()
    print("=" * 70)
    print(f"Audio-only Seed {seed}独立测试结果")
    print("=" * 70)
    print(f"测试参与者数量：{sample_count}")
    print(f"CCC：{result['ccc']:.6f}")
    print(f"RMSE：{result['rmse']:.6f}")
    print(f"MAE：{result['mae']:.6f}")
    print(f"完全相等比例：{result['exact_accuracy']:.6f}")
    print(f"预测总分范围：{predicted_total_scores.min().item()}～{predicted_total_scores.max().item()}")
    print(f"真实总分范围：{true_total_scores.min().item()}～{true_total_scores.max().item()}")
    print(f"预测文件：{prediction_path}")

    return result

In [119]:
audio_seed42_test_result = evaluate_audio_seed_on_test(seed=42, seed_results=audio_seed42_results, test_loader=test_audio_loader)

已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 14
题目： PHQ_8NoInterest
Seed： 42
缺少参数： []
多余参数： []
Seed 42，Q1/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_2/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 15
题目： PHQ_8Depressed
Seed： 42
缺少参数： []
多余参数： []
Seed 42，Q2/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_3/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 9
题目： PHQ_8Sleep
Seed： 42
缺少参数： []
多余参数： []
Seed 42，Q3/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_4/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 39
题目： PHQ_8Tired
Seed： 42
缺少参数： []
多余参数： []
Seed 42，Q4/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_5/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 18
题目： PHQ_8Appetite
Seed： 42
缺少参数： []
多余参数： []
Seed 42，Q5/8测试完成
已加载Checkpoint： /workspace/E-DAIC/c

In [120]:
audio_seed100_results, audio_seed100_summary_df = train_all_audio_questions_for_seed(seed=100)
display(audio_seed100_summary_df)

audio_seed1234_results, audio_seed1234_summary_df = train_all_audio_questions_for_seed(seed=1234)
display(audio_seed1234_summary_df)


################################################################################
开始处理Audio-only全部8道题，Seed=100
################################################################################
Q1 Seed=100尚未完成，开始正式训练
开始正式训练：Q1 PHQ_8NoInterest，Seed=100
Epoch 01/50 | Train Loss=2.263386 | Val Loss=2.610214 | Val Acc=0.3273 | Val MacroF1=0.1714 | Val CCC=-0.1982
Epoch 02/50 | Train Loss=2.135978 | Val Loss=2.688951 | Val Acc=0.3818 | Val MacroF1=0.1530 | Val CCC=-0.1017
Epoch 03/50 | Train Loss=2.174554 | Val Loss=2.640226 | Val Acc=0.3636 | Val MacroF1=0.1956 | Val CCC=-0.0542
Epoch 04/50 | Train Loss=2.095424 | Val Loss=2.614568 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 05/50 | Train Loss=2.178258 | Val Loss=2.655431 | Val Acc=0.3636 | Val MacroF1=0.1408 | Val CCC=-0.0961
Epoch 06/50 | Train Loss=2.195545 | Val Loss=2.561233 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 07/50 | Train Loss=2.101133 | Val Loss=2.582253 | Val Acc=0.4000 | Val MacroF1=0.1429 

,Question_Number,Question,Seed,Best_Loss,Best_Loss_Epoch,Best_CCC,Best_CCC_Epoch,Loaded_From_Disk,Elapsed_Seconds
0,1,PHQ_8NoInterest,100,2.529667,8,0.245012,43,False,19.169299
1,2,PHQ_8Depressed,100,2.281175,43,0.186750,17,False,17.870027
2,3,PHQ_8Sleep,100,2.475516,15,0.156694,13,False,17.867615
3,4,PHQ_8Tired,100,2.322259,47,0.183022,21,False,18.615410
4,5,PHQ_8Appetite,100,2.207972,7,0.083353,13,False,18.579286
5,6,PHQ_8Failure,100,2.257047,47,0.191169,23,False,18.629285
6,7,PHQ_8Concentrating,100,2.287100,4,0.130372,39,False,18.439207
7,8,PHQ_8Moving,100,1.778134,12,0.179163,20,False,18.516966



################################################################################
开始处理Audio-only全部8道题，Seed=1234
################################################################################
Q1 Seed=1234尚未完成，开始正式训练
开始正式训练：Q1 PHQ_8NoInterest，Seed=1234
Epoch 01/50 | Train Loss=2.195702 | Val Loss=2.603054 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 02/50 | Train Loss=2.205141 | Val Loss=2.573020 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 03/50 | Train Loss=2.141337 | Val Loss=2.565107 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 04/50 | Train Loss=2.133400 | Val Loss=2.522752 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 05/50 | Train Loss=2.092318 | Val Loss=2.575061 | Val Acc=0.4000 | Val MacroF1=0.1429 | Val CCC=0.0000
Epoch 06/50 | Train Loss=2.160015 | Val Loss=3.482110 | Val Acc=0.4545 | Val MacroF1=0.1562 | Val CCC=0.0000
Epoch 07/50 | Train Loss=2.223071 | Val Loss=2.581717 | Val Acc=0.4909 | Val MacroF1=0.2157 |

,Question_Number,Question,Seed,Best_Loss,Best_Loss_Epoch,Best_CCC,Best_CCC_Epoch,Loaded_From_Disk,Elapsed_Seconds
0,1,PHQ_8NoInterest,1234,2.517797,18,0.246618,12,False,18.495439
1,2,PHQ_8Depressed,1234,2.278944,11,0.185385,30,False,18.469127
2,3,PHQ_8Sleep,1234,2.470923,14,0.131100,10,False,18.485166
3,4,PHQ_8Tired,1234,2.331623,31,0.190841,13,False,17.664965
4,5,PHQ_8Appetite,1234,2.221607,22,0.094828,12,False,18.170670
5,6,PHQ_8Failure,1234,2.249147,48,0.179682,35,False,17.845043
6,7,PHQ_8Concentrating,1234,2.259439,4,0.059321,20,False,17.963823
7,8,PHQ_8Moving,1234,1.764127,21,0.129527,30,False,17.929601


In [121]:
all_audio_training_results = {}
all_audio_training_results[42] = audio_seed42_results
all_audio_training_results[100] = audio_seed100_results
all_audio_training_results[1234] = audio_seed1234_results

for seed, seed_results in all_audio_training_results.items():
    print(f"Seed {seed}结果数量：", len(seed_results))
    print(f"Seed {seed} Best Loss全部存在：", all(result["best_loss_path"].exists() for result in seed_results))
    print(f"Seed {seed} Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in seed_results))
    print(f"Seed {seed} Last全部存在：", all(result["last_path"].exists() for result in seed_results))

Seed 42结果数量： 8
Seed 42 Best Loss全部存在： True
Seed 42 Best CCC全部存在： True
Seed 42 Last全部存在： True
Seed 100结果数量： 8
Seed 100 Best Loss全部存在： True
Seed 100 Best CCC全部存在： True
Seed 100 Last全部存在： True
Seed 1234结果数量： 8
Seed 1234 Best Loss全部存在： True
Seed 1234 Best CCC全部存在： True
Seed 1234 Last全部存在： True


In [122]:
audio_seed100_test_result = evaluate_audio_seed_on_test(seed=100, seed_results=audio_seed100_results, test_loader=test_audio_loader)
audio_seed1234_test_result = evaluate_audio_seed_on_test(seed=1234, seed_results=audio_seed1234_results, test_loader=test_audio_loader)

已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_1/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 43
题目： PHQ_8NoInterest
Seed： 100
缺少参数： []
多余参数： []
Seed 100，Q1/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_2/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 17
题目： PHQ_8Depressed
Seed： 100
缺少参数： []
多余参数： []
Seed 100，Q2/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_3/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 13
题目： PHQ_8Sleep
Seed： 100
缺少参数： []
多余参数： []
Seed 100，Q3/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_4/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 21
题目： PHQ_8Tired
Seed： 100
缺少参数： []
多余参数： []
Seed 100，Q4/8测试完成
已加载Checkpoint： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_5/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 13
题目： PHQ_8Appetite
Seed： 100
缺少参数： []
多余参数： []
Seed 100，Q5/8测试完成
已加载Checkpoint： /wo

In [123]:
all_audio_test_results = [audio_seed42_test_result, audio_seed100_test_result, audio_seed1234_test_result]
audio_test_seed_rows = []

for result in all_audio_test_results:
    row = {}
    row["Seed"] = result["seed"]
    row["Sample_Count"] = result["sample_count"]
    row["CCC"] = result["ccc"]
    row["RMSE"] = result["rmse"]
    row["MAE"] = result["mae"]
    row["Exact_Accuracy"] = result["exact_accuracy"]
    audio_test_seed_rows.append(row)

audio_test_seed_summary_df = pd.DataFrame(audio_test_seed_rows)
display(audio_test_seed_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,56,0.275914,7.463004,5.839286,0.053571
1,100,56,0.237006,6.495878,5.267857,0.035714
2,1234,56,0.268115,7.418606,5.607143,0.160714


In [124]:
audio_three_seed_rows = []

for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    row = {}
    row["Metric"] = metric_name
    row["Mean"] = audio_test_seed_summary_df[metric_name].mean()
    row["Std"] = audio_test_seed_summary_df[metric_name].std(ddof=1)
    audio_three_seed_rows.append(row)

audio_three_seed_summary_df = pd.DataFrame(audio_three_seed_rows)
display(audio_three_seed_summary_df)

,Metric,Mean,Std
0,CCC,0.260345,0.020585
1,RMSE,7.125829,0.546005
2,MAE,5.571429,0.287384
3,Exact_Accuracy,0.083333,0.067606


论文：ccc   0.273+- 0.021
rmse   6.67+-  0.11

MAE    5.32 +- 0.05

In [125]:
audio_test_seed_summary_path = CHECKPOINT_ROOT / "three_seed_test_results_56.csv"
audio_three_seed_summary_path = CHECKPOINT_ROOT / "three_seed_test_summary_56.csv"

audio_test_seed_summary_df.to_csv(audio_test_seed_summary_path, index=False)
audio_three_seed_summary_df.to_csv(audio_three_seed_summary_path, index=False)

print("逐Seed测试结果：", audio_test_seed_summary_path)
print("三Seed均值与标准差：", audio_three_seed_summary_path)
print("逐Seed文件存在：", audio_test_seed_summary_path.exists())
print("三Seed汇总文件存在：", audio_three_seed_summary_path.exists())

逐Seed测试结果： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/three_seed_test_results_56.csv
三Seed均值与标准差： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/three_seed_test_summary_56.csv
逐Seed文件存在： True
三Seed汇总文件存在： True


In [126]:
from pathlib import Path
import random
import sys
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("/workspace/E-DAIC/original_data")
SPLIT_DIR = Path("/workspace/E-DAIC/splits")
LABEL_DIR = Path("/workspace/E-DAIC/others")

TRAIN_SPLIT_PATH = SPLIT_DIR / "train_split.csv"
VAL_SPLIT_PATH = SPLIT_DIR / "dev_split.csv"
TEST_SPLIT_PATH = SPLIT_DIR / "test_split.csv"
DETAILED_LABEL_PATH = LABEL_DIR / "Detailed_PHQ8_Labels.csv"

TEXT_CACHE_ROOT = Path("/workspace/E-DAIC/cache/phq8_text_embeddings")
AUDIO_CACHE_DIR = Path("/workspace/E-DAIC/cache/phq8_audio_egemaps/egemaps23_turnmean_100hz_turns120_v1")

TEXT_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_text_paperlike")
AUDIO_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_audio_paperlike")
FUSION_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike")

PHQ8_QUESTION_COLUMNS = ["PHQ_8NoInterest", "PHQ_8Depressed", "PHQ_8Sleep", "PHQ_8Tired", "PHQ_8Appetite", "PHQ_8Failure", "PHQ_8Concentrating", "PHQ_8Moving"]

MAX_TURNS = 120
TEXT_FEATURE_DIM = 768
AUDIO_FEATURE_DIM = 23
ENCODER_OUTPUT_DIM = 100
NUM_CLASSES = 4
ATTENTION_HEADS = 4

BATCH_SIZE = 10
NUM_EPOCHS = 20
LEARNING_RATE = 5e-4
ADAM_EPSILON = 1e-8
WEIGHT_DECAY = 1e-3
MAX_GRAD_NORM = 1.0

FUSION_DROPOUT = 0.8
MLP_FIRST_DROPOUT = 0.8
MLP_LAST_DROPOUT = 0.5

ALPHA = 1.0
BETA = 0.5
LOSS_EPSILON = 1e-12
SEEDS = [42, 100, 1234]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
FUSION_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("Python：", sys.version.split()[0])
print("PyTorch：", torch.__version__)
print("计算设备：", device)
print("Text缓存目录存在：", TEXT_CACHE_ROOT.exists())
print("Audio缓存目录存在：", AUDIO_CACHE_DIR.exists())
print("Text Checkpoint目录存在：", TEXT_CHECKPOINT_ROOT.exists())
print("Audio Checkpoint目录存在：", AUDIO_CHECKPOINT_ROOT.exists())
print("Fusion Checkpoint目录：", FUSION_CHECKPOINT_ROOT)
print("融合训练轮数：", NUM_EPOCHS)
print("融合Attention Dropout：", FUSION_DROPOUT)

Python： 3.12.3
PyTorch： 2.9.0
计算设备： cuda:0
Text缓存目录存在： True
Audio缓存目录存在： True
Text Checkpoint目录存在： True
Audio Checkpoint目录存在： True
Fusion Checkpoint目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike
融合训练轮数： 20
融合Attention Dropout： 0.8


In [141]:
text_cache_candidates = sorted(TEXT_CACHE_ROOT.rglob("302.pt"))
audio_cache_path_302 = AUDIO_CACHE_DIR / "302.pt"
text_best_loss_path = TEXT_CHECKPOINT_ROOT /"q1_PHQ_8NoInterest"/ "seed_42"/ "best_loss.pt"
audio_best_loss_path = AUDIO_CHECKPOINT_ROOT / "seed_42" / "question_1" / "best_loss.pt"

print("找到的302文本缓存：", text_cache_candidates)
print("302语音缓存存在：", audio_cache_path_302.exists())
print("Q1文本Best Loss存在：", text_best_loss_path.exists())
print("Q1语音Best Loss存在：", audio_best_loss_path.exists())

if len(text_cache_candidates) != 1:
    raise RuntimeError(f"预期找到一个302文本缓存，实际找到{len(text_cache_candidates)}个")

TEXT_CACHE_DIR = text_cache_candidates[0].parent

text_cache_302 = torch.load(text_cache_candidates[0], map_location="cpu", weights_only=False)
audio_cache_302 = torch.load(audio_cache_path_302, map_location="cpu", weights_only=False)
text_checkpoint_302 = torch.load(text_best_loss_path, map_location="cpu", weights_only=False)
audio_checkpoint_302 = torch.load(audio_best_loss_path, map_location="cpu", weights_only=False)

print("正式Text缓存目录：", TEXT_CACHE_DIR)
print("Text缓存字段：", list(text_cache_302.keys()))
print("Audio缓存字段：", list(audio_cache_302.keys()))
print("Text Checkpoint字段：", list(text_checkpoint_302.keys()))
print("Audio Checkpoint字段：", list(audio_checkpoint_302.keys()))
print("Text模型前10个参数名：", list(text_checkpoint_302["model_state_dict"].keys())[:10])
print("Audio模型前10个参数名：", list(audio_checkpoint_302["model_state_dict"].keys())[:10])

找到的302文本缓存： [PosixPath('/workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1/302.pt')]
302语音缓存存在： True
Q1文本Best Loss存在： True
Q1语音Best Loss存在： True
正式Text缓存目录： /workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1
Text缓存字段： ['participant_id', 'model_name', 'model_revision', 'max_turns', 'embedding_dim', 'source_size', 'source_mtime_ns', 'real_turn_count', 'embeddings', 'key_padding_mask']
Audio缓存字段： ['participant_id', 'cache_version', 'feature_rate', 'feature_dim', 'max_turns', 'original_transcript_turns', 'cleaned_turns', 'real_turn_count', 'audio_duration_seconds', 'wav_signature', 'transcript_signature', 'egemaps_signature', 'audio_features', 'padding_mask']
Text Checkpoint字段： ['checkpoint_type', 'epoch', 'question_index', 'question_number', 'question_name', 'seed', 'model_state_dict', 'train_metrics', 'val_metrics', 'class_counts', 'class_weights', 'best_val_loss', 'best_val_ccc', 'best_loss_epoch', 'best_ccc_epoch', 'model_name',

In [142]:
def find_checkpoint_by_metadata(checkpoint_root, expected_seed, expected_question_index, expected_checkpoint_type):
    matched_paths = []
    candidate_paths = sorted(checkpoint_root.rglob("*.pt"))

    for checkpoint_path in candidate_paths:
        try:
            checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        except Exception:
            continue

        if not isinstance(checkpoint, dict):
            continue

        checkpoint_seed = checkpoint.get("seed")
        checkpoint_question_index = checkpoint.get("question_index")
        checkpoint_type = checkpoint.get("checkpoint_type")

        if checkpoint_seed == expected_seed and checkpoint_question_index == expected_question_index and checkpoint_type == expected_checkpoint_type:
            matched_paths.append(checkpoint_path)

        del checkpoint

    if len(matched_paths) == 0:
        raise FileNotFoundError(f"没有找到Seed={expected_seed}、question_index={expected_question_index}、类型={expected_checkpoint_type}的Checkpoint，搜索目录：{checkpoint_root}")

    if len(matched_paths) > 1:
        print("找到多个符合条件的Checkpoint：")

        for matched_path in matched_paths:
            print(" ", matched_path)

        raise RuntimeError("符合条件的Checkpoint不唯一，需要先确认使用哪一个")

    return matched_paths[0]

In [129]:
text_best_loss_path = find_checkpoint_by_metadata(TEXT_CHECKPOINT_ROOT, expected_seed=42, expected_question_index=0, expected_checkpoint_type="best_loss")
audio_best_loss_path = find_checkpoint_by_metadata(AUDIO_CHECKPOINT_ROOT, expected_seed=42, expected_question_index=0, expected_checkpoint_type="best_loss")

print("Q1文本Best Loss：", text_best_loss_path)
print("Q1语音Best Loss：", audio_best_loss_path)
print("Q1文本Best Loss存在：", text_best_loss_path.exists())
print("Q1语音Best Loss存在：", audio_best_loss_path.exists())

Q1文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
Q1语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
Q1文本Best Loss存在： True
Q1语音Best Loss存在： True


In [130]:
text_cache_candidates = sorted(TEXT_CACHE_ROOT.rglob("302.pt"))
audio_cache_path_302 = AUDIO_CACHE_DIR / "302.pt"

print("找到的302文本缓存：", text_cache_candidates)
print("302语音缓存存在：", audio_cache_path_302.exists())

if len(text_cache_candidates) != 1:
    raise RuntimeError(f"预期找到一个302文本缓存，实际找到{len(text_cache_candidates)}个：{text_cache_candidates}")

TEXT_CACHE_DIR = text_cache_candidates[0].parent

text_cache_302 = torch.load(text_cache_candidates[0], map_location="cpu", weights_only=False)
audio_cache_302 = torch.load(audio_cache_path_302, map_location="cpu", weights_only=False)
text_checkpoint_302 = torch.load(text_best_loss_path, map_location="cpu", weights_only=False)
audio_checkpoint_302 = torch.load(audio_best_loss_path, map_location="cpu", weights_only=False)

print("正式Text缓存目录：", TEXT_CACHE_DIR)
print("Text缓存字段：", list(text_cache_302.keys()))
print("Audio缓存字段：", list(audio_cache_302.keys()))
print("Text Checkpoint字段：", list(text_checkpoint_302.keys()))
print("Audio Checkpoint字段：", list(audio_checkpoint_302.keys()))
print("Text模型前10个参数名：", list(text_checkpoint_302["model_state_dict"].keys())[:10])
print("Audio模型前10个参数名：", list(audio_checkpoint_302["model_state_dict"].keys())[:10])

找到的302文本缓存： [PosixPath('/workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1/302.pt')]
302语音缓存存在： True
正式Text缓存目录： /workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1
Text缓存字段： ['participant_id', 'model_name', 'model_revision', 'max_turns', 'embedding_dim', 'source_size', 'source_mtime_ns', 'real_turn_count', 'embeddings', 'key_padding_mask']
Audio缓存字段： ['participant_id', 'cache_version', 'feature_rate', 'feature_dim', 'max_turns', 'original_transcript_turns', 'cleaned_turns', 'real_turn_count', 'audio_duration_seconds', 'wav_signature', 'transcript_signature', 'egemaps_signature', 'audio_features', 'padding_mask']
Text Checkpoint字段： ['checkpoint_type', 'epoch', 'question_index', 'question_number', 'question_name', 'seed', 'model_state_dict', 'train_metrics', 'val_metrics', 'class_counts', 'class_weights', 'best_val_loss', 'best_val_ccc', 'best_loss_epoch', 'best_ccc_epoch', 'model_name', 'model_revision', 'max_turns', 'embedding_d

In [143]:
train_split_df = pd.read_csv(TRAIN_SPLIT_PATH)
val_split_df = pd.read_csv(VAL_SPLIT_PATH)
test_split_df = pd.read_csv(TEST_SPLIT_PATH)
detailed_label_df = pd.read_csv(DETAILED_LABEL_PATH)

train_metadata_df = train_split_df.merge(detailed_label_df[["Participant_ID"] + PHQ8_QUESTION_COLUMNS], on="Participant_ID", how="left", validate="one_to_one")
val_metadata_df = val_split_df.merge(detailed_label_df[["Participant_ID"] + PHQ8_QUESTION_COLUMNS], on="Participant_ID", how="left", validate="one_to_one")
test_metadata_df = test_split_df.copy()

if train_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise ValueError("训练集存在缺失的8道题标签")

if val_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise ValueError("验证集存在缺失的8道题标签")

print("训练样本数：", len(train_metadata_df))
print("验证样本数：", len(val_metadata_df))
print("测试样本数：", len(test_metadata_df))
print("训练题目标签形状：", train_metadata_df[PHQ8_QUESTION_COLUMNS].shape)
print("验证题目标签形状：", val_metadata_df[PHQ8_QUESTION_COLUMNS].shape)

训练样本数： 163
验证样本数： 55
测试样本数： 56
训练题目标签形状： (163, 8)
验证题目标签形状： (55, 8)


In [144]:
def load_text_audio_cache(participant_id):
    participant_id = int(participant_id)
    text_cache_path = TEXT_CACHE_DIR / f"{participant_id}.pt"
    audio_cache_path = AUDIO_CACHE_DIR / f"{participant_id}.pt"

    if not text_cache_path.exists():
        raise FileNotFoundError(f"文本缓存不存在：{text_cache_path}")

    if not audio_cache_path.exists():
        raise FileNotFoundError(f"语音缓存不存在：{audio_cache_path}")

    text_cache = torch.load(text_cache_path, map_location="cpu", weights_only=False)
    audio_cache = torch.load(audio_cache_path, map_location="cpu", weights_only=False)

    if int(text_cache["participant_id"]) != participant_id:
        raise RuntimeError(f"文本缓存参与者编号错误：{participant_id}")

    if int(audio_cache["participant_id"]) != participant_id:
        raise RuntimeError(f"语音缓存参与者编号错误：{participant_id}")

    text_features = text_cache["embeddings"].float()
    text_padding_mask = text_cache["key_padding_mask"].bool()
    audio_features = audio_cache["audio_features"].float()
    audio_padding_mask = audio_cache["padding_mask"].bool()

    if text_features.shape != (MAX_TURNS, TEXT_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id}文本形状错误：{text_features.shape}")

    if text_padding_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id}文本Mask形状错误：{text_padding_mask.shape}")

    if audio_features.shape != (MAX_TURNS, AUDIO_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id}语音形状错误：{audio_features.shape}")

    if audio_padding_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id}语音Mask形状错误：{audio_padding_mask.shape}")

    if not torch.isfinite(text_features).all():
        raise RuntimeError(f"参与者{participant_id}文本特征包含NaN或Inf")

    if not torch.isfinite(audio_features).all():
        raise RuntimeError(f"参与者{participant_id}语音特征包含NaN或Inf")

    return text_features, text_padding_mask, audio_features, audio_padding_mask

In [145]:
class TextAudioPHQ8Dataset(Dataset):
    def __init__(self, metadata_df, split_name):
        super().__init__()

        if split_name not in {"train", "val"}:
            raise ValueError("split_name只能是train或val")

        self.split_name = split_name
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.labels = torch.tensor(self.metadata_df[PHQ8_QUESTION_COLUMNS].to_numpy(), dtype=torch.long)
        text_feature_list = []
        text_mask_list = []
        audio_feature_list = []
        audio_mask_list = []

        for index, participant_id in enumerate(self.participant_ids):
            text_features, text_mask, audio_features, audio_mask = load_text_audio_cache(participant_id)
            text_feature_list.append(text_features)
            text_mask_list.append(text_mask)
            audio_feature_list.append(audio_features)
            audio_mask_list.append(audio_mask)

            if (index + 1) % 50 == 0 or index + 1 == len(self.participant_ids):
                print(f"{split_name}融合缓存加载进度：{index + 1}/{len(self.participant_ids)}")

        self.text_features = torch.stack(text_feature_list, dim=0)
        self.text_masks = torch.stack(text_mask_list, dim=0)
        self.audio_features = torch.stack(audio_feature_list, dim=0)
        self.audio_masks = torch.stack(audio_mask_list, dim=0)

        if self.text_features.shape != (len(self), MAX_TURNS, TEXT_FEATURE_DIM):
            raise RuntimeError(f"{split_name}文本整体形状错误：{self.text_features.shape}")

        if self.audio_features.shape != (len(self), MAX_TURNS, AUDIO_FEATURE_DIM):
            raise RuntimeError(f"{split_name}语音整体形状错误：{self.audio_features.shape}")

        if self.text_masks.shape != (len(self), MAX_TURNS):
            raise RuntimeError(f"{split_name}文本Mask形状错误：{self.text_masks.shape}")

        if self.audio_masks.shape != (len(self), MAX_TURNS):
            raise RuntimeError(f"{split_name}语音Mask形状错误：{self.audio_masks.shape}")

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        text_features = self.text_features[index]
        text_mask = self.text_masks[index]
        audio_features = self.audio_features[index]
        audio_mask = self.audio_masks[index]
        labels = self.labels[index]
        return text_features, text_mask, audio_features, audio_mask, labels

In [146]:
train_fusion_dataset = TextAudioPHQ8Dataset(train_metadata_df, split_name="train")
val_fusion_dataset = TextAudioPHQ8Dataset(val_metadata_df, split_name="val")

train融合缓存加载进度：50/163
train融合缓存加载进度：100/163
train融合缓存加载进度：150/163
train融合缓存加载进度：163/163
val融合缓存加载进度：50/55
val融合缓存加载进度：55/55


In [147]:
def create_fusion_dataloaders(seed):
    train_generator = torch.Generator()
    train_generator.manual_seed(seed)
    train_loader = DataLoader(train_fusion_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False, generator=train_generator)
    val_loader = DataLoader(val_fusion_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    return train_loader, val_loader

In [148]:
train_fusion_loader, val_fusion_loader = create_fusion_dataloaders(seed=42)
batch_text_features, batch_text_masks, batch_audio_features, batch_audio_masks, batch_all_labels = next(iter(train_fusion_loader))

print("训练样本数：", len(train_fusion_dataset))
print("验证样本数：", len(val_fusion_dataset))
print("训练批次数：", len(train_fusion_loader))
print("验证批次数：", len(val_fusion_loader))
print("Batch文本特征：", batch_text_features.shape)
print("Batch文本Mask：", batch_text_masks.shape)
print("Batch语音特征：", batch_audio_features.shape)
print("Batch语音Mask：", batch_audio_masks.shape)
print("Batch全部标签：", batch_all_labels.shape)
print("当前Batch Q1标签：", batch_all_labels[:, 0])

训练样本数： 163
验证样本数： 55
训练批次数： 17
验证批次数： 6
Batch文本特征： torch.Size([10, 120, 768])
Batch文本Mask： torch.Size([10, 120])
Batch语音特征： torch.Size([10, 120, 23])
Batch语音Mask： torch.Size([10, 120])
Batch全部标签： torch.Size([10, 8])
当前Batch Q1标签： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1])


In [149]:
class PretrainedTextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=TEXT_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.5, batch_first=True)

    def forward(self, text_features, text_padding_mask):
        lstm_output, _ = self.lstm(text_features)
        text_encoding, _ = self.attention(lstm_output, lstm_output, lstm_output, key_padding_mask=text_padding_mask, need_weights=False)
        return text_encoding


class PretrainedAudioEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=AUDIO_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention1 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.attention2 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)

    def forward(self, audio_features, audio_padding_mask):
        lstm_output, _ = self.lstm(audio_features)
        attention1_output, _ = self.attention1(lstm_output, lstm_output, lstm_output, key_padding_mask=audio_padding_mask, need_weights=False)
        audio_encoding, _ = self.attention2(attention1_output, attention1_output, attention1_output, key_padding_mask=audio_padding_mask, need_weights=False)
        return audio_encoding

In [150]:
def extract_state_dict_by_prefixes(full_state_dict, prefixes):
    selected_state_dict = {}

    for parameter_name, parameter_value in full_state_dict.items():
        if any(parameter_name.startswith(prefix) for prefix in prefixes):
            selected_state_dict[parameter_name] = parameter_value

    return selected_state_dict

In [272]:
# def load_pretrained_text_audio_encoders(question_index, seed):
#     text_checkpoint_path = find_checkpoint_by_metadata(TEXT_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=question_index, expected_checkpoint_type="best_loss")
#     audio_checkpoint_path = find_checkpoint_by_metadata(AUDIO_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=question_index, expected_checkpoint_type="best_loss")
#     text_checkpoint = torch.load(text_checkpoint_path, map_location="cpu", weights_only=False)
#     audio_checkpoint = torch.load(audio_checkpoint_path, map_location="cpu", weights_only=False)
def load_pretrained_text_audio_encoders(question_index, seed, checkpoint_type="best_loss"):
    text_checkpoint_path = find_checkpoint_by_metadata(TEXT_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=question_index, expected_checkpoint_type=checkpoint_type)
    audio_checkpoint_path = find_checkpoint_by_metadata(AUDIO_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=question_index, expected_checkpoint_type=checkpoint_type)
    text_checkpoint = torch.load(text_checkpoint_path, map_location="cpu", weights_only=False)
    audio_checkpoint = torch.load(audio_checkpoint_path, map_location="cpu", weights_only=False)
    text_encoder = PretrainedTextEncoder()
    audio_encoder = PretrainedAudioEncoder()
    text_encoder_state = extract_state_dict_by_prefixes(text_checkpoint["model_state_dict"], prefixes=("lstm.", "attention."))
    audio_encoder_state = extract_state_dict_by_prefixes(audio_checkpoint["model_state_dict"], prefixes=("lstm.", "attention1.", "attention2."))
    text_load_result = text_encoder.load_state_dict(text_encoder_state, strict=True)
    audio_load_result = audio_encoder.load_state_dict(audio_encoder_state, strict=True)

    print("文本Best Loss：", text_checkpoint_path)
    print("语音Best Loss：", audio_checkpoint_path)
    print("文本保存Epoch：", text_checkpoint["epoch"])
    print("语音保存Epoch：", audio_checkpoint["epoch"])
    print("文本缺少参数：", text_load_result.missing_keys)
    print("文本多余参数：", text_load_result.unexpected_keys)
    print("语音缺少参数：", audio_load_result.missing_keys)
    print("语音多余参数：", audio_load_result.unexpected_keys)

    return text_encoder, audio_encoder, text_checkpoint, audio_checkpoint, text_checkpoint_path, audio_checkpoint_path

In [154]:
class TextAudioQuestMF(nn.Module):
    def __init__(self, text_encoder, audio_encoder):
        super().__init__()
        self.text_encoder = text_encoder
        self.audio_encoder = audio_encoder
        self.audio_to_text_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_to_audio_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.audio_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Dropout(MLP_FIRST_DROPOUT), nn.Linear(MAX_TURNS * ENCODER_OUTPUT_DIM * 2, 256), nn.ReLU(), nn.Dropout(MLP_LAST_DROPOUT), nn.Linear(256, NUM_CLASSES))

        for parameter in self.text_encoder.parameters():
            parameter.requires_grad = False

        self.text_encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()
        return self

    def forward(self, text_features, text_padding_mask, audio_features, audio_padding_mask, return_intermediates=False):
        text_encoding = self.text_encoder(text_features, text_padding_mask)
        audio_encoding = self.audio_encoder(audio_features, audio_padding_mask)
        text_encoding=text_encoding.masked_fill(text_padding_mask.unsqueeze(-1),0.0)
        audio_encoding=audio_encoding.masked_fill(audio_padding_mask.unsqueeze(-1),0.0)
        audio_to_text, _ = self.audio_to_text_cross_attention(text_encoding, audio_encoding, audio_encoding, key_padding_mask=audio_padding_mask, need_weights=False)
        text_to_audio, _ = self.text_to_audio_cross_attention(audio_encoding, text_encoding, text_encoding, key_padding_mask=text_padding_mask, need_weights=False)
        text_fused_encoding, _ = self.text_self_attention(audio_to_text, audio_to_text, audio_to_text, key_padding_mask=text_padding_mask, need_weights=False)
        audio_fused_encoding, _ = self.audio_self_attention(text_to_audio, text_to_audio, text_to_audio, key_padding_mask=audio_padding_mask, need_weights=False)
        fused_encoding = torch.cat((text_fused_encoding, audio_fused_encoding), dim=2)
        logits = self.mlp(fused_encoding)

        if return_intermediates:
            intermediates = {}
            intermediates["text_encoding"] = text_encoding
            intermediates["audio_encoding"] = audio_encoding
            intermediates["audio_to_text"] = audio_to_text
            intermediates["text_to_audio"] = text_to_audio
            intermediates["text_fused_encoding"] = text_fused_encoding
            intermediates["audio_fused_encoding"] = audio_fused_encoding
            intermediates["fused_encoding"] = fused_encoding
            return logits, intermediates

        return logits

In [155]:
set_seed(42)

(
    q1_text_encoder,
    q1_audio_encoder,
    q1_text_checkpoint,
    q1_audio_checkpoint,
    q1_text_checkpoint_path,
    q1_audio_checkpoint_path
) = load_pretrained_text_audio_encoders(
    question_index=0,
    seed=42
)

q1_fusion_model = TextAudioQuestMF(
    text_encoder=q1_text_encoder,
    audio_encoder=q1_audio_encoder
).to(device)
q1_fusion_model.train()

text_trainable = sum(
    parameter.numel()
    for parameter in q1_fusion_model.text_encoder.parameters()
    if parameter.requires_grad
)

audio_trainable = sum(
    parameter.numel()
    for parameter in q1_fusion_model.audio_encoder.parameters()
    if parameter.requires_grad
)

total_trainable = sum(
    parameter.numel()
    for parameter in q1_fusion_model.parameters()
    if parameter.requires_grad
)

print("文本编码器处于训练模式：",
      q1_fusion_model.text_encoder.training)

print("语音编码器处于训练模式：",
      q1_fusion_model.audio_encoder.training)

print("文本可训练参数：", f"{text_trainable:,}")
print("语音可训练参数：", f"{audio_trainable:,}")
print("模型总可训练参数：", f"{total_trainable:,}")

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
语音保存Epoch： 10
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
文本编码器处于训练模式： False
语音编码器处于训练模式： True
文本可训练参数： 0
语音可训练参数： 110,800
模型总可训练参数： 6,417,684


In [156]:
q1_fusion_model.eval()

with torch.no_grad():
    logits, intermediates = q1_fusion_model(
        text_features=batch_text_features.to(device),
        text_padding_mask=batch_text_masks.to(device),
        audio_features=batch_audio_features.to(device),
        audio_padding_mask=batch_audio_masks.to(device),
        return_intermediates=True
    )

print("Text编码：",
      intermediates["text_encoding"].shape)

print("Audio编码：",
      intermediates["audio_encoding"].shape)

print("Audio → Text：",
      intermediates["audio_to_text"].shape)

print("Text → Audio：",
      intermediates["text_to_audio"].shape)

print("Text融合结果：",
      intermediates["text_fused_encoding"].shape)

print("Audio融合结果：",
      intermediates["audio_fused_encoding"].shape)

print("拼接结果：",
      intermediates["fused_encoding"].shape)

print("Logits：", logits.shape)
print("Logits全部有限：", torch.isfinite(logits).all().item())

Text编码： torch.Size([10, 120, 100])
Audio编码： torch.Size([10, 120, 100])
Audio → Text： torch.Size([10, 120, 100])
Text → Audio： torch.Size([10, 120, 100])
Text融合结果： torch.Size([10, 120, 100])
Audio融合结果： torch.Size([10, 120, 100])
拼接结果： torch.Size([10, 120, 200])
Logits： torch.Size([10, 4])
Logits全部有限： True


In [157]:
text_mask = batch_text_masks.to(device)
audio_mask = batch_audio_masks.to(device)

text_padding_values = intermediates[
    "text_fused_encoding"
].masked_select(
    text_mask.unsqueeze(-1).expand_as(
        intermediates["text_fused_encoding"]
    )
)

audio_padding_values = intermediates[
    "audio_fused_encoding"
].masked_select(
    audio_mask.unsqueeze(-1).expand_as(
        intermediates["audio_fused_encoding"]
    )
)

text_padding_max = (
    text_padding_values.abs().max().item()
    if text_padding_values.numel() > 0
    else 0.0
)

audio_padding_max = (
    audio_padding_values.abs().max().item()
    if audio_padding_values.numel() > 0
    else 0.0
)

print("文本填充位置最大绝对值：", text_padding_max)
print("语音填充位置最大绝对值：", audio_padding_max)

文本填充位置最大绝对值： 0.016936179250478745
语音填充位置最大绝对值： 0.03549167886376381


In [158]:
def gradient_norm(parameters):
    squared_norm_sum = 0.0
    gradient_tensor_count = 0

    for parameter in parameters:
        if parameter.grad is not None:
            if not torch.isfinite(parameter.grad).all():
                raise RuntimeError("发现NaN或Inf梯度")

            squared_norm_sum += (
                parameter.grad.detach().pow(2).sum().item()
            )
            gradient_tensor_count += 1

    return squared_norm_sum ** 0.5, gradient_tensor_count

In [159]:
q1_fusion_model.train()
q1_fusion_model.zero_grad(set_to_none=True)

text_batch = batch_text_features.to(device)
text_mask_batch = batch_text_masks.to(device)
audio_batch = batch_audio_features.to(device)
audio_mask_batch = batch_audio_masks.to(device)

q1_labels = batch_all_labels[:, 0].to(device)

logits = q1_fusion_model(
    text_batch,
    text_mask_batch,
    audio_batch,
    audio_mask_batch
)

diagnostic_loss = F.cross_entropy(logits, q1_labels)
diagnostic_loss.backward()
text_parameters = list(
    q1_fusion_model.text_encoder.parameters()
)

audio_parameters = list(
    q1_fusion_model.audio_encoder.parameters()
)

fusion_parameters = [
    parameter
    for name, parameter in q1_fusion_model.named_parameters()
    if not name.startswith(
        ("text_encoder.", "audio_encoder.")
    )
]

text_grad_norm, text_grad_count = gradient_norm(text_parameters)
audio_grad_norm, audio_grad_count = gradient_norm(audio_parameters)
fusion_grad_norm, fusion_grad_count = gradient_norm(fusion_parameters)

print("诊断损失：", diagnostic_loss.item())

print(
    "文本梯度：",
    text_grad_norm,
    "具有梯度的参数张量数：",
    text_grad_count
)

print(
    "语音梯度：",
    audio_grad_norm,
    "具有梯度的参数张量数：",
    audio_grad_count
)

print(
    "融合层梯度：",
    fusion_grad_norm,
    "具有梯度的参数张量数：",
    fusion_grad_count
)

诊断损失： 1.3675501346588135
文本梯度： 0.0 具有梯度的参数张量数： 0
语音梯度： 0.058201306435654215 具有梯度的参数张量数： 16
融合层梯度： 0.8055883720996126 具有梯度的参数张量数： 20


In [167]:
def calculate_imboll_weights(labels, beta):
    labels = labels.long()
    class_counts = torch.bincount(labels, minlength=NUM_CLASSES)
    if torch.any(class_counts == 0):
        raise ValueError(f"存在没有训练样本的类别：{class_counts.tolist()}")
    inverse_frequency_weights = labels.numel() / class_counts.float()
    imboll_weights = inverse_frequency_weights.pow(beta)
    return class_counts, inverse_frequency_weights, imboll_weights
q1_train_labels = train_fusion_dataset.labels[:, 0]
q1_class_counts, q1_inverse_frequency_weights, q1_imboll_weights = calculate_imboll_weights(q1_train_labels, BETA)
q1_imboll_weights_device = q1_imboll_weights.to(device)

print("Q1类别数量：", q1_class_counts.tolist())
print("Q1原始逆频率权重：", q1_inverse_frequency_weights.tolist())
print("Q1经过Beta后的ImbOLL权重：", q1_imboll_weights.tolist())
print("Q1样本总数：", int(q1_class_counts.sum().item()))

Q1类别数量： [81, 57, 20, 5]
Q1原始逆频率权重： [2.012345790863037, 2.859649181365967, 8.15000057220459, 32.60000228881836]
Q1经过Beta后的ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
Q1样本总数： 163


In [169]:
def build_imboll_distance_matrix(class_weights):
    class_ids = torch.arange(NUM_CLASSES, device=class_weights.device, dtype=class_weights.dtype)
    ordinal_distances = torch.abs(class_ids.unsqueeze(1) - class_ids.unsqueeze(0))
    distance_matrix = class_weights.unsqueeze(1) * ordinal_distances
    return distance_matrix

In [170]:
q1_distance_matrix = build_imboll_distance_matrix(q1_imboll_weights_device)

print("Q1 ImbOLL距离矩阵：")
print(q1_distance_matrix.detach().cpu())

Q1 ImbOLL距离矩阵：
tensor([[ 0.0000,  1.4186,  2.8371,  4.2557],
        [ 1.6910,  0.0000,  1.6910,  3.3821],
        [ 5.7096,  2.8548,  0.0000,  2.8548],
        [17.1289, 11.4193,  5.7096,  0.0000]])


In [172]:
def calculate_imboll_loss(logits, labels, class_weights, alpha, epsilon=LOSS_EPSILON):
    labels = labels.long()
    probabilities = torch.softmax(logits, dim=1)
    class_ids = torch.arange(logits.shape[1], device=logits.device, dtype=torch.long)
    ordinal_distances = torch.abs(labels.unsqueeze(1) - class_ids.unsqueeze(0)).to(logits.dtype)
    true_class_weights = class_weights[labels].unsqueeze(1)
    weighted_distances = (true_class_weights * ordinal_distances).pow(alpha)
    negative_log_complements = -torch.log(torch.clamp(1.0 - probabilities, min=epsilon))
    sample_losses = torch.sum(negative_log_complements * weighted_distances, dim=1)
    loss = sample_losses.mean()
    return loss
example_logits = torch.tensor([[0.2, 0.4, 1.5, -0.3]], dtype=torch.float32, device=device)
example_label = torch.tensor([0], dtype=torch.long, device=device)
example_probabilities = torch.softmax(example_logits, dim=1)
example_loss = calculate_imboll_loss(example_logits, example_label, q1_imboll_weights_device, ALPHA)

print("示例概率：", example_probabilities.detach().cpu())
print("示例真实类别：", example_label.item())
print("示例ImbOLL：", example_loss.item())

示例概率： tensor([[0.1539, 0.1880, 0.5647, 0.0934]])
示例真实类别： 0
示例ImbOLL： 3.0724897384643555


In [173]:
q1_fusion_model.train()
q1_fusion_model.zero_grad(set_to_none=True)

batch_text_features_device = batch_text_features.to(device)
batch_text_masks_device = batch_text_masks.to(device)
batch_audio_features_device = batch_audio_features.to(device)
batch_audio_masks_device = batch_audio_masks.to(device)
batch_q1_labels_device = batch_all_labels[:, 0].to(device)

q1_logits = q1_fusion_model(batch_text_features_device, batch_text_masks_device, batch_audio_features_device, batch_audio_masks_device)
q1_probabilities = torch.softmax(q1_logits, dim=1)
q1_predictions = torch.argmax(q1_logits, dim=1)
q1_imboll_loss = calculate_imboll_loss(q1_logits, batch_q1_labels_device, q1_imboll_weights_device, ALPHA)

print("Logits形状：", q1_logits.shape)
print("ImbOLL：", q1_imboll_loss.item())
print("预测概率全部有限：", torch.isfinite(q1_probabilities).all().item())
print("预测类别：", q1_predictions.detach().cpu())
print("真实类别：", batch_q1_labels_device.detach().cpu())
q1_imboll_loss.backward()

text_grad_norm, text_grad_count = gradient_norm(q1_fusion_model.text_encoder.parameters())
audio_grad_norm, audio_grad_count = gradient_norm(q1_fusion_model.audio_encoder.parameters())
fusion_parameters = [parameter for name, parameter in q1_fusion_model.named_parameters() if not name.startswith(("text_encoder.", "audio_encoder."))]
fusion_grad_norm, fusion_grad_count = gradient_norm(fusion_parameters)

print("文本梯度范数：", text_grad_norm)
print("文本梯度张量数：", text_grad_count)
print("语音梯度范数：", audio_grad_norm)
print("语音梯度张量数：", audio_grad_count)
print("融合层梯度范数：", fusion_grad_norm)
print("融合层梯度张量数：", fusion_grad_count)

Logits形状： torch.Size([10, 4])
ImbOLL： 2.2545089721679688
预测概率全部有限： True
预测类别： tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
真实类别： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1])
文本梯度范数： 0.0
文本梯度张量数： 0
语音梯度范数： 0.0941634756980071
语音梯度张量数： 16
融合层梯度范数： 1.1606160427137056
融合层梯度张量数： 20


In [174]:
def calculate_sample_ccc(predictions, targets, epsilon=LOSS_EPSILON):
    predictions = predictions.float()
    targets = targets.float()
    if predictions.numel() < 2:
        raise ValueError("CCC至少需要两个样本")
    predictions_mean = predictions.mean()
    targets_mean = targets.mean()
    predictions_centered = predictions - predictions_mean
    targets_centered = targets - targets_mean
    denominator_n = predictions.numel() - 1
    covariance = torch.sum(predictions_centered * targets_centered) / denominator_n
    predictions_variance = torch.sum(predictions_centered ** 2) / denominator_n
    targets_variance = torch.sum(targets_centered ** 2) / denominator_n
    ccc = (2.0 * covariance) / (predictions_variance + targets_variance + (predictions_mean - targets_mean) ** 2 + epsilon)
    return ccc
def calculate_multiclass_metrics(predictions, targets, num_classes=NUM_CLASSES):
    predictions = predictions.long()
    targets = targets.long()
    confusion_indices = targets * num_classes + predictions
    confusion_matrix = torch.bincount(confusion_indices, minlength=num_classes * num_classes).reshape(num_classes, num_classes).float()
    true_positives = torch.diag(confusion_matrix)
    false_positives = confusion_matrix.sum(dim=0) - true_positives
    false_negatives = confusion_matrix.sum(dim=1) - true_positives
    supports = confusion_matrix.sum(dim=1)
    precision = true_positives / torch.clamp(true_positives + false_positives, min=1.0)
    recall = true_positives / torch.clamp(true_positives + false_negatives, min=1.0)
    class_f1 = 2.0 * precision * recall / torch.clamp(precision + recall, min=LOSS_EPSILON)
    accuracy = true_positives.sum() / torch.clamp(confusion_matrix.sum(), min=1.0)
    micro_f1 = 2.0 * true_positives.sum() / torch.clamp(2.0 * true_positives.sum() + false_positives.sum() + false_negatives.sum(), min=1.0)
    macro_f1 = class_f1.mean()
    weighted_f1 = torch.sum(class_f1 * supports) / torch.clamp(supports.sum(), min=1.0)
    return {"accuracy": accuracy.item(), "micro_f1": micro_f1.item(), "macro_f1": macro_f1.item(), "weighted_f1": weighted_f1.item(), "confusion_matrix": confusion_matrix.long()}

In [175]:
def run_fusion_epoch(model, dataloader, question_index, class_weights, optimizer=None):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    total_samples = 0
    prediction_batches = []
    target_batches = []
    for batch in dataloader:
        text_features, text_masks, audio_features, audio_masks, all_labels = batch
        text_features = text_features.to(device)
        text_masks = text_masks.to(device)
        audio_features = audio_features.to(device)
        audio_masks = audio_masks.to(device)
        labels = all_labels[:, question_index].to(device)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(text_features, text_masks, audio_features, audio_masks)
            loss = calculate_imboll_loss(logits, labels, class_weights, ALPHA)
            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
        predictions = torch.argmax(logits, dim=1)
        batch_size = labels.shape[0]
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        prediction_batches.append(predictions.detach().cpu())
        target_batches.append(labels.detach().cpu())
    all_predictions = torch.cat(prediction_batches, dim=0)
    all_targets = torch.cat(target_batches, dim=0)
    average_loss = total_loss / total_samples
    classification_metrics = calculate_multiclass_metrics(all_predictions, all_targets)
    ccc = calculate_sample_ccc(all_predictions.float(), all_targets.float()).item()
    rmse = torch.sqrt(torch.mean((all_predictions.float() - all_targets.float()) ** 2)).item()
    mae = torch.mean(torch.abs(all_predictions.float() - all_targets.float())).item()
    return {"loss": average_loss, "accuracy": classification_metrics["accuracy"], "micro_f1": classification_metrics["micro_f1"], "macro_f1": classification_metrics["macro_f1"], "weighted_f1": classification_metrics["weighted_f1"], "ccc": ccc, "rmse": rmse, "mae": mae, "confusion_matrix": classification_metrics["confusion_matrix"], "predictions": all_predictions, "targets": all_targets}

In [176]:
def print_epoch_metrics(prefix, metrics):
    print(f"{prefix} Loss：{metrics['loss']:.6f}")
    print(f"{prefix} Accuracy：{metrics['accuracy']:.6f}")
    print(f"{prefix} Micro F1：{metrics['micro_f1']:.6f}")
    print(f"{prefix} Macro F1：{metrics['macro_f1']:.6f}")
    print(f"{prefix} Weighted F1：{metrics['weighted_f1']:.6f}")
    print(f"{prefix} CCC：{metrics['ccc']:.6f}")
    print(f"{prefix} RMSE：{metrics['rmse']:.6f}")
    print(f"{prefix} MAE：{metrics['mae']:.6f}")
    print(f"{prefix}混淆矩阵：")
    print(metrics["confusion_matrix"])

In [177]:
set_seed(42)

smoke_text_encoder, smoke_audio_encoder, smoke_text_checkpoint, smoke_audio_checkpoint, smoke_text_checkpoint_path, smoke_audio_checkpoint_path = load_pretrained_text_audio_encoders(question_index=0, seed=42)
smoke_model = TextAudioQuestMF(smoke_text_encoder, smoke_audio_encoder).to(device)
smoke_train_loader, smoke_val_loader = create_fusion_dataloaders(seed=42)
smoke_optimizer = torch.optim.AdamW([parameter for parameter in smoke_model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)

smoke_train_metrics = run_fusion_epoch(smoke_model, smoke_train_loader, question_index=0, class_weights=q1_imboll_weights_device, optimizer=smoke_optimizer)
smoke_val_metrics = run_fusion_epoch(smoke_model, smoke_val_loader, question_index=0, class_weights=q1_imboll_weights_device, optimizer=None)

print_epoch_metrics("一轮训练", smoke_train_metrics)
print_epoch_metrics("一轮验证", smoke_val_metrics)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
语音保存Epoch： 10
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
一轮训练 Loss：2.058525
一轮训练 Accuracy：0.435583
一轮训练 Micro F1：0.435583
一轮训练 Macro F1：0.255187
一轮训练 Weighted F1：0.418007
一轮训练 CCC：0.295829
一轮训练 RMSE：0.810211
一轮训练 MAE：0.595092
一轮训练混淆矩阵：
tensor([[43, 38,  0,  0],
        [30, 27,  0,  0],
        [ 1, 18,  1,  0],
        [ 0,  4,  1,  0]])
一轮验证 Loss：2.681503
一轮验证 Accuracy：0.490909
一轮验证 Micro F1：0.490909
一轮验证 Macro F1：0.273366
一轮验证 Weighted F1：0.409380
一轮验证 CCC：0.303162
一轮验证 RMSE：0.981650
一轮验证 MAE：0.636364
一轮验证混淆矩阵：
tensor([[23,  2,  0,  0],
        [15,  3,  4,  0],
        [ 2,  1,  1,  0],
        [ 2,  1,  1,  0]])


In [178]:
assert np.isfinite(smoke_train_metrics["loss"])
assert np.isfinite(smoke_val_metrics["loss"])
assert np.isfinite(smoke_val_metrics["ccc"])
assert 0.0 <= smoke_train_metrics["accuracy"] <= 1.0
assert 0.0 <= smoke_val_metrics["accuracy"] <= 1.0
assert -1.0 <= smoke_val_metrics["ccc"] <= 1.0

print("单Epoch训练与验证检查通过")

单Epoch训练与验证检查通过


In [179]:
def compact_fusion_metrics(metrics):
    compact_metrics = {}
    compact_metrics["loss"] = float(metrics["loss"])
    compact_metrics["accuracy"] = float(metrics["accuracy"])
    compact_metrics["micro_f1"] = float(metrics["micro_f1"])
    compact_metrics["macro_f1"] = float(metrics["macro_f1"])
    compact_metrics["weighted_f1"] = float(metrics["weighted_f1"])
    compact_metrics["ccc"] = float(metrics["ccc"])
    compact_metrics["rmse"] = float(metrics["rmse"])
    compact_metrics["mae"] = float(metrics["mae"])
    compact_metrics["confusion_matrix"] = metrics["confusion_matrix"].tolist()
    return compact_metrics

In [180]:
def create_fusion_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch):
    checkpoint = {}
    checkpoint["checkpoint_type"] = checkpoint_type
    checkpoint["epoch"] = epoch
    checkpoint["question_index"] = question_index
    checkpoint["question_number"] = question_index + 1
    checkpoint["question_name"] = PHQ8_QUESTION_COLUMNS[question_index]
    checkpoint["seed"] = seed
    checkpoint["model_state_dict"] = model.state_dict()
    checkpoint["optimizer_state_dict"] = optimizer.state_dict()
    checkpoint["train_metrics"] = compact_fusion_metrics(train_metrics)
    checkpoint["val_metrics"] = compact_fusion_metrics(val_metrics)
    checkpoint["class_counts"] = class_counts.tolist()
    checkpoint["imboll_weights"] = imboll_weights.tolist()
    checkpoint["best_val_loss"] = best_val_loss
    checkpoint["best_val_ccc"] = best_val_ccc
    checkpoint["best_loss_epoch"] = best_loss_epoch
    checkpoint["best_ccc_epoch"] = best_ccc_epoch
    checkpoint["text_checkpoint_path"] = str(text_checkpoint_path)
    checkpoint["audio_checkpoint_path"] = str(audio_checkpoint_path)
    checkpoint["config"] = {"max_turns": MAX_TURNS, "text_feature_dim": TEXT_FEATURE_DIM, "audio_feature_dim": AUDIO_FEATURE_DIM, "encoder_output_dim": ENCODER_OUTPUT_DIM, "num_classes": NUM_CLASSES, "attention_heads": ATTENTION_HEADS, "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE, "adam_epsilon": ADAM_EPSILON, "weight_decay": WEIGHT_DECAY, "max_grad_norm": MAX_GRAD_NORM, "fusion_dropout": FUSION_DROPOUT, "mlp_first_dropout": MLP_FIRST_DROPOUT, "mlp_last_dropout": MLP_LAST_DROPOUT, "alpha": ALPHA, "beta": BETA}
    return checkpoint

In [181]:
def train_fusion_question(question_index, seed):
    set_seed(seed)
    question_number = question_index + 1
    question_name = PHQ8_QUESTION_COLUMNS[question_index]
    output_dir = FUSION_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_number}"
    output_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = output_dir / "best_loss.pt"
    best_ccc_path = output_dir / "best_ccc.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.csv"
    train_loader, val_loader = create_fusion_dataloaders(seed)
    train_labels = train_fusion_dataset.labels[:, question_index]
    class_counts, inverse_frequency_weights, imboll_weights = calculate_imboll_weights(train_labels, BETA)
    imboll_weights_device = imboll_weights.to(device)
    text_encoder, audio_encoder, text_checkpoint, audio_checkpoint, text_checkpoint_path, audio_checkpoint_path = load_pretrained_text_audio_encoders(question_index=question_index, seed=seed)
    model = TextAudioQuestMF(text_encoder=text_encoder, audio_encoder=audio_encoder).to(device)
    optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = None
    best_ccc_epoch = None
    history_rows = []
    print("=" * 100)
    print(f"开始训练：Seed={seed}，Q{question_number}，{question_name}")
    print("类别数量：", class_counts.tolist())
    print("ImbOLL权重：", imboll_weights.tolist())
    print("=" * 100)
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.time()
        train_metrics = run_fusion_epoch(model, train_loader, question_index, imboll_weights_device, optimizer)
        val_metrics = run_fusion_epoch(model, val_loader, question_index, imboll_weights_device, optimizer=None)
        epoch_seconds = time.time() - epoch_start_time
        history_row = {"Epoch": epoch, "Train_Loss": train_metrics["loss"], "Train_Accuracy": train_metrics["accuracy"], "Train_CCC": train_metrics["ccc"], "Train_RMSE": train_metrics["rmse"], "Train_MAE": train_metrics["mae"], "Val_Loss": val_metrics["loss"], "Val_Accuracy": val_metrics["accuracy"], "Val_Micro_F1": val_metrics["micro_f1"], "Val_Macro_F1": val_metrics["macro_f1"], "Val_Weighted_F1": val_metrics["weighted_f1"], "Val_CCC": val_metrics["ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Seconds": epoch_seconds}
        history_rows.append(history_row)
        if not np.isfinite(val_metrics["loss"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限验证损失")
        if not np.isfinite(val_metrics["ccc"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限CCC")
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_loss_epoch = epoch
            best_loss_checkpoint = create_fusion_checkpoint(model, optimizer, "best_loss", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
            torch.save(best_loss_checkpoint, best_loss_path)
        if val_metrics["ccc"] > best_val_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
            best_ccc_checkpoint = create_fusion_checkpoint(model, optimizer, "best_ccc", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
            torch.save(best_ccc_checkpoint, best_ccc_path)
        last_checkpoint = create_fusion_checkpoint(model, optimizer, "last", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
        torch.save(last_checkpoint, last_path)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_metrics['loss']:.6f} | Val Loss {val_metrics['loss']:.6f} | Val CCC {val_metrics['ccc']:.6f} | Val Acc {val_metrics['accuracy']:.6f} | Val RMSE {val_metrics['rmse']:.6f} | {epoch_seconds:.2f}s")
    result = {"seed": seed, "question_index": question_index, "question_number": question_number, "question_name": question_name, "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}
    del model
    del optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("=" * 100)
    print(f"训练完成：Seed={seed}，Q{question_number}")
    print(f"最佳Val Loss：{best_val_loss:.6f}，Epoch={best_loss_epoch}")
    print(f"最佳Val CCC：{best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    print("=" * 100)
    return result

In [182]:
for variable_name in ["q1_fusion_model", "smoke_model", "smoke_optimizer", "smoke_text_encoder", "smoke_audio_encoder"]:
    globals().pop(variable_name, None)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("冒烟模型已释放")

冒烟模型已释放


In [183]:
q1_seed42_fusion_result = train_fusion_question(question_index=0, seed=42)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
语音保存Epoch： 10
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始训练：Seed=42，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
Epoch 01/20 | Train Loss 2.058525 | Val Loss 2.681503 | Val CCC 0.303162 | Val Acc 0.490909 | Val RMSE 0.981650 | 0.85s
Epoch 02/20 | Train Loss 1.263167 | Val Loss 3.217984 | Val CCC 0.338044 | Val Acc 0.600000 | Val RMSE 0.924416 | 0.80s
Epoch 03/20 | Train Loss 1.358892 | Val Loss 2.383685 | Val CCC 0.405946 | Val Acc 0.563636 | Val RMSE 0.831209 | 0.84s
Epoch 04/20 | Train Loss 1.261398 | Val Loss 2.916068 | Val CCC 0.400501 | Val Acc 0.581818 | Val RMSE 0.797724 | 0.75s
Epoch 05/20 | Train Loss 1.174877 | Val Loss 3.031433 | Val CCC 0.309958 | Val Acc 0.545455 | Val RMSE 0.9

In [184]:
def load_fusion_model_from_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"融合Checkpoint不存在：{checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    required_fields = {"checkpoint_type", "epoch", "question_index", "seed", "model_state_dict", "val_metrics", "imboll_weights"}
    missing_fields = required_fields.difference(checkpoint.keys())
    if missing_fields:
        raise RuntimeError(f"融合Checkpoint缺少字段：{sorted(missing_fields)}")
    text_encoder = PretrainedTextEncoder()
    audio_encoder = PretrainedAudioEncoder()
    model = TextAudioQuestMF(text_encoder=text_encoder, audio_encoder=audio_encoder).to(device)
    load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    print("Checkpoint：", checkpoint_path)
    print("类型：", checkpoint["checkpoint_type"])
    print("Epoch：", checkpoint["epoch"])
    print("缺少参数：", load_result.missing_keys)
    print("多余参数：", load_result.unexpected_keys)
    return model, checkpoint

In [185]:
def verify_fusion_checkpoint(checkpoint_path):
    model, checkpoint = load_fusion_model_from_checkpoint(checkpoint_path)
    question_index = int(checkpoint["question_index"])
    seed = int(checkpoint["seed"])
    imboll_weights = torch.tensor(checkpoint["imboll_weights"], dtype=torch.float32, device=device)
    train_loader, val_loader = create_fusion_dataloaders(seed)
    restored_metrics = run_fusion_epoch(model, val_loader, question_index, imboll_weights, optimizer=None)
    saved_metrics = checkpoint["val_metrics"]
    metric_names = ["loss", "accuracy", "micro_f1", "macro_f1", "weighted_f1", "ccc", "rmse", "mae"]
    print("保存指标与恢复指标对比：")
    for metric_name in metric_names:
        saved_value = float(saved_metrics[metric_name])
        restored_value = float(restored_metrics[metric_name])
        difference = abs(saved_value - restored_value)
        print(f"{metric_name:>12} | 保存 {saved_value:.9f} | 恢复 {restored_value:.9f} | 差值 {difference:.3e}")
        if not np.isclose(saved_value, restored_value, rtol=0.0, atol=1e-6):
            raise RuntimeError(f"{metric_name}恢复后不一致")
    saved_confusion_matrix = torch.tensor(saved_metrics["confusion_matrix"], dtype=torch.long)
    restored_confusion_matrix = restored_metrics["confusion_matrix"]
    if not torch.equal(saved_confusion_matrix, restored_confusion_matrix):
        raise RuntimeError("恢复后的混淆矩阵不一致")
    result = {"checkpoint_type": checkpoint["checkpoint_type"], "epoch": checkpoint["epoch"], "question_index": question_index, "seed": seed, "metrics": restored_metrics}
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Checkpoint恢复验证通过")
    return result

In [186]:
q1_seed42_best_loss_verification = verify_fusion_checkpoint(q1_seed42_fusion_result["best_loss_path"])

Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_1/best_loss.pt
类型： best_loss
Epoch： 3
缺少参数： []
多余参数： []
保存指标与恢复指标对比：
        loss | 保存 2.383685307 | 恢复 2.383685307 | 差值 0.000e+00
    accuracy | 保存 0.563636363 | 恢复 0.563636363 | 差值 0.000e+00
    micro_f1 | 保存 0.563636363 | 恢复 0.563636363 | 差值 0.000e+00
    macro_f1 | 保存 0.365768313 | 恢复 0.365768313 | 差值 0.000e+00
 weighted_f1 | 保存 0.549639344 | 恢复 0.549639344 | 差值 0.000e+00
         ccc | 保存 0.405946046 | 恢复 0.405946046 | 差值 0.000e+00
        rmse | 保存 0.831209421 | 恢复 0.831209421 | 差值 0.000e+00
         mae | 保存 0.509090900 | 恢复 0.509090900 | 差值 0.000e+00
Checkpoint恢复验证通过


In [187]:
q1_seed42_best_ccc_verification = verify_fusion_checkpoint(q1_seed42_fusion_result["best_ccc_path"])

Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_1/best_ccc.pt
类型： best_ccc
Epoch： 12
缺少参数： []
多余参数： []
保存指标与恢复指标对比：
        loss | 保存 2.702756123 | 恢复 2.702756123 | 差值 0.000e+00
    accuracy | 保存 0.545454562 | 恢复 0.545454562 | 差值 0.000e+00
    micro_f1 | 保存 0.545454562 | 恢复 0.545454562 | 差值 0.000e+00
    macro_f1 | 保存 0.352083355 | 恢复 0.352083355 | 差值 0.000e+00
 weighted_f1 | 保存 0.536515176 | 恢复 0.536515176 | 差值 0.000e+00
         ccc | 保存 0.484603375 | 恢复 0.484603375 | 差值 0.000e+00
        rmse | 保存 0.786245406 | 恢复 0.786245406 | 差值 0.000e+00
         mae | 保存 0.509090900 | 恢复 0.509090900 | 差值 0.000e+00
Checkpoint恢复验证通过


In [188]:
seed42_fusion_results = [q1_seed42_fusion_result]

for question_index in range(1, 8):
    question_result = train_fusion_question(question_index=question_index, seed=42)
    seed42_fusion_results.append(question_result)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q2_PHQ_8Depressed/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_2/best_loss.pt
文本保存Epoch： 9
语音保存Epoch： 11
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始训练：Seed=42，Q2，PHQ_8Depressed
类别数量： [68, 63, 18, 14]
ImbOLL权重： [1.548243761062622, 1.6085091829299927, 3.009244918823242, 3.412163257598877]
Epoch 01/20 | Train Loss 2.219187 | Val Loss 2.337435 | Val CCC 0.364986 | Val Acc 0.472727 | Val RMSE 0.990867 | 0.95s
Epoch 02/20 | Train Loss 1.572842 | Val Loss 2.022577 | Val CCC 0.540718 | Val Acc 0.490909 | Val RMSE 0.852803 | 0.81s
Epoch 03/20 | Train Loss 1.679481 | Val Loss 1.995854 | Val CCC 0.536761 | Val Acc 0.527273 | Val RMSE 0.797724 | 1.20s
Epoch 04/20 | Train Loss 1.597517 | Val Loss 1.985951 | Val CCC 0.510547 | Val Acc 0.527273 | Val RMSE 0.831209 | 0.87s
Epoch 05/20 | Train Loss 1.571728 | Val Loss 1.985882 | Val CCC 0.475122 | Val Acc 0.490909 | Val RMSE 0.8842

In [189]:
seed42_summary_rows = []

for result in seed42_fusion_results:
    summary_row = {}
    summary_row["Seed"] = result["seed"]
    summary_row["Question_Index"] = result["question_index"]
    summary_row["Question_Number"] = result["question_number"]
    summary_row["Question_Name"] = result["question_name"]
    summary_row["Best_Val_Loss"] = result["best_val_loss"]
    summary_row["Best_Loss_Epoch"] = result["best_loss_epoch"]
    summary_row["Best_Val_CCC"] = result["best_val_ccc"]
    summary_row["Best_CCC_Epoch"] = result["best_ccc_epoch"]
    summary_row["Best_Loss_Path"] = str(result["best_loss_path"])
    summary_row["Best_CCC_Path"] = str(result["best_ccc_path"])
    seed42_summary_rows.append(summary_row)

seed42_fusion_summary_df = pd.DataFrame(seed42_summary_rows)
seed42_fusion_summary_path = FUSION_CHECKPOINT_ROOT / "seed_42_question_summary.csv"
seed42_fusion_summary_df.to_csv(seed42_fusion_summary_path, index=False)

display(seed42_fusion_summary_df)
print("Seed 42汇总文件：", seed42_fusion_summary_path)

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch,Best_Loss_Path,Best_CCC_Path
0,42,0,1,PHQ_8NoInterest,2.383685,3,0.484603,12,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
1,42,1,2,PHQ_8Depressed,1.985882,5,0.543212,6,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
2,42,2,3,PHQ_8Sleep,2.164561,12,0.452531,4,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
3,42,3,4,PHQ_8Tired,2.170752,15,0.474041,4,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
4,42,4,5,PHQ_8Appetite,2.126681,1,0.522600,7,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
5,42,5,6,PHQ_8Failure,1.952753,12,0.535874,14,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
6,42,6,7,PHQ_8Concentrating,1.952867,4,0.566598,8,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
7,42,7,8,PHQ_8Moving,1.865783,1,0.344404,16,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...


Seed 42汇总文件： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42_question_summary.csv


In [190]:
print("结果数量：", len(seed42_fusion_results))
print("Best Loss全部存在：", all(result["best_loss_path"].exists() for result in seed42_fusion_results))
print("Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in seed42_fusion_results))
print("Last全部存在：", all(result["last_path"].exists() for result in seed42_fusion_results))
print("History全部存在：", all(result["history_path"].exists() for result in seed42_fusion_results))

结果数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True


In [191]:
for result in seed42_fusion_results:
    checkpoint = torch.load(result["best_ccc_path"], map_location="cpu", weights_only=False)
    assert checkpoint["checkpoint_type"] == "best_ccc"
    assert checkpoint["seed"] == 42
    assert checkpoint["question_index"] == result["question_index"]
    print(f"Q{checkpoint['question_number']} | Epoch {checkpoint['epoch']:02d} | Val CCC {checkpoint['val_metrics']['ccc']:.6f} | Val Loss {checkpoint['val_metrics']['loss']:.6f}")

Q1 | Epoch 12 | Val CCC 0.484603 | Val Loss 2.702756
Q2 | Epoch 06 | Val CCC 0.543212 | Val Loss 1.995511
Q3 | Epoch 04 | Val CCC 0.452531 | Val Loss 2.731936
Q4 | Epoch 04 | Val CCC 0.474041 | Val Loss 2.249264
Q5 | Epoch 07 | Val CCC 0.522600 | Val Loss 2.638709
Q6 | Epoch 14 | Val CCC 0.535874 | Val Loss 1.958378
Q7 | Epoch 08 | Val CCC 0.566598 | Val Loss 2.135073
Q8 | Epoch 16 | Val CCC 0.344404 | Val Loss 2.648369


In [192]:
def predict_fusion_question(model, dataloader, question_index):
    model.eval()
    prediction_batches = []
    target_batches = []
    with torch.no_grad():
        for batch in dataloader:
            text_features, text_masks, audio_features, audio_masks, all_labels = batch
            text_features = text_features.to(device)
            text_masks = text_masks.to(device)
            audio_features = audio_features.to(device)
            audio_masks = audio_masks.to(device)
            logits = model(text_features, text_masks, audio_features, audio_masks)
            predictions = torch.argmax(logits, dim=1)
            targets = all_labels[:, question_index]
            prediction_batches.append(predictions.cpu())
            target_batches.append(targets.cpu())
    all_predictions = torch.cat(prediction_batches, dim=0)
    all_targets = torch.cat(target_batches, dim=0)
    return all_predictions, all_targets

In [193]:
def evaluate_fusion_seed_on_validation(seed):
    train_loader, val_loader = create_fusion_dataloaders(seed)
    question_prediction_columns = []
    question_target_columns = []
    question_metric_rows = []
    for question_index in range(8):
        question_number = question_index + 1
        checkpoint_path = FUSION_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_number}" / "best_ccc.pt"
        model, checkpoint = load_fusion_model_from_checkpoint(checkpoint_path)
        if checkpoint["seed"] != seed:
            raise RuntimeError(f"Q{question_number}的Seed不匹配")
        if checkpoint["question_index"] != question_index:
            raise RuntimeError(f"Q{question_number}的题号不匹配")
        predictions, targets = predict_fusion_question(model, val_loader, question_index)
        question_prediction_columns.append(predictions)
        question_target_columns.append(targets)
        question_ccc = calculate_sample_ccc(predictions.float(), targets.float()).item()
        question_rmse = torch.sqrt(torch.mean((predictions.float() - targets.float()) ** 2)).item()
        question_mae = torch.mean(torch.abs(predictions.float() - targets.float())).item()
        question_accuracy = torch.mean((predictions == targets).float()).item()
        question_metric_rows.append({"Question_Number": question_number, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Checkpoint_Epoch": checkpoint["epoch"], "CCC": question_ccc, "RMSE": question_rmse, "MAE": question_mae, "Accuracy": question_accuracy})
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    predicted_question_scores = torch.stack(question_prediction_columns, dim=1)
    true_question_scores = torch.stack(question_target_columns, dim=1)
    expected_true_question_scores = val_fusion_dataset.labels.cpu()
    if not torch.equal(true_question_scores, expected_true_question_scores):
        raise RuntimeError("验证集8道题标签顺序不一致")
    predicted_total_scores = predicted_question_scores.sum(dim=1)
    true_total_scores = true_question_scores.sum(dim=1)
    total_ccc = calculate_sample_ccc(predicted_total_scores.float(), true_total_scores.float()).item()
    total_rmse = torch.sqrt(torch.mean((predicted_total_scores.float() - true_total_scores.float()) ** 2)).item()
    total_mae = torch.mean(torch.abs(predicted_total_scores.float() - true_total_scores.float())).item()
    exact_accuracy = torch.mean((predicted_total_scores == true_total_scores).float()).item()
    question_metrics_df = pd.DataFrame(question_metric_rows)
    prediction_df = pd.DataFrame({"Participant_ID": val_fusion_dataset.participant_ids})
    for question_index in range(8):
        prediction_df[f"Pred_Q{question_index + 1}"] = predicted_question_scores[:, question_index].numpy()
        prediction_df[f"True_Q{question_index + 1}"] = true_question_scores[:, question_index].numpy()
    prediction_df["Predicted_PHQ8_Total"] = predicted_total_scores.numpy()
    prediction_df["True_PHQ8_Total"] = true_total_scores.numpy()
    prediction_df["Error"] = prediction_df["Predicted_PHQ8_Total"] - prediction_df["True_PHQ8_Total"]
    question_metrics_path = FUSION_CHECKPOINT_ROOT / f"seed_{seed}_validation_question_metrics.csv"
    prediction_path = FUSION_CHECKPOINT_ROOT / f"seed_{seed}_validation_predictions.csv"
    question_metrics_df.to_csv(question_metrics_path, index=False)
    prediction_df.to_csv(prediction_path, index=False)
    result = {"seed": seed, "sample_count": len(prediction_df), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "predicted_min": int(predicted_total_scores.min().item()), "predicted_max": int(predicted_total_scores.max().item()), "true_min": int(true_total_scores.min().item()), "true_max": int(true_total_scores.max().item()), "question_metrics_df": question_metrics_df, "prediction_df": prediction_df, "question_metrics_path": question_metrics_path, "prediction_path": prediction_path}
    return result

In [194]:
seed42_fusion_validation_result = evaluate_fusion_seed_on_validation(seed=42)

print("=" * 80)
print("Text + Audio Seed 42验证集总分结果")
print("=" * 80)
print("验证参与者数量：", seed42_fusion_validation_result["sample_count"])
print("CCC：", f"{seed42_fusion_validation_result['ccc']:.6f}")
print("RMSE：", f"{seed42_fusion_validation_result['rmse']:.6f}")
print("MAE：", f"{seed42_fusion_validation_result['mae']:.6f}")
print("完全相等比例：", f"{seed42_fusion_validation_result['exact_accuracy']:.6f}")
print("预测总分范围：", f"{seed42_fusion_validation_result['predicted_min']}～{seed42_fusion_validation_result['predicted_max']}")
print("真实总分范围：", f"{seed42_fusion_validation_result['true_min']}～{seed42_fusion_validation_result['true_max']}")
print("预测文件：", seed42_fusion_validation_result["prediction_path"])

Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_1/best_ccc.pt
类型： best_ccc
Epoch： 12
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_2/best_ccc.pt
类型： best_ccc
Epoch： 6
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_3/best_ccc.pt
类型： best_ccc
Epoch： 4
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_4/best_ccc.pt
类型： best_ccc
Epoch： 4
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_5/best_ccc.pt
类型： best_ccc
Epoch： 7
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_6/best_ccc.pt
类型： best_ccc
Epoch： 14
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_7/best_ccc.pt
类型： best_ccc
Epoch： 8
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-

In [195]:
display(seed42_fusion_validation_result["question_metrics_df"])


,Question_Number,Question_Name,Checkpoint_Epoch,CCC,RMSE,MAE,Accuracy
0,1,PHQ_8NoInterest,12,0.484603,0.786245,0.509091,0.545455
1,2,PHQ_8Depressed,6,0.543212,0.820200,0.527273,0.545455
2,3,PHQ_8Sleep,4,0.452531,1.136182,0.745455,0.472727
3,4,PHQ_8Tired,4,0.474041,0.863397,0.636364,0.418182
4,5,PHQ_8Appetite,7,0.522600,0.924416,0.636364,0.472727
5,6,PHQ_8Failure,14,0.535874,0.894427,0.618182,0.472727
6,7,PHQ_8Concentrating,8,0.566598,0.863397,0.600000,0.472727
7,8,PHQ_8Moving,16,0.344404,0.809040,0.509091,0.563636


In [196]:
print("验证参与者数量：", seed42_fusion_validation_result["sample_count"])
print("总分CCC：", f"{seed42_fusion_validation_result['ccc']:.6f}")
print("总分RMSE：", f"{seed42_fusion_validation_result['rmse']:.6f}")
print("总分MAE：", f"{seed42_fusion_validation_result['mae']:.6f}")
print("完全相等比例：", f"{seed42_fusion_validation_result['exact_accuracy']:.6f}")
print("预测总分范围：", f"{seed42_fusion_validation_result['predicted_min']}～{seed42_fusion_validation_result['predicted_max']}")
print("真实总分范围：", f"{seed42_fusion_validation_result['true_min']}～{seed42_fusion_validation_result['true_max']}")

验证参与者数量： 55
总分CCC： 0.700692
总分RMSE： 4.316986
总分MAE： 3.181818
完全相等比例： 0.090909
预测总分范围： 0～22
真实总分范围： 0～20


In [197]:
def train_all_fusion_questions_for_seed(seed):
    seed_results = []
    for question_index in range(8):
        question_result = train_fusion_question(question_index=question_index, seed=seed)
        seed_results.append(question_result)
    summary_rows = []
    for result in seed_results:
        summary_row = {}
        summary_row["Seed"] = result["seed"]
        summary_row["Question_Index"] = result["question_index"]
        summary_row["Question_Number"] = result["question_number"]
        summary_row["Question_Name"] = result["question_name"]
        summary_row["Best_Val_Loss"] = result["best_val_loss"]
        summary_row["Best_Loss_Epoch"] = result["best_loss_epoch"]
        summary_row["Best_Val_CCC"] = result["best_val_ccc"]
        summary_row["Best_CCC_Epoch"] = result["best_ccc_epoch"]
        summary_row["Best_Loss_Path"] = str(result["best_loss_path"])
        summary_row["Best_CCC_Path"] = str(result["best_ccc_path"])
        summary_rows.append(summary_row)
    summary_df = pd.DataFrame(summary_rows)
    summary_path = FUSION_CHECKPOINT_ROOT / f"seed_{seed}_question_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Seed {seed}全部8道题训练完成")
    print("汇总文件：", summary_path)
    return seed_results, summary_df

In [198]:
seed100_fusion_results, seed100_fusion_summary_df = train_all_fusion_questions_for_seed(seed=100)
display(seed100_fusion_summary_df)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_100/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_1/best_loss.pt
文本保存Epoch： 10
语音保存Epoch： 8
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始训练：Seed=100，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
Epoch 01/20 | Train Loss 2.008674 | Val Loss 3.084688 | Val CCC 0.296198 | Val Acc 0.472727 | Val RMSE 1.018019 | 1.26s
Epoch 02/20 | Train Loss 1.533344 | Val Loss 3.309865 | Val CCC 0.319838 | Val Acc 0.509091 | Val RMSE 1.000000 | 0.79s
Epoch 03/20 | Train Loss 1.570934 | Val Loss 3.004611 | Val CCC 0.287760 | Val Acc 0.454545 | Val RMSE 0.953463 | 0.81s
Epoch 04/20 | Train Loss 1.592438 | Val Loss 2.486881 | Val CCC 0.311383 | Val Acc 0.472727 | Val RMSE 0.914529 | 0.79s
Epoch 05/20 | Train Loss 1.588860 | Val Loss 2.950630 | Val CCC 0.288447 | Val Acc 0.600000 | Val RMSE 0

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch,Best_Loss_Path,Best_CCC_Path
0,100,0,1,PHQ_8NoInterest,2.486881,4,0.417478,18,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
1,100,1,2,PHQ_8Depressed,1.910024,17,0.515920,6,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
2,100,2,3,PHQ_8Sleep,2.216350,3,0.423018,7,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
3,100,3,4,PHQ_8Tired,2.060219,10,0.526778,10,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
4,100,4,5,PHQ_8Appetite,2.294415,11,0.456688,17,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
5,100,5,6,PHQ_8Failure,2.112047,5,0.440976,5,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
6,100,6,7,PHQ_8Concentrating,2.032676,19,0.488424,18,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
7,100,7,8,PHQ_8Moving,2.263026,7,0.296571,20,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...


In [270]:
seed1234_fusion_results, seed1234_fusion_summary_df = train_all_fusion_questions_for_seed(seed=1234)
display(seed1234_fusion_summary_df)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_1234/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_1234/question_1/best_loss.pt
文本保存Epoch： 7
语音保存Epoch： 18
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始训练：Seed=1234，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
Epoch 01/20 | Train Loss 2.162262 | Val Loss 2.525034 | Val CCC 0.175429 | Val Acc 0.527273 | Val RMSE 0.990867 | 0.72s
Epoch 02/20 | Train Loss 1.636145 | Val Loss 3.339831 | Val CCC 0.282505 | Val Acc 0.472727 | Val RMSE 0.990867 | 0.73s
Epoch 03/20 | Train Loss 1.572229 | Val Loss 2.419742 | Val CCC 0.351554 | Val Acc 0.509091 | Val RMSE 0.894427 | 0.73s
Epoch 04/20 | Train Loss 1.515764 | Val Loss 2.677899 | Val CCC 0.406869 | Val Acc 0.527273 | Val RMSE 0.884205 | 0.72s
Epoch 05/20 | Train Loss 1.440375 | Val Loss 2.567060 | Val CCC 0.425187 | Val Acc 0.509091 | Val RMS

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch,Best_Loss_Path,Best_CCC_Path
0,1234,0,1,PHQ_8NoInterest,2.419742,3,0.451380,15,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
1,1234,1,2,PHQ_8Depressed,1.916883,7,0.540718,8,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
2,1234,2,3,PHQ_8Sleep,2.208273,12,0.441932,11,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
3,1234,3,4,PHQ_8Tired,2.091805,14,0.520995,2,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
4,1234,4,5,PHQ_8Appetite,2.010600,12,0.551726,15,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
5,1234,5,6,PHQ_8Failure,2.000219,2,0.488922,8,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
6,1234,6,7,PHQ_8Concentrating,2.022529,10,0.507869,9,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...
7,1234,7,8,PHQ_8Moving,1.793956,19,0.111695,14,/workspace/E-DAIC/checkpoints/phq8_text_audio_...,/workspace/E-DAIC/checkpoints/phq8_text_audio_...


In [200]:
all_fusion_training_results = {42: seed42_fusion_results, 100: seed100_fusion_results, 1234: seed1234_fusion_results}

for seed, seed_results in all_fusion_training_results.items():
    print("=" * 60)
    print("Seed：", seed)
    print("结果数量：", len(seed_results))
    print("Best Loss全部存在：", all(result["best_loss_path"].exists() for result in seed_results))
    print("Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in seed_results))
    print("Last全部存在：", all(result["last_path"].exists() for result in seed_results))
    print("History全部存在：", all(result["history_path"].exists() for result in seed_results))

Seed： 42
结果数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 100
结果数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 1234
结果数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True


In [202]:
seed100_fusion_validation_result = evaluate_fusion_seed_on_validation(seed=100)
seed1234_fusion_validation_result = evaluate_fusion_seed_on_validation(seed=1234)

Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_100/question_1/best_ccc.pt
类型： best_ccc
Epoch： 18
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_100/question_2/best_ccc.pt
类型： best_ccc
Epoch： 6
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_100/question_3/best_ccc.pt
类型： best_ccc
Epoch： 7
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_100/question_4/best_ccc.pt
类型： best_ccc
Epoch： 10
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_100/question_5/best_ccc.pt
类型： best_ccc
Epoch： 17
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_100/question_6/best_ccc.pt
类型： best_ccc
Epoch： 5
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_100/question_7/best_ccc.pt
类型： best_ccc
Epoch： 18
缺少参数： []
多余参数： []
Checkpoint： /wor

In [203]:
fusion_validation_results = [seed42_fusion_validation_result, seed100_fusion_validation_result, seed1234_fusion_validation_result]
fusion_validation_rows = []

for result in fusion_validation_results:
    validation_row = {}
    validation_row["Seed"] = result["seed"]
    validation_row["Sample_Count"] = result["sample_count"]
    validation_row["CCC"] = result["ccc"]
    validation_row["RMSE"] = result["rmse"]
    validation_row["MAE"] = result["mae"]
    validation_row["Exact_Accuracy"] = result["exact_accuracy"]
    validation_row["Predicted_Min"] = result["predicted_min"]
    validation_row["Predicted_Max"] = result["predicted_max"]
    validation_row["True_Min"] = result["true_min"]
    validation_row["True_Max"] = result["true_max"]
    fusion_validation_rows.append(validation_row)

fusion_three_seed_validation_df = pd.DataFrame(fusion_validation_rows)
fusion_three_seed_validation_path = FUSION_CHECKPOINT_ROOT / "three_seed_validation_results.csv"
fusion_three_seed_validation_df.to_csv(fusion_three_seed_validation_path, index=False)

display(fusion_three_seed_validation_df)
print("三Seed验证结果：", fusion_three_seed_validation_path)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy,Predicted_Min,Predicted_Max,True_Min,True_Max
0,42,55,0.700692,4.316986,3.181818,0.090909,0,22,0,20
1,100,55,0.644461,4.767313,3.600000,0.072727,0,20,0,20
2,1234,55,0.636087,4.778741,3.563636,0.163636,0,19,0,20


三Seed验证结果： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/three_seed_validation_results.csv


In [204]:
class TextAudioPHQ8TestDataset(Dataset):
    def __init__(self, metadata_df):
        super().__init__()
        if "Participant_ID" not in metadata_df.columns:
            raise ValueError("测试集缺少Participant_ID列")
        if "PHQ_Score" not in metadata_df.columns:
            raise ValueError("测试集缺少PHQ_Score列")
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.total_scores = torch.tensor(self.metadata_df["PHQ_Score"].to_numpy(), dtype=torch.long)
        text_feature_list = []
        text_mask_list = []
        audio_feature_list = []
        audio_mask_list = []
        for index, participant_id in enumerate(self.participant_ids):
            text_features, text_mask, audio_features, audio_mask = load_text_audio_cache(participant_id)
            text_feature_list.append(text_features)
            text_mask_list.append(text_mask)
            audio_feature_list.append(audio_features)
            audio_mask_list.append(audio_mask)
            if (index + 1) % 20 == 0 or index + 1 == len(self.participant_ids):
                print(f"测试缓存加载进度：{index + 1}/{len(self.participant_ids)}")
        self.text_features = torch.stack(text_feature_list, dim=0)
        self.text_masks = torch.stack(text_mask_list, dim=0)
        self.audio_features = torch.stack(audio_feature_list, dim=0)
        self.audio_masks = torch.stack(audio_mask_list, dim=0)
        if self.text_features.shape != (len(self), MAX_TURNS, TEXT_FEATURE_DIM):
            raise RuntimeError(f"测试文本形状错误：{self.text_features.shape}")
        if self.audio_features.shape != (len(self), MAX_TURNS, AUDIO_FEATURE_DIM):
            raise RuntimeError(f"测试语音形状错误：{self.audio_features.shape}")
    def __len__(self):
        return len(self.participant_ids)
    def __getitem__(self, index):
        return self.text_features[index], self.text_masks[index], self.audio_features[index], self.audio_masks[index], self.total_scores[index]

In [205]:
test_fusion_dataset = TextAudioPHQ8TestDataset(test_metadata_df)
test_fusion_loader = DataLoader(test_fusion_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

test_batch = next(iter(test_fusion_loader))
test_batch_text, test_batch_text_mask, test_batch_audio, test_batch_audio_mask, test_batch_total_scores = test_batch

print("测试样本数：", len(test_fusion_dataset))
print("测试批次数：", len(test_fusion_loader))
print("测试文本形状：", test_batch_text.shape)
print("测试文本Mask形状：", test_batch_text_mask.shape)
print("测试语音形状：", test_batch_audio.shape)
print("测试语音Mask形状：", test_batch_audio_mask.shape)
print("测试总分形状：", test_batch_total_scores.shape)
print("测试真实总分范围：", int(test_fusion_dataset.total_scores.min().item()), "～", int(test_fusion_dataset.total_scores.max().item()))

测试缓存加载进度：20/56
测试缓存加载进度：40/56
测试缓存加载进度：56/56
测试样本数： 56
测试批次数： 6
测试文本形状： torch.Size([10, 120, 768])
测试文本Mask形状： torch.Size([10, 120])
测试语音形状： torch.Size([10, 120, 23])
测试语音Mask形状： torch.Size([10, 120])
测试总分形状： torch.Size([10])
测试真实总分范围： 0 ～ 22


In [206]:
def predict_fusion_question_on_test(model, dataloader):
    model.eval()
    prediction_batches = []
    total_score_batches = []
    with torch.no_grad():
        for batch in dataloader:
            text_features, text_masks, audio_features, audio_masks, total_scores = batch
            text_features = text_features.to(device)
            text_masks = text_masks.to(device)
            audio_features = audio_features.to(device)
            audio_masks = audio_masks.to(device)
            logits = model(text_features, text_masks, audio_features, audio_masks)
            predictions = torch.argmax(logits, dim=1)
            prediction_batches.append(predictions.cpu())
            total_score_batches.append(total_scores.cpu())
    all_predictions = torch.cat(prediction_batches, dim=0)
    all_total_scores = torch.cat(total_score_batches, dim=0)
    return all_predictions, all_total_scores

In [207]:
def evaluate_fusion_seed_on_test(seed):
    question_prediction_columns = []
    reference_total_scores = None
    selected_checkpoint_rows = []
    for question_index in range(8):
        question_number = question_index + 1
        checkpoint_path = FUSION_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_number}" / "best_ccc.pt"
        model, checkpoint = load_fusion_model_from_checkpoint(checkpoint_path)
        if checkpoint["checkpoint_type"] != "best_ccc":
            raise RuntimeError(f"Q{question_number}加载的不是best_ccc")
        if checkpoint["seed"] != seed:
            raise RuntimeError(f"Q{question_number}的Seed不匹配")
        if checkpoint["question_index"] != question_index:
            raise RuntimeError(f"Q{question_number}的题号不匹配")
        predictions, total_scores = predict_fusion_question_on_test(model, test_fusion_loader)
        if reference_total_scores is None:
            reference_total_scores = total_scores
        elif not torch.equal(reference_total_scores, total_scores):
            raise RuntimeError("不同题目读取到的测试标签顺序不一致")
        question_prediction_columns.append(predictions)
        selected_checkpoint_rows.append({"Question_Number": question_number, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Checkpoint_Type": checkpoint["checkpoint_type"], "Checkpoint_Epoch": checkpoint["epoch"], "Validation_CCC": checkpoint["val_metrics"]["ccc"], "Checkpoint_Path": str(checkpoint_path)})
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    predicted_question_scores = torch.stack(question_prediction_columns, dim=1)
    predicted_total_scores = predicted_question_scores.sum(dim=1)
    true_total_scores = reference_total_scores.long()
    total_ccc = calculate_sample_ccc(predicted_total_scores.float(), true_total_scores.float()).item()
    total_rmse = torch.sqrt(torch.mean((predicted_total_scores.float() - true_total_scores.float()) ** 2)).item()
    total_mae = torch.mean(torch.abs(predicted_total_scores.float() - true_total_scores.float())).item()
    exact_accuracy = torch.mean((predicted_total_scores == true_total_scores).float()).item()
    prediction_df = pd.DataFrame({"Participant_ID": test_fusion_dataset.participant_ids})
    for question_index in range(8):
        prediction_df[f"Pred_Q{question_index + 1}"] = predicted_question_scores[:, question_index].numpy()
    prediction_df["Predicted_PHQ8_Total"] = predicted_total_scores.numpy()
    prediction_df["True_PHQ8_Total"] = true_total_scores.numpy()
    prediction_df["Error"] = prediction_df["Predicted_PHQ8_Total"] - prediction_df["True_PHQ8_Total"]
    selected_checkpoint_df = pd.DataFrame(selected_checkpoint_rows)
    prediction_path = FUSION_CHECKPOINT_ROOT / f"seed_{seed}_test_predictions_{len(prediction_df)}.csv"
    selected_checkpoint_path = FUSION_CHECKPOINT_ROOT / f"seed_{seed}_test_selected_checkpoints.csv"
    prediction_df.to_csv(prediction_path, index=False)
    selected_checkpoint_df.to_csv(selected_checkpoint_path, index=False)
    result = {"seed": seed, "sample_count": len(prediction_df), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "predicted_min": int(predicted_total_scores.min().item()), "predicted_max": int(predicted_total_scores.max().item()), "true_min": int(true_total_scores.min().item()), "true_max": int(true_total_scores.max().item()), "prediction_path": prediction_path, "selected_checkpoint_path": selected_checkpoint_path}
    print("=" * 80)
    print(f"Text + Audio Seed {seed}独立测试结果")
    print("=" * 80)
    print("测试参与者数量：", result["sample_count"])
    print("CCC：", f"{result['ccc']:.6f}")
    print("RMSE：", f"{result['rmse']:.6f}")
    print("MAE：", f"{result['mae']:.6f}")
    print("完全相等比例：", f"{result['exact_accuracy']:.6f}")
    print("预测总分范围：", f"{result['predicted_min']}～{result['predicted_max']}")
    print("真实总分范围：", f"{result['true_min']}～{result['true_max']}")
    print("预测文件：", result["prediction_path"])
    return result

In [208]:
seed42_fusion_test_result = evaluate_fusion_seed_on_test(seed=42)
seed100_fusion_test_result = evaluate_fusion_seed_on_test(seed=100)
seed1234_fusion_test_result = evaluate_fusion_seed_on_test(seed=1234)

Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_1/best_ccc.pt
类型： best_ccc
Epoch： 12
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_2/best_ccc.pt
类型： best_ccc
Epoch： 6
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_3/best_ccc.pt
类型： best_ccc
Epoch： 4
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_4/best_ccc.pt
类型： best_ccc
Epoch： 4
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_5/best_ccc.pt
类型： best_ccc
Epoch： 7
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_6/best_ccc.pt
类型： best_ccc
Epoch： 14
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/seed_42/question_7/best_ccc.pt
类型： best_ccc
Epoch： 8
缺少参数： []
多余参数： []
Checkpoint： /workspace/E-

In [209]:
fusion_test_results = [seed42_fusion_test_result, seed100_fusion_test_result, seed1234_fusion_test_result]
fusion_test_rows = []

for result in fusion_test_results:
    test_row = {}
    test_row["Seed"] = result["seed"]
    test_row["Sample_Count"] = result["sample_count"]
    test_row["CCC"] = result["ccc"]
    test_row["RMSE"] = result["rmse"]
    test_row["MAE"] = result["mae"]
    test_row["Exact_Accuracy"] = result["exact_accuracy"]
    test_row["Predicted_Min"] = result["predicted_min"]
    test_row["Predicted_Max"] = result["predicted_max"]
    test_row["True_Min"] = result["true_min"]
    test_row["True_Max"] = result["true_max"]
    fusion_test_rows.append(test_row)

fusion_test_seed_summary_df = pd.DataFrame(fusion_test_rows)
display(fusion_test_seed_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy,Predicted_Min,Predicted_Max,True_Min,True_Max
0,42,56,0.580356,6.209152,4.767857,0.053571,0,23,0,22
1,100,56,0.655986,5.465803,4.303571,0.035714,0,23,0,22
2,1234,56,0.662022,5.203021,3.964286,0.089286,0,20,0,22


In [210]:
fusion_three_seed_rows = []

for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    metric_row = {}
    metric_row["Metric"] = metric_name
    metric_row["Mean"] = fusion_test_seed_summary_df[metric_name].mean()
    metric_row["Std"] = fusion_test_seed_summary_df[metric_name].std(ddof=1)
    fusion_three_seed_rows.append(metric_row)

fusion_three_seed_summary_df = pd.DataFrame(fusion_three_seed_rows)
display(fusion_three_seed_summary_df)

,Metric,Mean,Std
0,CCC,0.632788,0.045508
1,RMSE,5.625992,0.521843
2,MAE,4.345238,0.403403
3,Exact_Accuracy,0.059524,0.027277


In [211]:
fusion_test_seed_summary_path = FUSION_CHECKPOINT_ROOT / "three_seed_test_results_56.csv"
fusion_three_seed_summary_path = FUSION_CHECKPOINT_ROOT / "three_seed_test_summary_56.csv"

fusion_test_seed_summary_df.to_csv(fusion_test_seed_summary_path, index=False)
fusion_three_seed_summary_df.to_csv(fusion_three_seed_summary_path, index=False)

print("逐Seed测试结果：", fusion_test_seed_summary_path)
print("三Seed均值与标准差：", fusion_three_seed_summary_path)
print("逐Seed文件存在：", fusion_test_seed_summary_path.exists())
print("三Seed汇总文件存在：", fusion_three_seed_summary_path.exists())

逐Seed测试结果： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/three_seed_test_results_56.csv
三Seed均值与标准差： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/three_seed_test_summary_56.csv
逐Seed文件存在： True
三Seed汇总文件存在： True


### TCmax

In [212]:
import math


# ==================================================================================================
# Text + Audio TCMax实验配置
# 不修改原始T+A基线配置和输出
# ==================================================================================================

TCMAX_LAMBDA = 0.01
TCMAX_NUM_NEGATIVE_SETS = 1
TCMAX_VALIDATION_SEED = 2026

TCMAX_CHECKPOINT_ROOT = (
    FUSION_CHECKPOINT_ROOT
    / "tcmax_text_audio"
)


# ==================================================================================================
# 计算Text + Audio + Label的TCMax估计
# ==================================================================================================

def calculate_text_audio_tcmax(
    model,
    text_features,
    text_masks,
    audio_features,
    audio_masks,
    labels,
    positive_logits,
    num_negative_sets=TCMAX_NUM_NEGATIVE_SETS,
    permutation_generator=None,
):
    """
    使用DV/TCNE形式估计TC(Text, Audio, Label)。

    正组合：
        (text_i, audio_i, label_i)

    独立负组合：
        (text_perm_t[i], audio_perm_a[i], label_perm_y[i])

    Text、Audio和Label分别独立打乱。

    返回：
        tc_estimate
        tc_loss = -tc_estimate
        positive_term
        negative_term
    """

    if num_negative_sets < 1:
        raise ValueError("num_negative_sets必须大于或等于1")

    labels = labels.long()
    batch_size = labels.shape[0]

    positive_scores = positive_logits.gather(
        dim=1,
        index=labels.unsqueeze(1),
    ).squeeze(1)

    # 最后一个batch可能只有一个样本，此时无法构造有效的独立组合。
    if batch_size < 2:
        zero = positive_scores.sum() * 0.0

        return {
            "tc_estimate": zero,
            "tc_loss": zero,
            "positive_term": zero,
            "negative_term": zero,
            "positive_scores": positive_scores,
            "negative_scores": positive_scores.new_empty(0),
        }

    def create_permutation():
        if permutation_generator is None:
            return torch.randperm(
                batch_size,
                device=labels.device,
            )

        # 验证阶段使用CPU Generator产生固定排列，然后移动到GPU。
        return torch.randperm(
            batch_size,
            generator=permutation_generator,
            device="cpu",
        ).to(labels.device)

    negative_score_batches = []

    for _ in range(num_negative_sets):
        text_indices = create_permutation()
        audio_indices = create_permutation()
        label_indices = create_permutation()

        negative_text_features = text_features[text_indices]
        negative_text_masks = text_masks[text_indices]

        negative_audio_features = audio_features[audio_indices]
        negative_audio_masks = audio_masks[audio_indices]

        negative_labels = labels[label_indices]

        negative_logits = model(
            negative_text_features,
            negative_text_masks,
            negative_audio_features,
            negative_audio_masks,
        )

        negative_scores = negative_logits.gather(
            dim=1,
            index=negative_labels.unsqueeze(1),
        ).squeeze(1)

        negative_score_batches.append(negative_scores)

    all_negative_scores = torch.cat(
        negative_score_batches,
        dim=0,
    )

    positive_term = positive_scores.mean()

    negative_term = (
        torch.logsumexp(all_negative_scores, dim=0)
        - math.log(all_negative_scores.numel())
    )

    tc_estimate = positive_term - negative_term
    tc_loss = -tc_estimate

    return {
        "tc_estimate": tc_estimate,
        "tc_loss": tc_loss,
        "positive_term": positive_term,
        "negative_term": negative_term,
        "positive_scores": positive_scores,
        "negative_scores": all_negative_scores,
    }

In [213]:
# ==================================================================================================
# TCMax版本的单Epoch训练和验证
# ==================================================================================================

def run_fusion_epoch_tcmax(
    model,
    dataloader,
    question_index,
    class_weights,
    optimizer=None,
    lambda_tc=TCMAX_LAMBDA,
    num_negative_sets=TCMAX_NUM_NEGATIVE_SETS,
    measure_validation_tc=True,
):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_objective_loss = 0.0
    total_imboll_loss = 0.0
    total_tc_loss = 0.0
    total_tc_estimate = 0.0
    total_positive_term = 0.0
    total_negative_term = 0.0
    total_samples = 0

    prediction_batches = []
    target_batches = []

    # 验证阶段使用固定排列，避免每个Epoch的Val TC仅因随机打乱而剧烈变化。
    validation_generator = None

    if not is_training and measure_validation_tc:
        validation_generator = torch.Generator(device="cpu")
        validation_generator.manual_seed(TCMAX_VALIDATION_SEED)

    for batch in dataloader:
        (
            text_features,
            text_masks,
            audio_features,
            audio_masks,
            all_labels,
        ) = batch

        text_features = text_features.to(device)
        text_masks = text_masks.to(device)

        audio_features = audio_features.to(device)
        audio_masks = audio_masks.to(device)

        labels = all_labels[:, question_index].to(device).long()

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            positive_logits = model(
                text_features,
                text_masks,
                audio_features,
                audio_masks,
            )

            imboll_loss = calculate_imboll_loss(
                positive_logits,
                labels,
                class_weights,
                ALPHA,
            )

            should_calculate_tc = (
                is_training
                or measure_validation_tc
            )

            if should_calculate_tc:
                tcmax_terms = calculate_text_audio_tcmax(
                    model=model,
                    text_features=text_features,
                    text_masks=text_masks,
                    audio_features=audio_features,
                    audio_masks=audio_masks,
                    labels=labels,
                    positive_logits=positive_logits,
                    num_negative_sets=num_negative_sets,
                    permutation_generator=validation_generator,
                )

                tc_estimate = tcmax_terms["tc_estimate"]
                tc_loss = tcmax_terms["tc_loss"]
                positive_term = tcmax_terms["positive_term"]
                negative_term = tcmax_terms["negative_term"]

            else:
                zero = imboll_loss * 0.0
                tc_estimate = zero
                tc_loss = zero
                positive_term = zero
                negative_term = zero

            total_loss = (
                imboll_loss
                + lambda_tc * tc_loss
            )

            if not torch.isfinite(total_loss):
                raise RuntimeError(
                    f"出现非有限TCMax损失："
                    f"ImbOLL={imboll_loss.detach().item()}，"
                    f"TC={tc_estimate.detach().item()}，"
                    f"Total={total_loss.detach().item()}"
                )

            if is_training:
                total_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                )

                optimizer.step()

        predictions = torch.argmax(
            positive_logits,
            dim=1,
        )

        batch_size = labels.shape[0]

        total_objective_loss += (
            total_loss.detach().item()
            * batch_size
        )

        total_imboll_loss += (
            imboll_loss.detach().item()
            * batch_size
        )

        total_tc_loss += (
            tc_loss.detach().item()
            * batch_size
        )

        total_tc_estimate += (
            tc_estimate.detach().item()
            * batch_size
        )

        total_positive_term += (
            positive_term.detach().item()
            * batch_size
        )

        total_negative_term += (
            negative_term.detach().item()
            * batch_size
        )

        total_samples += batch_size

        prediction_batches.append(
            predictions.detach().cpu()
        )

        target_batches.append(
            labels.detach().cpu()
        )

    all_predictions = torch.cat(
        prediction_batches,
        dim=0,
    )

    all_targets = torch.cat(
        target_batches,
        dim=0,
    )

    average_objective_loss = (
        total_objective_loss
        / total_samples
    )

    average_imboll_loss = (
        total_imboll_loss
        / total_samples
    )

    average_tc_loss = (
        total_tc_loss
        / total_samples
    )

    average_tc_estimate = (
        total_tc_estimate
        / total_samples
    )

    average_positive_term = (
        total_positive_term
        / total_samples
    )

    average_negative_term = (
        total_negative_term
        / total_samples
    )

    classification_metrics = calculate_multiclass_metrics(
        all_predictions,
        all_targets,
    )

    float_predictions = all_predictions.float()
    float_targets = all_targets.float()

    ccc = calculate_sample_ccc(
        float_predictions,
        float_targets,
    ).item()

    rmse = torch.sqrt(
        torch.mean(
            (float_predictions - float_targets) ** 2
        )
    ).item()

    mae = torch.mean(
        torch.abs(
            float_predictions - float_targets
        )
    ).item()

    return {
        # 训练时loss是实际反向传播的总损失；
        # 验证时loss只使用ImbOLL，保证和原始基线公平比较。
        "loss": (
            average_objective_loss
            if is_training
            else average_imboll_loss
        ),
        "total_loss": average_objective_loss,
        "imboll_loss": average_imboll_loss,
        "tc_loss": average_tc_loss,
        "tc_estimate": average_tc_estimate,
        "tc_positive_term": average_positive_term,
        "tc_negative_term": average_negative_term,
        "accuracy": classification_metrics["accuracy"],
        "micro_f1": classification_metrics["micro_f1"],
        "macro_f1": classification_metrics["macro_f1"],
        "weighted_f1": classification_metrics["weighted_f1"],
        "ccc": ccc,
        "rmse": rmse,
        "mae": mae,
        "confusion_matrix": classification_metrics["confusion_matrix"],
        "predictions": all_predictions,
        "targets": all_targets,
    }

In [214]:
# ==================================================================================================
# TCMax检查点
# 先调用原来的create_fusion_checkpoint，再增加TCMax实验信息
# ==================================================================================================

def create_fusion_tcmax_checkpoint(
    model,
    optimizer,
    checkpoint_type,
    epoch,
    question_index,
    seed,
    train_metrics,
    val_metrics,
    class_counts,
    imboll_weights,
    text_checkpoint_path,
    audio_checkpoint_path,
    best_val_loss,
    best_val_ccc,
    best_loss_epoch,
    best_ccc_epoch,
    lambda_tc,
    num_negative_sets,
):
    checkpoint = create_fusion_checkpoint(
        model,
        optimizer,
        checkpoint_type,
        epoch,
        question_index,
        seed,
        train_metrics,
        val_metrics,
        class_counts,
        imboll_weights,
        text_checkpoint_path,
        audio_checkpoint_path,
        best_val_loss,
        best_val_ccc,
        best_loss_epoch,
        best_ccc_epoch,
    )

    checkpoint["experiment_type"] = "text_audio_tcmax"
    checkpoint["tcmax_lambda"] = float(lambda_tc)
    checkpoint["tcmax_num_negative_sets"] = int(
        num_negative_sets
    )
    checkpoint["tcmax_score_function"] = (
        "target_class_logit"
    )
    checkpoint["tcmax_negative_sampling"] = (
        "independent_within_batch_permutation_of_text_audio_label"
    )
    checkpoint["tcmax_validation_selection_loss"] = (
        "imboll_only"
    )

    return checkpoint

In [216]:
# ==================================================================================================
# Text + Audio + TCMax训练函数
# ==================================================================================================
def train_fusion_question_tcmax(question_index, seed, lambda_tc=TCMAX_LAMBDA, num_negative_sets=TCMAX_NUM_NEGATIVE_SETS):
    set_seed(seed)
    question_number = question_index + 1
    question_name = PHQ8_QUESTION_COLUMNS[question_index]
    lambda_tag = f"{lambda_tc:.6g}".replace("-", "m").replace(".", "p")
    output_dir = TCMAX_CHECKPOINT_ROOT / f"lambda_{lambda_tag}_neg_{num_negative_sets}" / f"seed_{seed}" / f"question_{question_number}"
    output_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = output_dir / "best_loss.pt"
    best_ccc_path = output_dir / "best_ccc.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.csv"
    train_loader, val_loader = create_fusion_dataloaders(seed)
    train_labels = train_fusion_dataset.labels[:, question_index]
    class_counts, inverse_frequency_weights, imboll_weights = calculate_imboll_weights(train_labels, BETA)
    imboll_weights_device = imboll_weights.to(device)
    text_encoder, audio_encoder, text_checkpoint, audio_checkpoint, text_checkpoint_path, audio_checkpoint_path = load_pretrained_text_audio_encoders(question_index=question_index, seed=seed)
    model = TextAudioQuestMF(text_encoder=text_encoder, audio_encoder=audio_encoder).to(device)
    optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = None
    best_ccc_epoch = None
    history_rows = []
    print("=" * 100)
    print(f"开始TCMax训练：Seed={seed}，Q{question_number}，{question_name}")
    print("实验类型：Text + Audio + TCMax")
    print(f"TCMax Lambda：{lambda_tc}")
    print(f"每批负组合集合数：{num_negative_sets}")
    print("类别数量：", class_counts.tolist())
    print("ImbOLL权重：", imboll_weights.tolist())
    print("输出目录：", output_dir)
    print("=" * 100)
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.time()
        train_metrics = run_fusion_epoch_tcmax(model=model, dataloader=train_loader, question_index=question_index, class_weights=imboll_weights_device, optimizer=optimizer, lambda_tc=lambda_tc, num_negative_sets=num_negative_sets, measure_validation_tc=False)
        val_metrics = run_fusion_epoch_tcmax(model=model, dataloader=val_loader, question_index=question_index, class_weights=imboll_weights_device, optimizer=None, lambda_tc=lambda_tc, num_negative_sets=num_negative_sets, measure_validation_tc=True)
        epoch_seconds = time.time() - epoch_start_time
        # 验证集选模仍然使用ImbOLL，而不是带随机负组合的TC总目标。
        val_selection_loss = val_metrics["imboll_loss"]
        history_row = {"Epoch": epoch, "Train_Loss": train_metrics["total_loss"], "Train_Total_Loss": train_metrics["total_loss"], "Train_ImbOLL_Loss": train_metrics["imboll_loss"], "Train_TC_Loss": train_metrics["tc_loss"], "Train_TC_Estimate": train_metrics["tc_estimate"], "Train_TC_Positive_Term": train_metrics["tc_positive_term"], "Train_TC_Negative_Term": train_metrics["tc_negative_term"], "Train_Accuracy": train_metrics["accuracy"], "Train_CCC": train_metrics["ccc"], "Train_RMSE": train_metrics["rmse"], "Train_MAE": train_metrics["mae"], "Val_Loss": val_selection_loss, "Val_ImbOLL_Loss": val_metrics["imboll_loss"], "Val_Total_Objective": val_metrics["total_loss"], "Val_TC_Loss": val_metrics["tc_loss"], "Val_TC_Estimate": val_metrics["tc_estimate"], "Val_TC_Positive_Term": val_metrics["tc_positive_term"], "Val_TC_Negative_Term": val_metrics["tc_negative_term"], "Val_Accuracy": val_metrics["accuracy"], "Val_Micro_F1": val_metrics["micro_f1"], "Val_Macro_F1": val_metrics["macro_f1"], "Val_Weighted_F1": val_metrics["weighted_f1"], "Val_CCC": val_metrics["ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "TCMax_Lambda": lambda_tc, "TCMax_Negative_Sets": num_negative_sets, "Seconds": epoch_seconds}
        history_rows.append(history_row)
        if not np.isfinite(val_selection_loss):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限验证ImbOLL")
        if not np.isfinite(val_metrics["ccc"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限CCC")
        if val_selection_loss < best_val_loss:
            best_val_loss = val_selection_loss
            best_loss_epoch = epoch
            best_loss_checkpoint = create_fusion_tcmax_checkpoint(model=model, optimizer=optimizer, checkpoint_type="best_loss", epoch=epoch, question_index=question_index, seed=seed, train_metrics=train_metrics, val_metrics=val_metrics, class_counts=class_counts, imboll_weights=imboll_weights, text_checkpoint_path=text_checkpoint_path, audio_checkpoint_path=audio_checkpoint_path, best_val_loss=best_val_loss, best_val_ccc=best_val_ccc, best_loss_epoch=best_loss_epoch, best_ccc_epoch=best_ccc_epoch, lambda_tc=lambda_tc, num_negative_sets=num_negative_sets)
            torch.save(best_loss_checkpoint, best_loss_path)
        if val_metrics["ccc"] > best_val_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
            best_ccc_checkpoint = create_fusion_tcmax_checkpoint(model=model, optimizer=optimizer, checkpoint_type="best_ccc", epoch=epoch, question_index=question_index, seed=seed, train_metrics=train_metrics, val_metrics=val_metrics, class_counts=class_counts, imboll_weights=imboll_weights, text_checkpoint_path=text_checkpoint_path, audio_checkpoint_path=audio_checkpoint_path, best_val_loss=best_val_loss, best_val_ccc=best_val_ccc, best_loss_epoch=best_loss_epoch, best_ccc_epoch=best_ccc_epoch, lambda_tc=lambda_tc, num_negative_sets=num_negative_sets)
            torch.save(best_ccc_checkpoint, best_ccc_path)
        last_checkpoint = create_fusion_tcmax_checkpoint(model=model, optimizer=optimizer, checkpoint_type="last", epoch=epoch, question_index=question_index, seed=seed, train_metrics=train_metrics, val_metrics=val_metrics, class_counts=class_counts, imboll_weights=imboll_weights, text_checkpoint_path=text_checkpoint_path, audio_checkpoint_path=audio_checkpoint_path, best_val_loss=best_val_loss, best_val_ccc=best_val_ccc, best_loss_epoch=best_loss_epoch, best_ccc_epoch=best_ccc_epoch, lambda_tc=lambda_tc, num_negative_sets=num_negative_sets)
        torch.save(last_checkpoint, last_path)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Total {train_metrics['total_loss']:.6f} | ImbOLL {train_metrics['imboll_loss']:.6f} | TC Est {train_metrics['tc_estimate']:.6f} | Val ImbOLL {val_metrics['imboll_loss']:.6f} | Val TC Est {val_metrics['tc_estimate']:.6f} | Val CCC {val_metrics['ccc']:.6f} | Val Acc {val_metrics['accuracy']:.6f} | Val RMSE {val_metrics['rmse']:.6f} | {epoch_seconds:.2f}s")
    result = {"experiment_type": "text_audio_tcmax", "seed": seed, "question_index": question_index, "question_number": question_number, "question_name": question_name, "lambda_tc": lambda_tc, "num_negative_sets": num_negative_sets, "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}
    del model
    del optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("=" * 100)
    print(f"TCMax训练完成：Seed={seed}，Q{question_number}")
    print(f"最佳Val ImbOLL：{best_val_loss:.6f}，Epoch={best_loss_epoch}")
    print(f"最佳Val CCC：{best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    print("=" * 100)
    return result

In [215]:
# ==================================================================================================
# TCMax烟雾测试：新建模型，不复用已经执行过基线训练的smoke_model
# ==================================================================================================
set_seed(42)

(
    tcmax_smoke_text_encoder,
    tcmax_smoke_audio_encoder,
    tcmax_smoke_text_checkpoint,
    tcmax_smoke_audio_checkpoint,
    tcmax_smoke_text_checkpoint_path,
    tcmax_smoke_audio_checkpoint_path,
) = load_pretrained_text_audio_encoders(question_index=0, seed=42)

tcmax_smoke_model = TextAudioQuestMF(tcmax_smoke_text_encoder, tcmax_smoke_audio_encoder).to(device)

(tcmax_smoke_train_loader, tcmax_smoke_val_loader) = create_fusion_dataloaders(seed=42)

tcmax_smoke_optimizer = torch.optim.AdamW([parameter for parameter in tcmax_smoke_model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)

tcmax_smoke_train_metrics = run_fusion_epoch_tcmax(model=tcmax_smoke_model, dataloader=tcmax_smoke_train_loader, question_index=0, class_weights=q1_imboll_weights_device, optimizer=tcmax_smoke_optimizer, lambda_tc=TCMAX_LAMBDA, num_negative_sets=TCMAX_NUM_NEGATIVE_SETS)

tcmax_smoke_val_metrics = run_fusion_epoch_tcmax(model=tcmax_smoke_model, dataloader=tcmax_smoke_val_loader, question_index=0, class_weights=q1_imboll_weights_device, optimizer=None, lambda_tc=TCMAX_LAMBDA, num_negative_sets=TCMAX_NUM_NEGATIVE_SETS)

print("TCMax一轮训练：")
print({"total_loss": tcmax_smoke_train_metrics["total_loss"], "imboll_loss": tcmax_smoke_train_metrics["imboll_loss"], "tc_loss": tcmax_smoke_train_metrics["tc_loss"], "tc_estimate": tcmax_smoke_train_metrics["tc_estimate"], "ccc": tcmax_smoke_train_metrics["ccc"]})

print("TCMax一轮验证：")
print({"imboll_loss": tcmax_smoke_val_metrics["imboll_loss"], "tc_estimate": tcmax_smoke_val_metrics["tc_estimate"], "ccc": tcmax_smoke_val_metrics["ccc"]})

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
语音保存Epoch： 10
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
TCMax一轮训练：
{'total_loss': 2.062846410493909, 'imboll_loss': 2.060044734755908, 'tc_loss': 0.2801674886440938, 'tc_estimate': -0.2801674886440938, 'ccc': 0.30838555097579956}
TCMax一轮验证：
{'imboll_loss': 2.776166959242387, 'tc_estimate': -0.5609102899377997, 'ccc': 0.29493552446365356}


In [217]:
q1_seed42_tcmax_result = train_fusion_question_tcmax(
    question_index=0,
    seed=42,
    lambda_tc=0.01,
    num_negative_sets=1,
)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
语音保存Epoch： 10
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始TCMax训练：Seed=42，Q1，PHQ_8NoInterest
实验类型：Text + Audio + TCMax
TCMax Lambda：0.01
每批负组合集合数：1
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/tcmax_text_audio/lambda_0p01_neg_1/seed_42/question_1
Epoch 01/20 | Train Total 2.062846 | ImbOLL 2.060045 | TC Est -0.280167 | Val ImbOLL 2.776167 | Val TC Est -0.560910 | Val CCC 0.294936 | Val Acc 0.472727 | Val RMSE 0.990867 | 2.02s
Epoch 02/20 | Train Total 1.395509 | ImbOLL 1.386280 | TC Est -0.922897 | Val ImbOLL 3.017033 | Val TC Est -1.653427 | Val CCC 0.263217 | Val Acc 0.563636 | Val RMSE 0.972345 | 1.68s
Epoch 03/20 | Train Total 1.330751 | ImbOLL 

In [218]:
baseline_history_path = FUSION_CHECKPOINT_ROOT / "seed_42" / "question_1" / "history.csv"
baseline_history = pd.read_csv(baseline_history_path)
tcmax_history = pd.read_csv(q1_seed42_tcmax_result["history_path"])
baseline_best_row = baseline_history.loc[baseline_history["Val_CCC"].idxmax()]
tcmax_best_row = tcmax_history.loc[tcmax_history["Val_CCC"].idxmax()]
comparison = pd.DataFrame([{"Experiment": "T+A Baseline", "Best_Epoch": int(baseline_best_row["Epoch"]), "Best_Val_CCC": baseline_best_row["Val_CCC"], "Val_Loss": baseline_best_row["Val_Loss"], "Val_RMSE": baseline_best_row["Val_RMSE"], "Val_MAE": baseline_best_row["Val_MAE"]}, {"Experiment": "T+A + TCMax", "Best_Epoch": int(tcmax_best_row["Epoch"]), "Best_Val_CCC": tcmax_best_row["Val_CCC"], "Val_Loss": tcmax_best_row["Val_ImbOLL_Loss"], "Val_RMSE": tcmax_best_row["Val_RMSE"], "Val_MAE": tcmax_best_row["Val_MAE"], "Val_TC_Estimate": tcmax_best_row["Val_TC_Estimate"]}])
comparison["CCC_Change_vs_Baseline"] = comparison["Best_Val_CCC"] - comparison.loc[0, "Best_Val_CCC"]
display(comparison)

,Experiment,Best_Epoch,Best_Val_CCC,Val_Loss,Val_RMSE,Val_MAE,Val_TC_Estimate,CCC_Change_vs_Baseline
0,T+A Baseline,12,0.484603,2.702756,0.786245,0.509091,NaN,0.000000
1,T+A + TCMax,19,0.496848,2.489021,0.797724,0.490909,-0.469579,0.012244


In [219]:
display(
    tcmax_history[
        [
            "Epoch",
            "Train_ImbOLL_Loss",
            "Train_TC_Estimate",
            "Val_ImbOLL_Loss",
            "Val_TC_Estimate",
            "Val_CCC",
        ]
    ]
)

,Epoch,Train_ImbOLL_Loss,Train_TC_Estimate,Val_ImbOLL_Loss,Val_TC_Estimate,Val_CCC
0,1,2.060045,-0.280167,2.776167,-0.560910,0.294936
1,2,1.386280,-0.922897,3.017033,-1.653427,0.263217
2,3,1.322582,-0.816907,2.494054,-1.116477,0.354915
3,4,1.222903,-1.119469,2.775958,-1.397750,0.276271
4,5,1.164710,-1.087601,2.564885,-0.928457,0.336296
5,6,1.149168,-0.408108,2.565341,-1.038309,0.335924
6,7,1.138881,-1.361912,3.102482,-2.036326,0.314890
7,8,1.151059,-0.922014,2.527980,-1.057503,0.463934
8,9,1.001203,-0.474595,2.795005,-1.203682,0.459547
9,10,1.008791,-0.520194,3.092823,-1.078212,0.329385


In [222]:
q1_seed123_tcmax_result = train_fusion_question_tcmax(question_index=0, seed=1234, lambda_tc=0.01, num_negative_sets=1)
q1_seed2024_tcmax_result = train_fusion_question_tcmax(question_index=0, seed=100, lambda_tc=0.01, num_negative_sets=1)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_1234/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_1234/question_1/best_loss.pt
文本保存Epoch： 7
语音保存Epoch： 18
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始TCMax训练：Seed=1234，Q1，PHQ_8NoInterest
实验类型：Text + Audio + TCMax
TCMax Lambda：0.01
每批负组合集合数：1
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/tcmax_text_audio/lambda_0p01_neg_1/seed_1234/question_1
Epoch 01/20 | Train Total 2.236861 | ImbOLL 2.234992 | TC Est -0.186889 | Val ImbOLL 2.428502 | Val TC Est -0.153747 | Val CCC 0.184928 | Val Acc 0.563636 | Val RMSE 0.924416 | 2.16s
Epoch 02/20 | Train Total 1.771017 | ImbOLL 1.763589 | TC Est -0.742753 | Val ImbOLL 3.103845 | Val TC Est -2.032110 | Val CCC 0.352747 | Val Acc 0.509091 | Val RMSE 0.924416 | 1.75s
Epoch 03/20 | Train Total 1.671143 | 

In [223]:
baseline_seed42_ccc = pd.read_csv(FUSION_CHECKPOINT_ROOT / "seed_42" / "question_1" / "history.csv")["Val_CCC"].max()
baseline_seed100_ccc = pd.read_csv(FUSION_CHECKPOINT_ROOT / "seed_100" / "question_1" / "history.csv")["Val_CCC"].max()
baseline_seed1234_ccc = pd.read_csv(FUSION_CHECKPOINT_ROOT / "seed_1234" / "question_1" / "history.csv")["Val_CCC"].max()

In [224]:
q1_comparison = pd.DataFrame({"Seed": [42, 100, 1234], "Baseline_CCC": [baseline_seed42_ccc, baseline_seed100_ccc, baseline_seed1234_ccc], "TCMax_CCC": [0.496848, 0.401227, 0.423085]})
q1_comparison["CCC_Change"] = q1_comparison["TCMax_CCC"] - q1_comparison["Baseline_CCC"]
display(q1_comparison)
print("Baseline Mean ± Std：", q1_comparison["Baseline_CCC"].mean(), q1_comparison["Baseline_CCC"].std(ddof=1))
print("TCMax Mean ± Std：", q1_comparison["TCMax_CCC"].mean(), q1_comparison["TCMax_CCC"].std(ddof=1))
print("平均CCC变化：", q1_comparison["CCC_Change"].mean())

,Seed,Baseline_CCC,TCMax_CCC,CCC_Change
0,42,0.484603,0.496848,0.012245
1,100,0.417478,0.401227,-0.016251
2,1234,0.451380,0.423085,-0.028295


Baseline Mean ± Std： 0.451153685649236 0.033563320903678556
TCMax Mean ± Std： 0.44038666666666665 0.050103439625771545
平均CCC变化： -0.01076701898256934


In [225]:
q1_seed42_tcmax_neg4_result = train_fusion_question_tcmax(question_index=0, seed=42, lambda_tc=0.01, num_negative_sets=4)
q1_seed100_tcmax_neg4_result = train_fusion_question_tcmax(question_index=0, seed=100, lambda_tc=0.01, num_negative_sets=4)
q1_seed1234_tcmax_neg4_result = train_fusion_question_tcmax(question_index=0, seed=1234, lambda_tc=0.01, num_negative_sets=4)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
语音保存Epoch： 10
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始TCMax训练：Seed=42，Q1，PHQ_8NoInterest
实验类型：Text + Audio + TCMax
TCMax Lambda：0.01
每批负组合集合数：4
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/tcmax_text_audio/lambda_0p01_neg_4/seed_42/question_1
Epoch 01/20 | Train Total 2.179019 | ImbOLL 2.178170 | TC Est -0.084934 | Val ImbOLL 2.596469 | Val TC Est -0.360573 | Val CCC 0.302873 | Val Acc 0.509091 | Val RMSE 0.972345 | 4.05s
Epoch 02/20 | Train Total 1.272620 | ImbOLL 1.266543 | TC Est -0.607703 | Val ImbOLL 3.152666 | Val TC Est -1.872716 | Val CCC 0.260641 | Val Acc 0.581818 | Val RMSE 0.962950 | 3.55s
Epoch 03/20 | Train Total 1.273550 | ImbOLL 

In [226]:
q1_seed42_tcmax_low_result = train_fusion_question_tcmax(question_index=0, seed=42, lambda_tc=0.001, num_negative_sets=4)
q1_seed100_tcmax_low_result = train_fusion_question_tcmax(question_index=0, seed=100, lambda_tc=0.001, num_negative_sets=4)
q1_seed1234_tcmax_low_result = train_fusion_question_tcmax(question_index=0, seed=1234, lambda_tc=0.001, num_negative_sets=4)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
语音保存Epoch： 10
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始TCMax训练：Seed=42，Q1，PHQ_8NoInterest
实验类型：Text + Audio + TCMax
TCMax Lambda：0.001
每批负组合集合数：4
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/tcmax_text_audio/lambda_0p001_neg_4/seed_42/question_1
Epoch 01/20 | Train Total 2.165002 | ImbOLL 2.164919 | TC Est -0.083103 | Val ImbOLL 2.612851 | Val TC Est -0.429186 | Val CCC 0.294487 | Val Acc 0.490909 | Val RMSE 0.981650 | 3.88s
Epoch 02/20 | Train Total 1.270405 | ImbOLL 1.269646 | TC Est -0.758893 | Val ImbOLL 3.171395 | Val TC Est -2.220035 | Val CCC 0.280751 | Val Acc 0.600000 | Val RMSE 0.953463 | 3.37s
Epoch 03/20 | Train Total 1.279758 | ImbOL

In [227]:
q8_seed42_tcmax_result = train_fusion_question_tcmax(question_index=7, seed=42, lambda_tc=0.01, num_negative_sets=4)
q8_seed100_tcmax_result = train_fusion_question_tcmax(question_index=7, seed=100, lambda_tc=0.01, num_negative_sets=4)
q8_seed1234_tcmax_result = train_fusion_question_tcmax(question_index=7, seed=1234, lambda_tc=0.01, num_negative_sets=4)


文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_loss.pt
文本保存Epoch： 11
语音保存Epoch： 12
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始TCMax训练：Seed=42，Q8，PHQ_8Moving
实验类型：Text + Audio + TCMax
TCMax Lambda：0.01
每批负组合集合数：4
类别数量： [128, 20, 9, 6]
ImbOLL权重： [1.1284668445587158, 2.854820728302002, 4.2557148933410645, 5.212165355682373]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/tcmax_text_audio/lambda_0p01_neg_4/seed_42/question_8
Epoch 01/20 | Train Total 3.058477 | ImbOLL 3.051190 | TC Est -0.728701 | Val ImbOLL 1.806783 | Val TC Est -0.454753 | Val CCC 0.134564 | Val Acc 0.654545 | Val RMSE 0.738549 | 4.25s
Epoch 02/20 | Train Total 1.552104 | ImbOLL 1.533513 | TC Est -1.859126 | Val ImbOLL 2.506559 | Val TC Est -2.925446 | Val CCC 0.303862 | Val Acc 0.563636 | Val RMSE 0.700649 | 3.52s
Epoch 03/20 | Train Total 1.336695 | ImbOLL 1.285849

In [228]:
AUDIO_CONTRAST_LAMBDA = 0.01
AUDIO_CONTRAST_MARGIN = 0.0
AUDIO_CONTRAST_NUM_NEGATIVE_SETS = 1
AUDIO_CONTRAST_VALIDATION_SEED = 2026
AUDIO_CONTRAST_CHECKPOINT_ROOT = FUSION_CHECKPOINT_ROOT / "directional_audio_contrast"

In [229]:
def create_audio_derangement_indices(batch_size, tensor_device, generator=None):
    if batch_size < 2:
        return torch.arange(batch_size, device=tensor_device)
    if generator is None:
        shift = int(torch.randint(1, batch_size, (1,), device=tensor_device).item())
    else:
        shift = int(torch.randint(1, batch_size, (1,), generator=generator, device="cpu").item())
    return torch.roll(torch.arange(batch_size, device=tensor_device), shifts=shift)

In [230]:
def calculate_directional_audio_contrast(model, text_features, text_masks, audio_features, audio_masks, labels, positive_logits, margin=AUDIO_CONTRAST_MARGIN, num_negative_sets=AUDIO_CONTRAST_NUM_NEGATIVE_SETS, permutation_generator=None):
    if num_negative_sets < 1:
        raise ValueError("num_negative_sets必须大于或等于1")
    labels = labels.long()
    batch_size = labels.shape[0]
    positive_log_probabilities = torch.log_softmax(positive_logits, dim=1)
    positive_scores = positive_log_probabilities.gather(dim=1, index=labels.unsqueeze(1)).squeeze(1)
    if batch_size < 2:
        zero = positive_scores.sum() * 0.0
        return {"audio_contrast_loss": zero, "positive_score": zero, "negative_score": zero, "score_gap": zero, "pairwise_accuracy": zero}
    negative_score_batches = []
    for _ in range(num_negative_sets):
        audio_indices = create_audio_derangement_indices(batch_size, labels.device, permutation_generator)
        negative_audio_features = audio_features[audio_indices]
        negative_audio_masks = audio_masks[audio_indices]
        negative_logits = model(text_features, text_masks, negative_audio_features, negative_audio_masks)
        negative_log_probabilities = torch.log_softmax(negative_logits, dim=1)
        negative_scores = negative_log_probabilities.gather(dim=1, index=labels.unsqueeze(1)).squeeze(1)
        negative_score_batches.append(negative_scores)
    all_negative_scores = torch.cat(negative_score_batches, dim=0)
    repeated_positive_scores = positive_scores.repeat(num_negative_sets)
    score_gaps = repeated_positive_scores - all_negative_scores
    audio_contrast_loss = torch.nn.functional.softplus(margin - score_gaps).mean()
    pairwise_accuracy = (score_gaps > 0.0).to(positive_logits.dtype).mean()
    return {"audio_contrast_loss": audio_contrast_loss, "positive_score": repeated_positive_scores.mean(), "negative_score": all_negative_scores.mean(), "score_gap": score_gaps.mean(), "pairwise_accuracy": pairwise_accuracy}

In [231]:
def run_fusion_epoch_audio_contrast(model, dataloader, question_index, class_weights, optimizer=None, lambda_audio=AUDIO_CONTRAST_LAMBDA, margin=AUDIO_CONTRAST_MARGIN, num_negative_sets=AUDIO_CONTRAST_NUM_NEGATIVE_SETS, measure_validation_contrast=True):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()
    total_objective_loss = 0.0
    total_imboll_loss = 0.0
    total_audio_contrast_loss = 0.0
    total_positive_score = 0.0
    total_negative_score = 0.0
    total_score_gap = 0.0
    total_pairwise_accuracy = 0.0
    total_samples = 0
    prediction_batches = []
    target_batches = []
    validation_generator = None
    if not is_training and measure_validation_contrast:
        validation_generator = torch.Generator(device="cpu")
        validation_generator.manual_seed(AUDIO_CONTRAST_VALIDATION_SEED)
    for batch in dataloader:
        text_features, text_masks, audio_features, audio_masks, all_labels = batch
        text_features = text_features.to(device)
        text_masks = text_masks.to(device)
        audio_features = audio_features.to(device)
        audio_masks = audio_masks.to(device)
        labels = all_labels[:, question_index].to(device).long()
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            positive_logits = model(text_features, text_masks, audio_features, audio_masks)
            imboll_loss = calculate_imboll_loss(positive_logits, labels, class_weights, ALPHA)
            should_calculate_contrast = is_training or measure_validation_contrast
            if should_calculate_contrast:
                contrast_terms = calculate_directional_audio_contrast(model, text_features, text_masks, audio_features, audio_masks, labels, positive_logits, margin, num_negative_sets, validation_generator)
                audio_contrast_loss = contrast_terms["audio_contrast_loss"]
                positive_score = contrast_terms["positive_score"]
                negative_score = contrast_terms["negative_score"]
                score_gap = contrast_terms["score_gap"]
                pairwise_accuracy = contrast_terms["pairwise_accuracy"]
            else:
                zero = imboll_loss * 0.0
                audio_contrast_loss = zero
                positive_score = zero
                negative_score = zero
                score_gap = zero
                pairwise_accuracy = zero
            total_loss = imboll_loss + lambda_audio * audio_contrast_loss
            if not torch.isfinite(total_loss):
                raise RuntimeError(f"出现非有限定向Audio损失：ImbOLL={imboll_loss.detach().item()}，AudioContrast={audio_contrast_loss.detach().item()}，Total={total_loss.detach().item()}")
            if is_training:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
        predictions = torch.argmax(positive_logits, dim=1)
        batch_size = labels.shape[0]
        total_objective_loss += total_loss.detach().item() * batch_size
        total_imboll_loss += imboll_loss.detach().item() * batch_size
        total_audio_contrast_loss += audio_contrast_loss.detach().item() * batch_size
        total_positive_score += positive_score.detach().item() * batch_size
        total_negative_score += negative_score.detach().item() * batch_size
        total_score_gap += score_gap.detach().item() * batch_size
        total_pairwise_accuracy += pairwise_accuracy.detach().item() * batch_size
        total_samples += batch_size
        prediction_batches.append(predictions.detach().cpu())
        target_batches.append(labels.detach().cpu())
    all_predictions = torch.cat(prediction_batches, dim=0)
    all_targets = torch.cat(target_batches, dim=0)
    average_objective_loss = total_objective_loss / total_samples
    average_imboll_loss = total_imboll_loss / total_samples
    average_audio_contrast_loss = total_audio_contrast_loss / total_samples
    average_positive_score = total_positive_score / total_samples
    average_negative_score = total_negative_score / total_samples
    average_score_gap = total_score_gap / total_samples
    average_pairwise_accuracy = total_pairwise_accuracy / total_samples
    classification_metrics = calculate_multiclass_metrics(all_predictions, all_targets)
    float_predictions = all_predictions.float()
    float_targets = all_targets.float()
    ccc = calculate_sample_ccc(float_predictions, float_targets).item()
    rmse = torch.sqrt(torch.mean((float_predictions - float_targets) ** 2)).item()
    mae = torch.mean(torch.abs(float_predictions - float_targets)).item()
    return {"loss": average_objective_loss if is_training else average_imboll_loss, "total_loss": average_objective_loss, "imboll_loss": average_imboll_loss, "audio_contrast_loss": average_audio_contrast_loss, "positive_score": average_positive_score, "negative_score": average_negative_score, "score_gap": average_score_gap, "pairwise_accuracy": average_pairwise_accuracy, "accuracy": classification_metrics["accuracy"], "micro_f1": classification_metrics["micro_f1"], "macro_f1": classification_metrics["macro_f1"], "weighted_f1": classification_metrics["weighted_f1"], "ccc": ccc, "rmse": rmse, "mae": mae, "confusion_matrix": classification_metrics["confusion_matrix"], "predictions": all_predictions, "targets": all_targets}

In [232]:
set_seed(42)
audio_contrast_text_encoder, audio_contrast_audio_encoder, _, _, _, _ = load_pretrained_text_audio_encoders(question_index=7, seed=42)
audio_contrast_smoke_model = TextAudioQuestMF(audio_contrast_text_encoder, audio_contrast_audio_encoder).to(device)
audio_contrast_train_loader, audio_contrast_val_loader = create_fusion_dataloaders(seed=42)
q8_train_labels = train_fusion_dataset.labels[:, 7]
_, _, q8_imboll_weights = calculate_imboll_weights(q8_train_labels, BETA)
q8_imboll_weights_device = q8_imboll_weights.to(device)
audio_contrast_smoke_optimizer = torch.optim.AdamW([parameter for parameter in audio_contrast_smoke_model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
audio_contrast_smoke_train_metrics = run_fusion_epoch_audio_contrast(audio_contrast_smoke_model, audio_contrast_train_loader, question_index=7, class_weights=q8_imboll_weights_device, optimizer=audio_contrast_smoke_optimizer)
audio_contrast_smoke_val_metrics = run_fusion_epoch_audio_contrast(audio_contrast_smoke_model, audio_contrast_val_loader, question_index=7, class_weights=q8_imboll_weights_device, optimizer=None)
print({key: audio_contrast_smoke_train_metrics[key] for key in ["total_loss", "imboll_loss", "audio_contrast_loss", "score_gap", "pairwise_accuracy", "ccc"]})
print({key: audio_contrast_smoke_val_metrics[key] for key in ["imboll_loss", "audio_contrast_loss", "score_gap", "pairwise_accuracy", "ccc"]})

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_loss.pt
文本保存Epoch： 11
语音保存Epoch： 12
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
{'total_loss': 2.699870407947002, 'imboll_loss': 2.692877487902261, 'audio_contrast_loss': 0.6992915130100368, 'score_gap': 0.1444248457329792, 'pairwise_accuracy': 0.546012281823012, 'ccc': 0.05228118225932121}
{'imboll_loss': 1.810570164160295, 'audio_contrast_loss': 0.6907699270681902, 'score_gap': 0.04585832763801922, 'pairwise_accuracy': 0.472727278416807, 'ccc': 0.20139183104038239}


In [251]:
def create_fusion_audio_contrast_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, lambda_audio, margin, num_negative_sets):
    checkpoint = create_fusion_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
    checkpoint["experiment_type"] = "text_audio_label_aware_audio_contrast"
    checkpoint["audio_contrast_lambda"] = float(lambda_audio)
    checkpoint["audio_contrast_margin"] = float(margin)
    checkpoint["audio_contrast_num_negative_sets"] = int(num_negative_sets)
    checkpoint["audio_contrast_score_function"] = "target_class_log_probability"
    checkpoint["audio_contrast_text_fixed"] = True
    checkpoint["audio_contrast_label_fixed"] = True
    checkpoint["audio_contrast_negative_sampling"] = "different_label_audio_within_batch"
    return checkpoint

In [279]:
def train_fusion_question_audio_contrast(question_index, seed, lambda_audio=AUDIO_CONTRAST_LAMBDA, margin=AUDIO_CONTRAST_MARGIN, num_negative_sets=AUDIO_CONTRAST_NUM_NEGATIVE_SETS,encoder_checkpoint_type="best_loss"):
    set_seed(seed)
    question_number = question_index + 1
    question_name = PHQ8_QUESTION_COLUMNS[question_index]
    lambda_tag = f"{lambda_audio:.6g}".replace("-", "m").replace(".", "p")
    margin_tag = f"{margin:.6g}".replace("-", "m").replace(".", "p")
    output_dir = AUDIO_CONTRAST_CHECKPOINT_ROOT / f"lambda_{lambda_tag}_margin_{margin_tag}_neg_{num_negative_sets}" / f"seed_{seed}" / f"question_{question_number}"
    output_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = output_dir / "best_loss.pt"
    best_ccc_path = output_dir / "best_ccc.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.csv"
    train_loader, val_loader = create_fusion_dataloaders(seed)
    train_labels = train_fusion_dataset.labels[:, question_index]
    class_counts, inverse_frequency_weights, imboll_weights = calculate_imboll_weights(train_labels, BETA)
    imboll_weights_device = imboll_weights.to(device)
    text_encoder, audio_encoder, text_checkpoint, audio_checkpoint, text_checkpoint_path, audio_checkpoint_path = load_pretrained_text_audio_encoders(question_index=question_index, seed=seed, checkpoint_type=encoder_checkpoint_type)
    if encoder_checkpoint_type == "best_ccc" and text_checkpoint_path.name != "best_ccc.pt":
        raise RuntimeError(f"Text加载的不是best_ccc：{text_checkpoint_path}")
    if encoder_checkpoint_type == "best_ccc" and audio_checkpoint_path.name != "best_ccc.pt":
        raise RuntimeError(f"Audio加载的不是best_ccc：{audio_checkpoint_path}")
    model = TextAudioQuestMF(text_encoder=text_encoder, audio_encoder=audio_encoder).to(device)
    optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = None
    best_ccc_epoch = None
    history_rows = []
    print("=" * 100)
    print(f"开始定向Audio训练：Seed={seed}，Q{question_number}，{question_name}")
    print(f"Lambda={lambda_audio}，Margin={margin}，Negative Sets={num_negative_sets}")
    print("Text固定：True，Label固定：True，只打乱Audio：True")
    print("类别数量：", class_counts.tolist())
    print("输出目录：", output_dir)
    print("=" * 100)
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.time()
        train_metrics = run_fusion_epoch_audio_contrast(model, train_loader, question_index, imboll_weights_device, optimizer, lambda_audio, margin, num_negative_sets, False)
        val_metrics = run_fusion_epoch_audio_contrast(model, val_loader, question_index, imboll_weights_device, None, lambda_audio, margin, num_negative_sets, True)
        epoch_seconds = time.time() - epoch_start_time
        val_selection_loss = val_metrics["imboll_loss"]
        history_row = {"Epoch": epoch, "Train_Loss": train_metrics["total_loss"], "Train_ImbOLL_Loss": train_metrics["imboll_loss"], "Train_Audio_Contrast_Loss": train_metrics["audio_contrast_loss"], "Train_Positive_Score": train_metrics["positive_score"], "Train_Negative_Score": train_metrics["negative_score"], "Train_Score_Gap": train_metrics["score_gap"], "Train_Pairwise_Accuracy": train_metrics["pairwise_accuracy"], "Train_Accuracy": train_metrics["accuracy"], "Train_CCC": train_metrics["ccc"], "Train_RMSE": train_metrics["rmse"], "Train_MAE": train_metrics["mae"], "Val_Loss": val_selection_loss, "Val_ImbOLL_Loss": val_metrics["imboll_loss"], "Val_Total_Objective": val_metrics["total_loss"], "Val_Audio_Contrast_Loss": val_metrics["audio_contrast_loss"], "Val_Positive_Score": val_metrics["positive_score"], "Val_Negative_Score": val_metrics["negative_score"], "Val_Score_Gap": val_metrics["score_gap"], "Val_Pairwise_Accuracy": val_metrics["pairwise_accuracy"], "Val_Accuracy": val_metrics["accuracy"], "Val_Micro_F1": val_metrics["micro_f1"], "Val_Macro_F1": val_metrics["macro_f1"], "Val_Weighted_F1": val_metrics["weighted_f1"], "Val_CCC": val_metrics["ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Audio_Contrast_Lambda": lambda_audio, "Audio_Contrast_Margin": margin, "Audio_Contrast_Negative_Sets": num_negative_sets, "Seconds": epoch_seconds}
        history_row["Encoder_Checkpoint_Type"] = encoder_checkpoint_type
        history_rows.append(history_row)
        if not np.isfinite(val_selection_loss):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限验证ImbOLL")
        if not np.isfinite(val_metrics["ccc"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限CCC")
        if val_selection_loss < best_val_loss:
            best_val_loss = val_selection_loss
            best_loss_epoch = epoch
            best_loss_checkpoint = create_fusion_audio_contrast_checkpoint(model, optimizer, "best_loss", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, lambda_audio, margin, num_negative_sets)
            torch.save(best_loss_checkpoint, best_loss_path)
        if val_metrics["ccc"] > best_val_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
            best_ccc_checkpoint = create_fusion_audio_contrast_checkpoint(model, optimizer, "best_ccc", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, lambda_audio, margin, num_negative_sets)
            torch.save(best_ccc_checkpoint, best_ccc_path)
        last_checkpoint = create_fusion_audio_contrast_checkpoint(model, optimizer, "last", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, lambda_audio, margin, num_negative_sets)
        torch.save(last_checkpoint, last_path)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Total {train_metrics['total_loss']:.6f} | Contrast {train_metrics['audio_contrast_loss']:.6f} | Train Gap {train_metrics['score_gap']:.6f} | Val ImbOLL {val_metrics['imboll_loss']:.6f} | Val Gap {val_metrics['score_gap']:.6f} | Val Pair {val_metrics['pairwise_accuracy']:.6f} | Val CCC {val_metrics['ccc']:.6f} | {epoch_seconds:.2f}s")
    result = {"experiment_type": "text_audio_directional_audio_contrast", "seed": seed, "question_index": question_index, "question_number": question_number, "question_name": question_name, "lambda_audio": lambda_audio, "margin": margin, "num_negative_sets": num_negative_sets, "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}
    result["encoder_checkpoint_type"] = encoder_checkpoint_type
    del model
    del optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("=" * 100)
    print(f"定向Audio训练完成：Seed={seed}，Q{question_number}")
    print(f"最佳Val ImbOLL：{best_val_loss:.6f}，Epoch={best_loss_epoch}")
    print(f"最佳Val CCC：{best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    print("=" * 100)
    return result

In [235]:
del audio_contrast_smoke_model
del audio_contrast_smoke_optimizer
torch.cuda.empty_cache()

In [236]:
q8_seed42_audio_contrast_result = train_fusion_question_audio_contrast(question_index=7, seed=42, lambda_audio=0.01, margin=0.0, num_negative_sets=1)
q8_seed100_audio_contrast_result = train_fusion_question_audio_contrast(question_index=7, seed=100, lambda_audio=0.01, margin=0.0, num_negative_sets=1)
q8_seed1234_audio_contrast_result = train_fusion_question_audio_contrast(question_index=7, seed=1234, lambda_audio=0.01, margin=0.0, num_negative_sets=1)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_loss.pt
文本保存Epoch： 11
语音保存Epoch： 12
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始定向Audio训练：Seed=42，Q8，PHQ_8Moving
Lambda=0.01，Margin=0.0，Negative Sets=1
Text固定：True，Label固定：True，只打乱Audio：True
类别数量： [128, 20, 9, 6]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/directional_audio_contrast/lambda_0p01_margin_0_neg_1/seed_42/question_8
Epoch 01/20 | Train Total 2.699870 | Contrast 0.699292 | Train Gap 0.144425 | Val ImbOLL 1.810570 | Val Gap 0.045858 | Val Pair 0.472727 | Val CCC 0.201392 | 1.66s
Epoch 02/20 | Train Total 1.414516 | Contrast 0.703128 | Train Gap -0.000646 | Val ImbOLL 3.196010 | Val Gap 0.001536 | Val Pair 0.472727 | Val CCC 0.281276 | 1.60s
Epoch 03/20 | Train Total 1.364850 | Contrast 0.702860 | Train Gap -0.000927 | Val ImbOLL 1.932331 | Val Gap 0.003155 | Val Pair 0.527

In [237]:
def get_best_audio_contrast_row(result):
    history = pd.read_csv(result["history_path"])
    return history.loc[history["Val_CCC"].idxmax()]

q8_seed42_audio_row = get_best_audio_contrast_row(q8_seed42_audio_contrast_result)
q8_seed100_audio_row = get_best_audio_contrast_row(q8_seed100_audio_contrast_result)
q8_seed1234_audio_row = get_best_audio_contrast_row(q8_seed1234_audio_contrast_result)

q8_audio_diagnostics = pd.DataFrame([{"Seed": 42, "Val_CCC": q8_seed42_audio_row["Val_CCC"], "Val_Score_Gap": q8_seed42_audio_row["Val_Score_Gap"], "Val_Pairwise_Accuracy": q8_seed42_audio_row["Val_Pairwise_Accuracy"], "Val_Contrast_Loss": q8_seed42_audio_row["Val_Audio_Contrast_Loss"]}, {"Seed": 100, "Val_CCC": q8_seed100_audio_row["Val_CCC"], "Val_Score_Gap": q8_seed100_audio_row["Val_Score_Gap"], "Val_Pairwise_Accuracy": q8_seed100_audio_row["Val_Pairwise_Accuracy"], "Val_Contrast_Loss": q8_seed100_audio_row["Val_Audio_Contrast_Loss"]}, {"Seed": 1234, "Val_CCC": q8_seed1234_audio_row["Val_CCC"], "Val_Score_Gap": q8_seed1234_audio_row["Val_Score_Gap"], "Val_Pairwise_Accuracy": q8_seed1234_audio_row["Val_Pairwise_Accuracy"], "Val_Contrast_Loss": q8_seed1234_audio_row["Val_Audio_Contrast_Loss"]}])

display(q8_audio_diagnostics)

,Seed,Val_CCC,Val_Score_Gap,Val_Pairwise_Accuracy,Val_Contrast_Loss
0,42,0.317345,-0.004759,0.436364,0.695760
1,100,0.260499,0.009566,0.563636,0.688840
2,1234,0.179163,0.009434,0.490909,0.698876


In [238]:
for seed, row in [(42, q8_seed42_audio_row), (100, q8_seed100_audio_row), (1234, q8_seed1234_audio_row)]:
    print({"Seed": seed, "Train_Gap": row["Train_Score_Gap"], "Train_Pair": row["Train_Pairwise_Accuracy"], "Val_Gap": row["Val_Score_Gap"], "Val_Pair": row["Val_Pairwise_Accuracy"], "Val_CCC": row["Val_CCC"]})

{'Seed': 42, 'Train_Gap': np.float64(-0.0207897936982428), 'Train_Pair': np.float64(0.4723926475442991), 'Val_Gap': np.float64(-0.0047587235458195), 'Val_Pair': np.float64(0.4363636401566592), 'Val_CCC': np.float64(0.3173450231552124)}
{'Seed': 100, 'Train_Gap': np.float64(-0.0244682600630298), 'Train_Pair': np.float64(0.4662576785848185), 'Val_Gap': np.float64(0.0095662069506943), 'Val_Pair': np.float64(0.563636376099153), 'Val_CCC': np.float64(0.2604990899562835)}
{'Seed': 1234, 'Train_Gap': np.float64(0.1056207110705368), 'Train_Pair': np.float64(0.5766871260719065), 'Val_Gap': np.float64(0.009434108368375), 'Val_Pair': np.float64(0.4909090995788574), 'Val_CCC': np.float64(0.1791628152132034)}


In [239]:
q8_seed42_audio_contrast_lambda_0p1_result = train_fusion_question_audio_contrast(question_index=7, seed=42, lambda_audio=0.1, margin=0.0, num_negative_sets=1)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_loss.pt
文本保存Epoch： 11
语音保存Epoch： 12
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始定向Audio训练：Seed=42，Q8，PHQ_8Moving
Lambda=0.1，Margin=0.0，Negative Sets=1
Text固定：True，Label固定：True，只打乱Audio：True
类别数量： [128, 20, 9, 6]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/directional_audio_contrast/lambda_0p1_margin_0_neg_1/seed_42/question_8
Epoch 01/20 | Train Total 2.731598 | Contrast 0.701667 | Train Gap 0.150915 | Val ImbOLL 1.805295 | Val Gap 0.050060 | Val Pair 0.472727 | Val CCC 0.201392 | 2.10s
Epoch 02/20 | Train Total 1.519645 | Contrast 0.702638 | Train Gap 0.003532 | Val ImbOLL 2.790766 | Val Gap -0.000325 | Val Pair 0.363636 | Val CCC 0.344404 | 1.63s
Epoch 03/20 | Train Total 1.414559 | Contrast 0.699401 | Train Gap 0.006373 | Val ImbOLL 2.403059 | Val Gap 0.001916 | Val Pair 0.509091

In [240]:
q8_seed42_audio_contrast_lambda_0p1_history = pd.read_csv(q8_seed42_audio_contrast_lambda_0p1_result["history_path"])
display(q8_seed42_audio_contrast_lambda_0p1_history[["Epoch", "Train_Audio_Contrast_Loss", "Train_Score_Gap", "Train_Pairwise_Accuracy", "Val_Audio_Contrast_Loss", "Val_Score_Gap", "Val_Pairwise_Accuracy", "Val_CCC"]])

,Epoch,Train_Audio_Contrast_Loss,Train_Score_Gap,Train_Pairwise_Accuracy,Val_Audio_Contrast_Loss,Val_Score_Gap,Val_Pairwise_Accuracy,Val_CCC
0,1,0.701667,0.150915,0.552147,0.691978,0.050060,0.472727,0.201392
1,2,0.702638,0.003532,0.472393,0.693326,-0.000325,0.363636,0.344404
2,3,0.699401,0.006373,0.490798,0.692204,0.001916,0.509091,0.221589
3,4,0.700308,0.027444,0.539877,0.691958,0.002449,0.527273,0.344404
4,5,0.693940,0.020531,0.539877,0.693261,-0.000203,0.472727,0.244778
5,6,0.702450,-0.004936,0.521472,0.693746,-0.001116,0.436364,0.224752
6,7,0.694470,0.010583,0.466258,0.694146,-0.001674,0.490909,0.218234
7,8,0.677766,0.043858,0.558282,0.694883,-0.002723,0.527273,0.278332
8,9,0.699663,0.006718,0.490798,0.706371,-0.020833,0.545455,0.162897
9,10,0.717324,-0.019803,0.496933,0.688281,0.021138,0.400000,0.266001


In [241]:
print("最大Train Pair：", q8_seed42_audio_contrast_lambda_0p1_history["Train_Pairwise_Accuracy"].max())
print("最大Val Pair：", q8_seed42_audio_contrast_lambda_0p1_history["Val_Pairwise_Accuracy"].max())
print("最后一轮Train Pair：", q8_seed42_audio_contrast_lambda_0p1_history.iloc[-1]["Train_Pairwise_Accuracy"])
print("最后一轮Val Pair：", q8_seed42_audio_contrast_lambda_0p1_history.iloc[-1]["Val_Pairwise_Accuracy"])
print("最佳Val CCC：", q8_seed42_audio_contrast_lambda_0p1_history["Val_CCC"].max())

最大Train Pair： 0.5644171837648732
最大Val Pair： 0.5454545590010557
最后一轮Train Pair： 0.496932523382222
最后一轮Val Pair： 0.4545454613187096
最佳Val CCC： 0.3444039821624756


In [243]:
q8_labels = torch.as_tensor(train_fusion_dataset.labels[:, 7]).long()
q8_class_counts = torch.bincount(q8_labels, minlength=NUM_CLASSES)
q8_sample_count = q8_labels.numel()
print(q8_class_counts)
q8_expected_same_label_rate = (q8_class_counts * (q8_class_counts - 1)).sum().float().div(q8_sample_count * (q8_sample_count - 1)).item()
print("Q8类别数量：", q8_class_counts.tolist())
print("随机打乱后仍为相同标签的概率：", q8_expected_same_label_rate)
print("真正不同标签负样本的概率：", 1.0 - q8_expected_same_label_rate)

tensor([128,  20,   9,   6])
Q8类别数量： [128, 20, 9, 6]
随机打乱后仍为相同标签的概率： 0.6338710784912109
真正不同标签负样本的概率： 0.36612892150878906


In [244]:
def sample_different_label_audio_indices(labels, generator=None):
    batch_size = labels.shape[0]
    different_label_mask = labels.unsqueeze(1) != labels.unsqueeze(0)
    valid_mask = different_label_mask.any(dim=1)
    if generator is None:
        random_scores = torch.rand((batch_size, batch_size), device=labels.device)
    else:
        random_scores = torch.rand((batch_size, batch_size), generator=generator, device="cpu").to(labels.device)
    random_scores = random_scores.masked_fill(~different_label_mask, -1.0)
    negative_indices = torch.argmax(random_scores, dim=1)
    negative_indices = torch.where(valid_mask, negative_indices, torch.arange(batch_size, device=labels.device))
    return negative_indices, valid_mask

In [248]:
q8_sampler_labels = q8_labels.to(device)
audio_indices, valid_mask = sample_different_label_audio_indices(q8_sampler_labels, generator=None)
print("有效样本比例：", valid_mask.float().mean().item())
print("是否全部为不同标签：", torch.all(q8_sampler_labels[audio_indices][valid_mask] != q8_sampler_labels[valid_mask]).item())

有效样本比例： 1.0
是否全部为不同标签： True


In [249]:
def calculate_directional_audio_contrast(model, text_features, text_masks, audio_features, audio_masks, labels, positive_logits, margin=AUDIO_CONTRAST_MARGIN, num_negative_sets=AUDIO_CONTRAST_NUM_NEGATIVE_SETS, permutation_generator=None):
    if num_negative_sets < 1:
        raise ValueError("num_negative_sets必须大于或等于1")
    labels = labels.long()
    positive_log_probabilities = torch.log_softmax(positive_logits, dim=1)
    positive_scores = positive_log_probabilities.gather(dim=1, index=labels.unsqueeze(1)).squeeze(1)
    valid_positive_score_batches = []
    valid_negative_score_batches = []
    for _ in range(num_negative_sets):
        audio_indices, valid_mask = sample_different_label_audio_indices(labels, generator=permutation_generator)
        if not valid_mask.any():
            continue
        negative_audio_features = audio_features[audio_indices]
        negative_audio_masks = audio_masks[audio_indices]
        negative_logits = model(text_features, text_masks, negative_audio_features, negative_audio_masks)
        negative_log_probabilities = torch.log_softmax(negative_logits, dim=1)
        negative_scores = negative_log_probabilities.gather(dim=1, index=labels.unsqueeze(1)).squeeze(1)
        valid_positive_score_batches.append(positive_scores[valid_mask])
        valid_negative_score_batches.append(negative_scores[valid_mask])
    if len(valid_negative_score_batches) == 0:
        zero = positive_scores.sum() * 0.0
        return {"audio_contrast_loss": zero, "positive_score": zero, "negative_score": zero, "score_gap": zero, "pairwise_accuracy": zero}
    all_positive_scores = torch.cat(valid_positive_score_batches, dim=0)
    all_negative_scores = torch.cat(valid_negative_score_batches, dim=0)
    score_gaps = all_positive_scores - all_negative_scores
    audio_contrast_loss = torch.nn.functional.softplus(margin - score_gaps).mean()
    pairwise_accuracy = (score_gaps > 0.0).to(positive_logits.dtype).mean()
    return {"audio_contrast_loss": audio_contrast_loss, "positive_score": all_positive_scores.mean(), "negative_score": all_negative_scores.mean(), "score_gap": score_gaps.mean(), "pairwise_accuracy": pairwise_accuracy}

In [250]:
AUDIO_CONTRAST_CHECKPOINT_ROOT = FUSION_CHECKPOINT_ROOT / "label_aware_directional_audio_contrast"

In [252]:
q8_seed42_label_aware_audio_result = train_fusion_question_audio_contrast(question_index=7, seed=42, lambda_audio=0.1, margin=0.0, num_negative_sets=1)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_loss.pt
文本保存Epoch： 11
语音保存Epoch： 12
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始定向Audio训练：Seed=42，Q8，PHQ_8Moving
Lambda=0.1，Margin=0.0，Negative Sets=1
Text固定：True，Label固定：True，只打乱Audio：True
类别数量： [128, 20, 9, 6]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/label_aware_directional_audio_contrast/lambda_0p1_margin_0_neg_1/seed_42/question_8
Epoch 01/20 | Train Total 3.043979 | Contrast 0.631625 | Train Gap 0.132835 | Val ImbOLL 1.803658 | Val Gap -0.012360 | Val Pair 0.418182 | Val CCC 0.266001 | 2.15s
Epoch 02/20 | Train Total 1.484569 | Contrast 0.616008 | Train Gap 0.030237 | Val ImbOLL 3.521623 | Val Gap -0.000715 | Val Pair 0.400000 | Val CCC 0.283230 | 1.55s
Epoch 03/20 | Train Total 1.542495 | Contrast 0.669183 | Train Gap -0.011670 | Val ImbOLL 2.382593 | Val Gap 0.001091 | Val

In [253]:
q8_seed42_label_aware_history = pd.read_csv(q8_seed42_label_aware_audio_result["history_path"])
q8_seed42_label_aware_best_row = q8_seed42_label_aware_history.loc[q8_seed42_label_aware_history["Val_CCC"].idxmax()]
print("最佳CCC行：", q8_seed42_label_aware_best_row[["Epoch", "Val_CCC", "Train_Score_Gap", "Train_Pairwise_Accuracy", "Val_Score_Gap", "Val_Pairwise_Accuracy", "Val_Audio_Contrast_Loss"]].to_dict())
print("最大Train Pair：", q8_seed42_label_aware_history["Train_Pairwise_Accuracy"].max())
print("最大Val Pair：", q8_seed42_label_aware_history["Val_Pairwise_Accuracy"].max())
print("最后一轮Train Pair：", q8_seed42_label_aware_history.iloc[-1]["Train_Pairwise_Accuracy"])
print("最后一轮Val Pair：", q8_seed42_label_aware_history.iloc[-1]["Val_Pairwise_Accuracy"])
display(q8_seed42_label_aware_history[["Epoch", "Train_Score_Gap", "Train_Pairwise_Accuracy", "Val_Score_Gap", "Val_Pairwise_Accuracy", "Val_Audio_Contrast_Loss", "Val_CCC"]])

最佳CCC行： {'Epoch': 10.0, 'Val_CCC': 0.3444039821624756, 'Train_Score_Gap': -0.0043419964748657, 'Train_Pairwise_Accuracy': 0.4233128931259085, 'Val_Score_Gap': -1.281957917275246e-05, 'Val_Pairwise_Accuracy': 0.5636363679712469, 'Val_Audio_Contrast_Loss': 0.6931559876962141}
最大Train Pair： 0.5460122776177764
最大Val Pair： 0.6909090930765326
最后一轮Train Pair： 0.4785276126642168
最后一轮Val Pair： 0.5636363652619448


,Epoch,Train_Score_Gap,Train_Pairwise_Accuracy,Val_Score_Gap,Val_Pairwise_Accuracy,Val_Audio_Contrast_Loss,Val_CCC
0,1,0.132835,0.447853,-0.012360,0.418182,0.699722,0.266001
1,2,0.030237,0.429448,-0.000715,0.400000,0.693507,0.283230
2,3,-0.011670,0.411043,0.001091,0.436364,0.692642,0.262452
3,4,-0.005951,0.429448,0.000198,0.600000,0.693059,0.269767
4,5,0.032024,0.496933,0.001261,0.636364,0.692535,0.301673
5,6,0.002334,0.478528,0.001592,0.690909,0.692372,0.269767
6,7,0.016916,0.441718,0.001880,0.490909,0.692216,0.268652
7,8,-0.006206,0.472393,0.000914,0.436364,0.692695,0.255943
8,9,-0.024645,0.429448,0.003028,0.600000,0.691682,0.199813
9,10,-0.004342,0.423313,-0.000013,0.563636,0.693156,0.344404


In [254]:
AUDIO_CONTRAST_CHECKPOINT_ROOT = FUSION_CHECKPOINT_ROOT / "label_aware_deterministic_audio_contrast"

In [259]:
def calculate_directional_audio_contrast(model, text_features, text_masks, audio_features, audio_masks, labels, positive_logits, margin=AUDIO_CONTRAST_MARGIN, num_negative_sets=AUDIO_CONTRAST_NUM_NEGATIVE_SETS, permutation_generator=None):
    if num_negative_sets < 1:
        raise ValueError("num_negative_sets必须大于或等于1")
    labels = labels.long()
    sampled_negative_sets = []
    for _ in range(num_negative_sets):
        audio_indices, valid_mask = sample_different_label_audio_indices(labels, generator=permutation_generator)
        if valid_mask.any():
            sampled_negative_sets.append((audio_indices, valid_mask))
    if len(sampled_negative_sets) == 0:
        zero = positive_logits.sum() * 0.0
        return {"audio_contrast_loss": zero, "positive_score": zero, "negative_score": zero, "score_gap": zero, "pairwise_accuracy": zero}
    was_training = model.training
    try:
        if was_training:
            model.eval()
        with torch.backends.cudnn.flags(enabled=False):
            contrast_positive_logits = model(text_features, text_masks, audio_features, audio_masks)
            contrast_positive_log_probabilities = torch.log_softmax(contrast_positive_logits, dim=1)
            contrast_positive_scores = contrast_positive_log_probabilities.gather(dim=1, index=labels.unsqueeze(1)).squeeze(1)
            valid_positive_score_batches = []
            valid_negative_score_batches = []
            for audio_indices, valid_mask in sampled_negative_sets:
                negative_audio_features = audio_features[audio_indices]
                negative_audio_masks = audio_masks[audio_indices]
                contrast_negative_logits = model(text_features, text_masks, negative_audio_features, negative_audio_masks)
                contrast_negative_log_probabilities = torch.log_softmax(contrast_negative_logits, dim=1)
                contrast_negative_scores = contrast_negative_log_probabilities.gather(dim=1, index=labels.unsqueeze(1)).squeeze(1)
                valid_positive_score_batches.append(contrast_positive_scores[valid_mask])
                valid_negative_score_batches.append(contrast_negative_scores[valid_mask])
    finally:
        if was_training:
            model.train()
    all_positive_scores = torch.cat(valid_positive_score_batches, dim=0)
    all_negative_scores = torch.cat(valid_negative_score_batches, dim=0)
    score_gaps = all_positive_scores - all_negative_scores
    audio_contrast_loss = torch.nn.functional.softplus(margin - score_gaps).mean()
    pairwise_accuracy = (score_gaps > 0.0).to(positive_logits.dtype).mean()
    return {"audio_contrast_loss": audio_contrast_loss, "positive_score": all_positive_scores.mean(), "negative_score": all_negative_scores.mean(), "score_gap": score_gaps.mean(), "pairwise_accuracy": pairwise_accuracy}

In [260]:
def create_fusion_audio_contrast_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, lambda_audio, margin, num_negative_sets):
    checkpoint = create_fusion_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
    checkpoint["experiment_type"] = "text_audio_label_aware_deterministic_audio_contrast"
    checkpoint["audio_contrast_lambda"] = float(lambda_audio)
    checkpoint["audio_contrast_margin"] = float(margin)
    checkpoint["audio_contrast_num_negative_sets"] = int(num_negative_sets)
    checkpoint["audio_contrast_score_function"] = "target_class_log_probability"
    checkpoint["audio_contrast_text_fixed"] = True
    checkpoint["audio_contrast_label_fixed"] = True
    checkpoint["audio_contrast_negative_sampling"] = "different_label_audio_within_batch"
    checkpoint["audio_contrast_dropout_mode"] = "disabled_for_positive_and_negative_contrast_forwards"
    return checkpoint

In [280]:
def train_fusion_question_label_aware_deterministic_audio_contrast(question_index, seed, lambda_audio=0.1, margin=0.0, num_negative_sets=1, encoder_checkpoint_type="best_ccc"):
    result = train_fusion_question_audio_contrast(question_index=question_index, seed=seed, lambda_audio=lambda_audio, margin=margin, num_negative_sets=num_negative_sets, encoder_checkpoint_type=encoder_checkpoint_type)
    result["experiment_type"] = "text_audio_label_aware_deterministic_audio_contrast_best_ccc_encoders"
    return result

In [ ]:
q8_seed42_deterministic_audio_result = train_fusion_question_label_aware_deterministic_audio_contrast(question_index=7, seed=42, lambda_audio=0.1, margin=0.0, num_negative_sets=1)

In [263]:
q8_seed42_deterministic_history = pd.read_csv(q8_seed42_deterministic_audio_result["history_path"])
q8_seed42_deterministic_best_row = q8_seed42_deterministic_history.loc[q8_seed42_deterministic_history["Val_CCC"].idxmax()]
print(q8_seed42_deterministic_best_row[["Epoch", "Train_Audio_Contrast_Loss", "Train_Score_Gap", "Train_Pairwise_Accuracy", "Val_Audio_Contrast_Loss", "Val_Score_Gap", "Val_Pairwise_Accuracy", "Val_CCC"]].to_dict())
display(q8_seed42_deterministic_history[["Epoch", "Train_Audio_Contrast_Loss", "Train_Score_Gap", "Train_Pairwise_Accuracy", "Val_Audio_Contrast_Loss", "Val_Score_Gap", "Val_Pairwise_Accuracy", "Val_CCC"]])

{'Epoch': 12.0, 'Train_Audio_Contrast_Loss': 0.6365378758658661, 'Train_Score_Gap': 0.0027376376492388, 'Train_Pairwise_Accuracy': 0.5705521607691525, 'Val_Audio_Contrast_Loss': 0.6924244815653021, 'Val_Score_Gap': 0.0015122308501635, 'Val_Pairwise_Accuracy': 0.6727272868156433, 'Val_CCC': 0.3598466515541076}


,Epoch,Train_Audio_Contrast_Loss,Train_Score_Gap,Train_Pairwise_Accuracy,Val_Audio_Contrast_Loss,Val_Score_Gap,Val_Pairwise_Accuracy,Val_CCC
0,1,0.609594,0.113061,0.595092,0.698709,-0.010470,0.436364,0.223922
1,2,0.608643,0.008510,0.355828,0.693188,-0.000050,0.509091,0.267135
2,3,0.649973,0.001431,0.380368,0.692683,0.001134,0.472727,0.229003
3,4,0.636793,0.002608,0.355828,0.693657,-0.001000,0.581818,0.317577
4,5,0.638490,-0.001023,0.546012,0.692237,0.001889,0.600000,0.172821
5,6,0.679299,0.002430,0.472393,0.693228,0.000150,0.345455,0.267719
6,7,0.593545,0.003927,0.312883,0.694040,-0.001682,0.327273,0.206927
7,8,0.693072,0.000286,0.374233,0.693400,-0.000448,0.400000,0.163165
8,9,0.650173,0.001008,0.386503,0.692025,0.002293,0.618182,0.199813
9,10,0.606559,0.003212,0.619632,0.692525,0.001317,0.618182,0.317345


In [264]:
q8_seed42_baseline_history = pd.read_csv(FUSION_CHECKPOINT_ROOT / "seed_42" / "question_8" / "history.csv")
q8_seed42_baseline_ccc = q8_seed42_baseline_history["Val_CCC"].max()
print("Q8 Seed 42基线CCC：", q8_seed42_baseline_ccc)
print("确定性Audio CCC：", q8_seed42_deterministic_history["Val_CCC"].max())
print("CCC变化：", q8_seed42_deterministic_history["Val_CCC"].max() - q8_seed42_baseline_ccc)

Q8 Seed 42基线CCC： 0.3444039821624756
确定性Audio CCC： 0.3598466515541076
CCC变化： 0.015442669391632025


In [265]:
q8_seed100_deterministic_audio_result = train_fusion_question_label_aware_deterministic_audio_contrast(question_index=7, seed=100, lambda_audio=0.1, margin=0.0, num_negative_sets=1)
q8_seed1234_deterministic_audio_result = train_fusion_question_label_aware_deterministic_audio_contrast(question_index=7, seed=1234, lambda_audio=0.1, margin=0.0, num_negative_sets=1)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_100/best_loss.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_8/best_loss.pt
文本保存Epoch： 12
语音保存Epoch： 12
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始定向Audio训练：Seed=100，Q8，PHQ_8Moving
Lambda=0.1，Margin=0.0，Negative Sets=1
Text固定：True，Label固定：True，只打乱Audio：True
类别数量： [128, 20, 9, 6]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_paperlike/label_aware_deterministic_audio_contrast/lambda_0p1_margin_0_neg_1/seed_100/question_8
Epoch 01/20 | Train Total 1.854322 | Contrast 0.619196 | Train Gap 0.053925 | Val ImbOLL 2.648616 | Val Gap -0.007759 | Val Pair 0.581818 | Val CCC -0.031154 | 6.26s
Epoch 02/20 | Train Total 1.484695 | Contrast 0.599608 | Train Gap 0.035946 | Val ImbOLL 3.442812 | Val Gap -0.002813 | Val Pair 0.563636 | Val CCC 0.161482 | 5.66s
Epoch 03/20 | Train Total 1.609550 | Contrast 0.637859 | Train Gap 0.000109 | Val ImbOLL 2.903889 | Val Gap 0.002320

In [266]:
q8_seed100_deterministic_history = pd.read_csv(q8_seed100_deterministic_audio_result["history_path"])
q8_seed1234_deterministic_history = pd.read_csv(q8_seed1234_deterministic_audio_result["history_path"])
q8_seed100_baseline_history = pd.read_csv(FUSION_CHECKPOINT_ROOT / "seed_100" / "question_8" / "history.csv")
q8_seed1234_baseline_history = pd.read_csv(FUSION_CHECKPOINT_ROOT / "seed_1234" / "question_8" / "history.csv")
q8_deterministic_comparison = pd.DataFrame({"Seed": [42, 100, 1234], "Baseline_CCC": [q8_seed42_baseline_history["Val_CCC"].max(), q8_seed100_baseline_history["Val_CCC"].max(), q8_seed1234_baseline_history["Val_CCC"].max()], "Deterministic_Audio_CCC": [q8_seed42_deterministic_history["Val_CCC"].max(), q8_seed100_deterministic_history["Val_CCC"].max(), q8_seed1234_deterministic_history["Val_CCC"].max()]})
q8_deterministic_comparison["CCC_Change"] = q8_deterministic_comparison["Deterministic_Audio_CCC"] - q8_deterministic_comparison["Baseline_CCC"]
display(q8_deterministic_comparison)
print("基线Mean ± Std：", q8_deterministic_comparison["Baseline_CCC"].mean(), q8_deterministic_comparison["Baseline_CCC"].std(ddof=1))
print("定向Audio Mean ± Std：", q8_deterministic_comparison["Deterministic_Audio_CCC"].mean(), q8_deterministic_comparison["Deterministic_Audio_CCC"].std(ddof=1))
print("平均配对变化：", q8_deterministic_comparison["CCC_Change"].mean())

,Seed,Baseline_CCC,Deterministic_Audio_CCC,CCC_Change
0,42,0.344404,0.359847,0.015443
1,100,0.296571,0.278332,-0.018239
2,1234,0.111695,0.114211,0.002516


基线Mean ± Std： 0.2508901705344518 0.12289596307617623
定向Audio Mean ± Std： 0.25079659372568125 0.12511135255730063
平均配对变化： -9.357680877052539e-05


In [267]:
for seed in [42, 100, 1234]:
    text_path = find_checkpoint_by_metadata(TEXT_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=7, expected_checkpoint_type="best_loss")
    audio_path = find_checkpoint_by_metadata(AUDIO_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=7, expected_checkpoint_type="best_loss")
    text_checkpoint = torch.load(text_path, map_location="cpu", weights_only=False)
    audio_checkpoint = torch.load(audio_path, map_location="cpu", weights_only=False)
    print({"Seed": seed, "Text_Best_CCC": text_checkpoint.get("best_val_ccc"), "Text_Best_Loss": text_checkpoint.get("best_val_loss"), "Audio_Best_CCC": audio_checkpoint.get("best_val_ccc"), "Audio_Best_Loss": audio_checkpoint.get("best_val_loss"), "Text_Path": str(text_path), "Audio_Path": str(audio_path)})

{'Seed': 42, 'Text_Best_CCC': 0.2790893316268921, 'Text_Best_Loss': 1.678180369463834, 'Audio_Best_CCC': 0.0, 'Audio_Best_Loss': 1.7814135551452637, 'Text_Path': '/workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_loss.pt', 'Audio_Path': '/workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_loss.pt'}
{'Seed': 100, 'Text_Best_CCC': 0.2043190598487854, 'Text_Best_Loss': 1.7342357202009722, 'Audio_Best_CCC': 0.09952832758426666, 'Audio_Best_Loss': 1.7781342701478438, 'Text_Path': '/workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_100/best_loss.pt', 'Audio_Path': '/workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_8/best_loss.pt'}
{'Seed': 1234, 'Text_Best_CCC': 0.0, 'Text_Best_Loss': 1.7919207486239346, 'Audio_Best_CCC': 0.11169521510601044, 'Audio_Best_Loss': 1.7641272328116677, 'Text_Path': '/workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_1234/best_loss.pt', 'Audio_Path': '/works

In [268]:
q8_seed1234_baseline_checkpoint = torch.load(FUSION_CHECKPOINT_ROOT / "seed_1234" / "question_8" / "best_ccc.pt", map_location="cpu", weights_only=False)
print(q8_seed1234_baseline_checkpoint.keys())
print(q8_seed1234_baseline_checkpoint.get("val_metrics", {}).get("confusion_matrix"))
print(torch.bincount(q8_seed1234_baseline_checkpoint.get("val_metrics", {}).get("predictions").long(), minlength=NUM_CLASSES))

dict_keys(['checkpoint_type', 'epoch', 'question_index', 'question_number', 'question_name', 'seed', 'model_state_dict', 'optimizer_state_dict', 'train_metrics', 'val_metrics', 'class_counts', 'imboll_weights', 'best_val_loss', 'best_val_ccc', 'best_loss_epoch', 'best_ccc_epoch', 'text_checkpoint_path', 'audio_checkpoint_path', 'config'])
[[34, 5, 0, 0], [10, 2, 0, 0], [3, 0, 0, 0], [0, 1, 0, 0]]


AttributeError: 'NoneType' object has no attribute 'long'

In [269]:
for seed in [42, 100, 1234]:
    for checkpoint_type in ["best_loss", "best_ccc"]:
        text_path = find_checkpoint_by_metadata(TEXT_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=7, expected_checkpoint_type=checkpoint_type)
        audio_path = find_checkpoint_by_metadata(AUDIO_CHECKPOINT_ROOT, expected_seed=seed, expected_question_index=7, expected_checkpoint_type=checkpoint_type)
        text_checkpoint = torch.load(text_path, map_location="cpu", weights_only=False)
        audio_checkpoint = torch.load(audio_path, map_location="cpu", weights_only=False)
        print({"Seed": seed, "Type": checkpoint_type, "Text_Current_CCC": text_checkpoint.get("val_metrics", {}).get("ccc"), "Audio_Current_CCC": audio_checkpoint.get("val_metrics", {}).get("ccc"), "Text_Epoch": text_checkpoint.get("epoch"), "Audio_Epoch": audio_checkpoint.get("epoch")})

{'Seed': 42, 'Type': 'best_loss', 'Text_Current_CCC': 0.2790893316268921, 'Audio_Current_CCC': 0.10121756792068481, 'Text_Epoch': 11, 'Audio_Epoch': 12}
{'Seed': 42, 'Type': 'best_ccc', 'Text_Current_CCC': 0.33740898966789246, 'Audio_Current_CCC': 0.1263793408870697, 'Text_Epoch': 12, 'Audio_Epoch': 44}
{'Seed': 100, 'Type': 'best_loss', 'Text_Current_CCC': 0.2043190598487854, 'Audio_Current_CCC': 0.09117738902568817, 'Text_Epoch': 12, 'Audio_Epoch': 12}
{'Seed': 100, 'Type': 'best_ccc', 'Text_Current_CCC': 0.25735971331596375, 'Audio_Current_CCC': 0.17916281521320343, 'Text_Epoch': 14, 'Audio_Epoch': 20}
{'Seed': 1234, 'Type': 'best_loss', 'Text_Current_CCC': 0.0, 'Audio_Current_CCC': 0.023355869576334953, 'Text_Epoch': 1, 'Audio_Epoch': 21}
{'Seed': 1234, 'Type': 'best_ccc', 'Text_Current_CCC': 0.2839356064796448, 'Audio_Current_CCC': 0.12952688336372375, 'Text_Epoch': 13, 'Audio_Epoch': 30}


In [271]:
for question_index, question_name in enumerate(PHQ8_QUESTION_COLUMNS):
    train_question_labels = torch.as_tensor(train_fusion_dataset.labels[:, question_index]).long()
    val_question_labels = torch.as_tensor(val_fusion_dataset.labels[:, question_index]).long()
    train_question_counts = torch.bincount(train_question_labels, minlength=NUM_CLASSES)
    val_question_counts = torch.bincount(val_question_labels, minlength=NUM_CLASSES)
    print({"Question": question_index + 1, "Name": question_name, "Train_Counts": train_question_counts.tolist(), "Val_Counts": val_question_counts.tolist()})

{'Question': 1, 'Name': 'PHQ_8NoInterest', 'Train_Counts': [81, 57, 20, 5], 'Val_Counts': [25, 22, 4, 4]}
{'Question': 2, 'Name': 'PHQ_8Depressed', 'Train_Counts': [68, 63, 18, 14], 'Val_Counts': [25, 19, 7, 4]}
{'Question': 3, 'Name': 'PHQ_8Sleep', 'Train_Counts': [69, 40, 27, 27], 'Val_Counts': [22, 17, 8, 8]}
{'Question': 4, 'Name': 'PHQ_8Tired', 'Train_Counts': [50, 65, 27, 21], 'Val_Counts': [16, 23, 9, 7]}
{'Question': 5, 'Name': 'PHQ_8Appetite', 'Train_Counts': [77, 42, 24, 20], 'Val_Counts': [19, 22, 10, 4]}
{'Question': 6, 'Name': 'PHQ_8Failure', 'Train_Counts': [76, 44, 24, 19], 'Val_Counts': [25, 17, 8, 5]}
{'Question': 7, 'Name': 'PHQ_8Concentrating', 'Train_Counts': [93, 37, 14, 19], 'Val_Counts': [31, 13, 8, 3]}
{'Question': 8, 'Name': 'PHQ_8Moving', 'Train_Counts': [128, 20, 9, 6], 'Val_Counts': [39, 12, 3, 1]}


In [273]:
FUSION_BEST_CCC_ENCODER_CHECKPOINT_ROOT = FUSION_CHECKPOINT_ROOT.parent / f"{FUSION_CHECKPOINT_ROOT.name}_best_ccc_encoders"

In [274]:
def create_fusion_best_ccc_encoder_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch):
    checkpoint = create_fusion_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
    checkpoint["experiment_type"] = "text_audio_best_ccc_encoders"
    checkpoint["encoder_checkpoint_type"] = "best_ccc"
    return checkpoint

In [275]:
def train_fusion_question_best_ccc_encoders(question_index, seed):
    set_seed(seed)
    question_number = question_index + 1
    question_name = PHQ8_QUESTION_COLUMNS[question_index]
    output_dir = FUSION_BEST_CCC_ENCODER_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_number}"
    output_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = output_dir / "best_loss.pt"
    best_ccc_path = output_dir / "best_ccc.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.csv"
    train_loader, val_loader = create_fusion_dataloaders(seed)
    train_labels = train_fusion_dataset.labels[:, question_index]
    class_counts, inverse_frequency_weights, imboll_weights = calculate_imboll_weights(train_labels, BETA)
    imboll_weights_device = imboll_weights.to(device)
    text_encoder, audio_encoder, text_checkpoint, audio_checkpoint, text_checkpoint_path, audio_checkpoint_path = load_pretrained_text_audio_encoders(question_index=question_index, seed=seed, checkpoint_type="best_ccc")
    model = TextAudioQuestMF(text_encoder=text_encoder, audio_encoder=audio_encoder).to(device)
    optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = None
    best_ccc_epoch = None
    history_rows = []
    print("=" * 100)
    print(f"开始训练Best-CCC编码器版本：Seed={seed}，Q{question_number}，{question_name}")
    print("编码器检查点类型：best_ccc")
    print("文本检查点：", text_checkpoint_path)
    print("语音检查点：", audio_checkpoint_path)
    print("类别数量：", class_counts.tolist())
    print("=" * 100)
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.time()
        train_metrics = run_fusion_epoch(model, train_loader, question_index, imboll_weights_device, optimizer)
        val_metrics = run_fusion_epoch(model, val_loader, question_index, imboll_weights_device, optimizer=None)
        epoch_seconds = time.time() - epoch_start_time
        history_row = {"Epoch": epoch, "Train_Loss": train_metrics["loss"], "Train_Accuracy": train_metrics["accuracy"], "Train_CCC": train_metrics["ccc"], "Train_RMSE": train_metrics["rmse"], "Train_MAE": train_metrics["mae"], "Val_Loss": val_metrics["loss"], "Val_Accuracy": val_metrics["accuracy"], "Val_Micro_F1": val_metrics["micro_f1"], "Val_Macro_F1": val_metrics["macro_f1"], "Val_Weighted_F1": val_metrics["weighted_f1"], "Val_CCC": val_metrics["ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Seconds": epoch_seconds}
        history_rows.append(history_row)
        if not np.isfinite(val_metrics["loss"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限验证损失")
        if not np.isfinite(val_metrics["ccc"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限CCC")
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_loss_epoch = epoch
            best_loss_checkpoint = create_fusion_best_ccc_encoder_checkpoint(model, optimizer, "best_loss", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
            torch.save(best_loss_checkpoint, best_loss_path)
        if val_metrics["ccc"] > best_val_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
            best_ccc_checkpoint = create_fusion_best_ccc_encoder_checkpoint(model, optimizer, "best_ccc", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
            torch.save(best_ccc_checkpoint, best_ccc_path)
        last_checkpoint = create_fusion_best_ccc_encoder_checkpoint(model, optimizer, "last", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
        torch.save(last_checkpoint, last_path)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_metrics['loss']:.6f} | Val Loss {val_metrics['loss']:.6f} | Val CCC {val_metrics['ccc']:.6f} | Val Acc {val_metrics['accuracy']:.6f} | Val RMSE {val_metrics['rmse']:.6f} | {epoch_seconds:.2f}s")
    result = {"experiment_type": "text_audio_best_ccc_encoders", "encoder_checkpoint_type": "best_ccc", "seed": seed, "question_index": question_index, "question_number": question_number, "question_name": question_name, "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}
    del model
    del optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("=" * 100)
    print(f"Best-CCC编码器版本训练完成：Seed={seed}，Q{question_number}")
    print(f"最佳Val Loss：{best_val_loss:.6f}，Epoch={best_loss_epoch}")
    print(f"最佳Val CCC：{best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    print("=" * 100)
    return result

In [276]:
q8_seed42_best_ccc_encoder_result = train_fusion_question_best_ccc_encoders(question_index=7, seed=42)
q8_seed100_best_ccc_encoder_result = train_fusion_question_best_ccc_encoders(question_index=7, seed=100)
q8_seed1234_best_ccc_encoder_result = train_fusion_question_best_ccc_encoders(question_index=7, seed=1234)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_ccc.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_ccc.pt
文本保存Epoch： 12
语音保存Epoch： 44
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始训练Best-CCC编码器版本：Seed=42，Q8，PHQ_8Moving
编码器检查点类型：best_ccc
文本检查点： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_ccc.pt
语音检查点： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_ccc.pt
类别数量： [128, 20, 9, 6]
Epoch 01/20 | Train Loss 2.746542 | Val Loss 1.990569 | Val CCC 0.134101 | Val Acc 0.690909 | Val RMSE 0.726135 | 1.50s
Epoch 02/20 | Train Loss 1.290186 | Val Loss 2.848801 | Val CCC 0.356175 | Val Acc 0.618182 | Val RMSE 0.774597 | 1.00s
Epoch 03/20 | Train Loss 1.436498 | Val Loss 2.169817 | Val CCC 0.275278 | Val Acc 0.563636 | Val RMSE 0.700649 | 1.46s
Epoch 04/20 | Train Loss 1.138206 | Val Loss 2.363530 | Val CCC 0.380641 | Val Acc 0.563636 | Val RMSE 0.738549 |

In [277]:
q8_encoder_checkpoint_comparison = pd.DataFrame({"Seed": [42, 100, 1234], "Best_Loss_Encoder_CCC": [0.3444039821624756, 0.2965708374977112, 0.11169521510601044], "Best_CCC_Encoder_CCC": [q8_seed42_best_ccc_encoder_result["best_val_ccc"], q8_seed100_best_ccc_encoder_result["best_val_ccc"], q8_seed1234_best_ccc_encoder_result["best_val_ccc"]]})
q8_encoder_checkpoint_comparison["CCC_Change"] = q8_encoder_checkpoint_comparison["Best_CCC_Encoder_CCC"] - q8_encoder_checkpoint_comparison["Best_Loss_Encoder_CCC"]
display(q8_encoder_checkpoint_comparison)
print("原始Mean ± Std：", q8_encoder_checkpoint_comparison["Best_Loss_Encoder_CCC"].mean(), q8_encoder_checkpoint_comparison["Best_Loss_Encoder_CCC"].std(ddof=1))
print("Best-CCC编码器Mean ± Std：", q8_encoder_checkpoint_comparison["Best_CCC_Encoder_CCC"].mean(), q8_encoder_checkpoint_comparison["Best_CCC_Encoder_CCC"].std(ddof=1))

,Seed,Best_Loss_Encoder_CCC,Best_CCC_Encoder_CCC,CCC_Change
0,42,0.344404,0.388024,0.043620
1,100,0.296571,0.314689,0.018118
2,1234,0.111695,0.291190,0.179494


原始Mean ± Std： 0.2508900115887324 0.12289587445488095
Best-CCC编码器Mean ± Std： 0.3313007056713104 0.05050935662022826


In [278]:
AUDIO_CONTRAST_CHECKPOINT_ROOT = FUSION_CHECKPOINT_ROOT.parent / "phq8_text_audio_label_aware_deterministic_best_ccc_encoders"

In [281]:
q8_seed42_final_audio_result = train_fusion_question_label_aware_deterministic_audio_contrast(question_index=7, seed=42, lambda_audio=0.1, margin=0.0, num_negative_sets=1, encoder_checkpoint_type="best_ccc")
q8_seed100_final_audio_result = train_fusion_question_label_aware_deterministic_audio_contrast(question_index=7, seed=100, lambda_audio=0.1, margin=0.0, num_negative_sets=1, encoder_checkpoint_type="best_ccc")
q8_seed1234_final_audio_result = train_fusion_question_label_aware_deterministic_audio_contrast(question_index=7, seed=1234, lambda_audio=0.1, margin=0.0, num_negative_sets=1, encoder_checkpoint_type="best_ccc")

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_ccc.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_ccc.pt
文本保存Epoch： 12
语音保存Epoch： 44
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始定向Audio训练：Seed=42，Q8，PHQ_8Moving
Lambda=0.1，Margin=0.0，Negative Sets=1
Text固定：True，Label固定：True，只打乱Audio：True
类别数量： [128, 20, 9, 6]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_label_aware_deterministic_best_ccc_encoders/lambda_0p1_margin_0_neg_1/seed_42/question_8
Epoch 01/20 | Train Total 3.096949 | Contrast 0.602836 | Train Gap 0.193233 | Val ImbOLL 2.164485 | Val Gap -0.020220 | Val Pair 0.472727 | Val CCC 0.074137 | 6.19s
Epoch 02/20 | Train Total 1.403739 | Contrast 0.619338 | Train Gap 0.010127 | Val ImbOLL 3.491480 | Val Gap -0.000128 | Val Pair 0.400000 | Val CCC 0.405585 | 5.53s
Epoch 03/20 | Train Total 1.683149 | Contrast 0.650435 | Train Gap 0.000719 | Val ImbOLL 2.484164 | Val Gap 0.005195 | Val Pair 0.

In [283]:
FINAL_SEEDS = [42, 100, 1234]
FINAL_QUESTION_INDICES = list(range(8))
FINAL_AUDIO_CONTRAST_LAMBDA = 0.1
FINAL_AUDIO_CONTRAST_MARGIN = 0.0
FINAL_AUDIO_CONTRAST_NEGATIVE_SETS = 1
FUSION_BEST_CCC_ENCODER_CHECKPOINT_ROOT = FUSION_CHECKPOINT_ROOT.parent / f"{FUSION_CHECKPOINT_ROOT.name}_best_ccc_encoders"
AUDIO_CONTRAST_CHECKPOINT_ROOT = FUSION_CHECKPOINT_ROOT.parent / "phq8_text_audio_label_aware_deterministic_best_ccc_encoders"

In [284]:
def load_completed_question_result(output_dir, question_index, seed, experiment_type):
    output_dir = Path(output_dir)
    history_path = output_dir / "history.csv"
    best_loss_path = output_dir / "best_loss.pt"
    best_ccc_path = output_dir / "best_ccc.pt"
    last_path = output_dir / "last.pt"
    if not history_path.exists():
        return None
    history = pd.read_csv(history_path)
    if len(history) < NUM_EPOCHS:
        return None
    if not best_loss_path.exists() or not best_ccc_path.exists() or not last_path.exists():
        return None
    best_loss_row_index = history["Val_Loss"].idxmin()
    best_ccc_row_index = history["Val_CCC"].idxmax()
    best_val_loss = float(history.loc[best_loss_row_index, "Val_Loss"])
    best_val_ccc = float(history.loc[best_ccc_row_index, "Val_CCC"])
    best_loss_epoch = int(history.loc[best_loss_row_index, "Epoch"])
    best_ccc_epoch = int(history.loc[best_ccc_row_index, "Epoch"])
    return {"experiment_type": experiment_type, "seed": seed, "question_index": question_index, "question_number": question_index + 1, "question_name": PHQ8_QUESTION_COLUMNS[question_index], "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}

In [285]:
def get_best_ccc_baseline_output_dir(question_index, seed):
    return FUSION_BEST_CCC_ENCODER_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}"

def get_directional_audio_output_dir(question_index, seed):
    lambda_tag = f"{FINAL_AUDIO_CONTRAST_LAMBDA:.6g}".replace("-", "m").replace(".", "p")
    margin_tag = f"{FINAL_AUDIO_CONTRAST_MARGIN:.6g}".replace("-", "m").replace(".", "p")
    return AUDIO_CONTRAST_CHECKPOINT_ROOT / f"lambda_{lambda_tag}_margin_{margin_tag}_neg_{FINAL_AUDIO_CONTRAST_NEGATIVE_SETS}" / f"seed_{seed}" / f"question_{question_index + 1}"

In [288]:
final_best_ccc_baseline_results = {}
final_directional_audio_results = {}

for question_index in FINAL_QUESTION_INDICES:
    for seed in FINAL_SEEDS:
        baseline_output_dir = get_best_ccc_baseline_output_dir(question_index, seed)
        baseline_result = load_completed_question_result(baseline_output_dir, question_index, seed, "text_audio_best_ccc_encoders")
        if baseline_result is None:
            print(f"开始训练基线：Seed={seed}，Q{question_index + 1}")
            baseline_result = train_fusion_question_best_ccc_encoders(question_index=question_index, seed=seed)
        else:
            print(f"跳过已完成基线：Seed={seed}，Q{question_index + 1}，CCC={baseline_result['best_val_ccc']:.6f}")
        final_best_ccc_baseline_results[(question_index, seed)] = baseline_result
        directional_output_dir = get_directional_audio_output_dir(question_index, seed)
        directional_result = load_completed_question_result(directional_output_dir, question_index, seed, "text_audio_label_aware_deterministic_audio_contrast_best_ccc_encoders")
        if directional_result is None:
            print(f"开始训练定向Audio：Seed={seed}，Q{question_index + 1}")
            directional_result = train_fusion_question_label_aware_deterministic_audio_contrast(question_index=question_index, seed=seed, lambda_audio=FINAL_AUDIO_CONTRAST_LAMBDA, margin=FINAL_AUDIO_CONTRAST_MARGIN, num_negative_sets=FINAL_AUDIO_CONTRAST_NEGATIVE_SETS, encoder_checkpoint_type="best_ccc")
        else:
            print(f"跳过已完成定向Audio：Seed={seed}，Q{question_index + 1}，CCC={directional_result['best_val_ccc']:.6f}")
        final_directional_audio_results[(question_index, seed)] = directional_result
        print(f"完成配对：Seed={seed}，Q{question_index + 1}，Baseline={baseline_result['best_val_ccc']:.6f}，Directional={directional_result['best_val_ccc']:.6f}，Change={directional_result['best_val_ccc'] - baseline_result['best_val_ccc']:.6f}")

跳过已完成基线：Seed=42，Q1，CCC=0.461966
开始训练定向Audio：Seed=42，Q1
文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_ccc.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_ccc.pt
文本保存Epoch： 9
语音保存Epoch： 14
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始定向Audio训练：Seed=42，Q1，PHQ_8NoInterest
Lambda=0.1，Margin=0.0，Negative Sets=1
Text固定：True，Label固定：True，只打乱Audio：True
类别数量： [81, 57, 20, 5]
输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_label_aware_deterministic_best_ccc_encoders/lambda_0p1_margin_0_neg_1/seed_42/question_1
Epoch 01/20 | Train Total 2.139447 | Contrast 0.658275 | Train Gap 0.062142 | Val ImbOLL 2.367797 | Val Gap -0.051313 | Val Pair 0.545455 | Val CCC 0.387652 | 5.98s
Epoch 02/20 | Train Total 1.602852 | Contrast 0.692744 | Train Gap 0.008210 | Val ImbOLL 2.953332 | Val Gap -0.030663 | Val Pair 0.400000 | Val CCC 0.381247 | 5.80s
Epoch 03/20 | Train Total 1.496962 | Contrast 0.686312 | Train Gap 0

In [289]:
paired_question_rows = []

for question_index in FINAL_QUESTION_INDICES:
    for seed in FINAL_SEEDS:
        baseline_result = final_best_ccc_baseline_results[(question_index, seed)]
        directional_result = final_directional_audio_results[(question_index, seed)]
        paired_question_rows.append({"Question_Index": question_index, "Question_Number": question_index + 1, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Seed": seed, "Baseline_CCC": baseline_result["best_val_ccc"], "Directional_Audio_CCC": directional_result["best_val_ccc"], "CCC_Change": directional_result["best_val_ccc"] - baseline_result["best_val_ccc"], "Baseline_Best_Epoch": baseline_result["best_ccc_epoch"], "Directional_Best_Epoch": directional_result["best_ccc_epoch"]})

paired_question_results = pd.DataFrame(paired_question_rows)
display(paired_question_results)

,Question_Index,Question_Number,Question_Name,Seed,Baseline_CCC,Directional_Audio_CCC,CCC_Change,Baseline_Best_Epoch,Directional_Best_Epoch
0,0,1,PHQ_8NoInterest,42,0.461966,0.481013,1.904786e-02,6,14
1,0,1,PHQ_8NoInterest,100,0.406229,0.395314,-1.091495e-02,16,15
2,0,1,PHQ_8NoInterest,1234,0.447762,0.444196,-3.565490e-03,1,5
3,1,2,PHQ_8Depressed,42,0.515744,0.528385,1.264054e-02,5,2
4,1,2,PHQ_8Depressed,100,0.492937,0.502262,9.325296e-03,4,14
5,1,2,PHQ_8Depressed,1234,0.513291,0.531648,1.835692e-02,5,11
6,2,3,PHQ_8Sleep,42,0.500697,0.500697,-5.960464e-08,7,15
7,2,3,PHQ_8Sleep,100,0.434638,0.446400,1.176217e-02,7,13
8,2,3,PHQ_8Sleep,1234,0.399652,0.418473,1.882121e-02,13,16
9,3,4,PHQ_8Tired,42,0.503766,0.436482,-6.728438e-02,15,19


In [290]:
paired_question_summary = paired_question_results.groupby(["Question_Index", "Question_Number", "Question_Name"], as_index=False).agg(Baseline_Mean_CCC=("Baseline_CCC", "mean"), Baseline_Std_CCC=("Baseline_CCC", "std"), Directional_Mean_CCC=("Directional_Audio_CCC", "mean"), Directional_Std_CCC=("Directional_Audio_CCC", "std"), Mean_CCC_Change=("CCC_Change", "mean"), Improved_Seeds=("CCC_Change", lambda values: int((values > 0).sum())), Unchanged_Seeds=("CCC_Change", lambda values: int(np.isclose(values, 0.0).sum())), Decreased_Seeds=("CCC_Change", lambda values: int((values < 0).sum())))
display(paired_question_summary)

,Question_Index,Question_Number,Question_Name,Baseline_Mean_CCC,Baseline_Std_CCC,Directional_Mean_CCC,Directional_Std_CCC,Mean_CCC_Change,Improved_Seeds,Unchanged_Seeds,Decreased_Seeds
0,0,1,PHQ_8NoInterest,0.438652,0.028963,0.440175,0.042991,0.001522,1,0,2
1,1,2,PHQ_8Depressed,0.507324,0.012520,0.520765,0.016107,0.013441,3,0,0
2,2,3,PHQ_8Sleep,0.444996,0.051313,0.455190,0.041811,0.010194,2,0,1
3,3,4,PHQ_8Tired,0.527444,0.068355,0.489923,0.059904,-0.037520,1,0,2
4,4,5,PHQ_8Appetite,0.531853,0.051839,0.525050,0.048971,-0.006802,1,0,2
5,5,6,PHQ_8Failure,0.516003,0.042060,0.499347,0.064129,-0.016656,1,0,2
6,6,7,PHQ_8Concentrating,0.543718,0.012347,0.539520,0.005462,-0.004199,2,0,1
7,7,8,PHQ_8Moving,0.331301,0.050509,0.362273,0.058598,0.030972,2,1,0


In [291]:
print("24组配对中的平均CCC变化：", paired_question_results["CCC_Change"].mean())
print("提高的Question-Seed数量：", int((paired_question_results["CCC_Change"] > 0).sum()))
print("持平的Question-Seed数量：", int(np.isclose(paired_question_results["CCC_Change"], 0.0).sum()))
print("下降的Question-Seed数量：", int((paired_question_results["CCC_Change"] < 0).sum()))
print("平均基线CCC：", paired_question_results["Baseline_CCC"].mean())
print("平均定向Audio CCC：", paired_question_results["Directional_Audio_CCC"].mean())

24组配对中的平均CCC变化： -0.001130929837624232
提高的Question-Seed数量： 13
持平的Question-Seed数量： 1
下降的Question-Seed数量： 10
平均基线CCC： 0.4801612322529157
平均定向Audio CCC： 0.4790303024152915


In [292]:
AUDIO_CONTRAST_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
paired_question_results.to_csv(AUDIO_CONTRAST_CHECKPOINT_ROOT / "paired_question_seed_results.csv", index=False)
paired_question_summary.to_csv(AUDIO_CONTRAST_CHECKPOINT_ROOT / "paired_question_summary.csv", index=False)

In [293]:
AUDIO_AUX_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_fusion_audio_aux_best_ccc")
AUDIO_AUX_LAMBDA = 0.1

class TextAudioQuestMFAudioAux(nn.Module):
    def __init__(self, text_encoder, audio_encoder):
        super().__init__()
        self.text_encoder = text_encoder
        self.audio_encoder = audio_encoder
        self.audio_to_text_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_to_audio_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.audio_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Dropout(MLP_FIRST_DROPOUT), nn.Linear(MAX_TURNS * ENCODER_OUTPUT_DIM * 2, 256), nn.ReLU(), nn.Dropout(MLP_LAST_DROPOUT), nn.Linear(256, NUM_CLASSES))
        self.audio_aux_head = nn.Sequential(nn.LayerNorm(ENCODER_OUTPUT_DIM), nn.Dropout(FUSION_DROPOUT), nn.Linear(ENCODER_OUTPUT_DIM, NUM_CLASSES))
        for parameter in self.text_encoder.parameters():
            parameter.requires_grad = False
        self.text_encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()
        return self

    def forward(self, text_features, text_padding_mask, audio_features, audio_padding_mask, return_intermediates=False):
        text_encoding = self.text_encoder(text_features, text_padding_mask)
        audio_encoding = self.audio_encoder(audio_features, audio_padding_mask)
        text_encoding = text_encoding.masked_fill(text_padding_mask.unsqueeze(-1), 0.0)
        audio_encoding = audio_encoding.masked_fill(audio_padding_mask.unsqueeze(-1), 0.0)
        audio_valid_mask = (~audio_padding_mask).unsqueeze(-1).to(audio_encoding.dtype)
        audio_sum = torch.sum(audio_encoding * audio_valid_mask, dim=1)
        audio_count = torch.sum(audio_valid_mask, dim=1).clamp_min(1.0)
        audio_pooled = audio_sum / audio_count
        audio_aux_logits = self.audio_aux_head(audio_pooled)
        audio_to_text, _ = self.audio_to_text_cross_attention(text_encoding, audio_encoding, audio_encoding, key_padding_mask=audio_padding_mask, need_weights=False)
        text_to_audio, _ = self.text_to_audio_cross_attention(audio_encoding, text_encoding, text_encoding, key_padding_mask=text_padding_mask, need_weights=False)
        text_fused_encoding, _ = self.text_self_attention(audio_to_text, audio_to_text, audio_to_text, key_padding_mask=text_padding_mask, need_weights=False)
        audio_fused_encoding, _ = self.audio_self_attention(text_to_audio, text_to_audio, text_to_audio, key_padding_mask=audio_padding_mask, need_weights=False)
        fused_encoding = torch.cat((text_fused_encoding, audio_fused_encoding), dim=2)
        fusion_logits = self.mlp(fused_encoding)
        if return_intermediates:
            intermediates = {"text_encoding": text_encoding, "audio_encoding": audio_encoding, "audio_pooled": audio_pooled, "audio_to_text": audio_to_text, "text_to_audio": text_to_audio, "text_fused_encoding": text_fused_encoding, "audio_fused_encoding": audio_fused_encoding, "fused_encoding": fused_encoding}
            return fusion_logits, audio_aux_logits, intermediates
        return fusion_logits, audio_aux_logits

In [294]:
def run_fusion_epoch_audio_aux(model, dataloader, question_index, class_weights, lambda_audio_aux, optimizer=None):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()
    total_combined_loss = 0.0
    total_fusion_loss = 0.0
    total_audio_aux_loss = 0.0
    total_samples = 0
    fusion_prediction_batches = []
    audio_aux_prediction_batches = []
    target_batches = []
    for batch in dataloader:
        text_features, text_masks, audio_features, audio_masks, all_labels = batch
        text_features = text_features.to(device)
        text_masks = text_masks.to(device)
        audio_features = audio_features.to(device)
        audio_masks = audio_masks.to(device)
        labels = all_labels[:, question_index].to(device)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            fusion_logits, audio_aux_logits = model(text_features, text_masks, audio_features, audio_masks)
            fusion_loss = calculate_imboll_loss(fusion_logits, labels, class_weights, ALPHA)
            audio_aux_loss = calculate_imboll_loss(audio_aux_logits, labels, class_weights, ALPHA)
            combined_loss = fusion_loss + lambda_audio_aux * audio_aux_loss
            if is_training:
                combined_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
        fusion_predictions = torch.argmax(fusion_logits, dim=1)
        audio_aux_predictions = torch.argmax(audio_aux_logits, dim=1)
        batch_size = labels.shape[0]
        total_combined_loss += combined_loss.item() * batch_size
        total_fusion_loss += fusion_loss.item() * batch_size
        total_audio_aux_loss += audio_aux_loss.item() * batch_size
        total_samples += batch_size
        fusion_prediction_batches.append(fusion_predictions.detach().cpu())
        audio_aux_prediction_batches.append(audio_aux_predictions.detach().cpu())
        target_batches.append(labels.detach().cpu())
    fusion_predictions = torch.cat(fusion_prediction_batches, dim=0)
    audio_aux_predictions = torch.cat(audio_aux_prediction_batches, dim=0)
    all_targets = torch.cat(target_batches, dim=0)
    fusion_metrics = calculate_multiclass_metrics(fusion_predictions, all_targets)
    audio_aux_metrics = calculate_multiclass_metrics(audio_aux_predictions, all_targets)
    fusion_ccc = calculate_sample_ccc(fusion_predictions.float(), all_targets.float()).item()
    audio_aux_ccc = calculate_sample_ccc(audio_aux_predictions.float(), all_targets.float()).item()
    fusion_rmse = torch.sqrt(torch.mean((fusion_predictions.float() - all_targets.float()) ** 2)).item()
    fusion_mae = torch.mean(torch.abs(fusion_predictions.float() - all_targets.float())).item()
    return {"loss": total_combined_loss / total_samples, "total_loss": total_combined_loss / total_samples, "fusion_imboll_loss": total_fusion_loss / total_samples, "audio_aux_imboll_loss": total_audio_aux_loss / total_samples, "accuracy": fusion_metrics["accuracy"], "micro_f1": fusion_metrics["micro_f1"], "macro_f1": fusion_metrics["macro_f1"], "weighted_f1": fusion_metrics["weighted_f1"], "ccc": fusion_ccc, "rmse": fusion_rmse, "mae": fusion_mae, "confusion_matrix": fusion_metrics["confusion_matrix"], "predictions": fusion_predictions, "targets": all_targets, "audio_aux_accuracy": audio_aux_metrics["accuracy"], "audio_aux_macro_f1": audio_aux_metrics["macro_f1"], "audio_aux_ccc": audio_aux_ccc, "audio_aux_confusion_matrix": audio_aux_metrics["confusion_matrix"], "audio_aux_predictions": audio_aux_predictions}

In [295]:
def calculate_gradient_group_norm(gradients):
    squared_norm = sum(gradient.detach().float().pow(2).sum().item() for gradient in gradients if gradient is not None)
    return squared_norm ** 0.5

def check_audio_aux_gradient_routing(model, dataloader, question_index, class_weights):
    model.train()
    batch = next(iter(dataloader))
    text_features, text_masks, audio_features, audio_masks, all_labels = batch
    text_features = text_features.to(device)
    text_masks = text_masks.to(device)
    audio_features = audio_features.to(device)
    audio_masks = audio_masks.to(device)
    labels = all_labels[:, question_index].to(device)
    fusion_logits, audio_aux_logits = model(text_features, text_masks, audio_features, audio_masks)
    audio_aux_loss = calculate_imboll_loss(audio_aux_logits, labels, class_weights, ALPHA)
    audio_parameters = [parameter for parameter in model.audio_encoder.parameters() if parameter.requires_grad]
    audio_aux_head_parameters = [parameter for parameter in model.audio_aux_head.parameters() if parameter.requires_grad]
    fusion_parameters = list(model.audio_to_text_cross_attention.parameters()) + list(model.text_to_audio_cross_attention.parameters()) + list(model.text_self_attention.parameters()) + list(model.audio_self_attention.parameters()) + list(model.mlp.parameters())
    tracked_parameters = audio_parameters + audio_aux_head_parameters + fusion_parameters
    gradients = torch.autograd.grad(audio_aux_loss, tracked_parameters, allow_unused=True)
    audio_end = len(audio_parameters)
    audio_aux_head_end = audio_end + len(audio_aux_head_parameters)
    audio_gradients = gradients[:audio_end]
    audio_aux_head_gradients = gradients[audio_end:audio_aux_head_end]
    fusion_gradients = gradients[audio_aux_head_end:]
    result = {"Audio_Encoder_Aux_Grad": calculate_gradient_group_norm(audio_gradients), "Audio_Aux_Head_Grad": calculate_gradient_group_norm(audio_aux_head_gradients), "Fusion_Blocks_Aux_Grad": calculate_gradient_group_norm(fusion_gradients), "Trainable_Text_Parameters": sum(parameter.numel() for parameter in model.text_encoder.parameters() if parameter.requires_grad)}
    model.zero_grad(set_to_none=True)
    print(result)
    return result

In [296]:
set_seed(42)
q8_class_counts, q8_inverse_frequency_weights, q8_imboll_weights = calculate_imboll_weights(train_fusion_dataset.labels[:, 7], BETA)
q8_imboll_weights_device = q8_imboll_weights.to(device)
q8_smoke_text_encoder, q8_smoke_audio_encoder, _, _, _, _ = load_pretrained_text_audio_encoders(question_index=7, seed=42, checkpoint_type="best_ccc")
q8_audio_aux_smoke_model = TextAudioQuestMFAudioAux(q8_smoke_text_encoder, q8_smoke_audio_encoder).to(device)
q8_smoke_train_loader, q8_smoke_val_loader = create_fusion_dataloaders(seed=42)
q8_audio_aux_gradient_result = check_audio_aux_gradient_routing(q8_audio_aux_smoke_model, q8_smoke_train_loader, question_index=7, class_weights=q8_imboll_weights_device)
del q8_audio_aux_smoke_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_ccc.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_ccc.pt
文本保存Epoch： 12
语音保存Epoch： 44
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
{'Audio_Encoder_Aux_Grad': 42.70122988274316, 'Audio_Aux_Head_Grad': 11.26226592148774, 'Fusion_Blocks_Aux_Grad': 0.0, 'Trainable_Text_Parameters': 0}


In [297]:
def create_fusion_audio_aux_checkpoint(model, optimizer, checkpoint_type, epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, lambda_audio_aux, best_val_fusion_loss, best_val_fusion_ccc, best_val_audio_aux_ccc, best_fusion_loss_epoch, best_fusion_ccc_epoch, best_audio_aux_ccc_epoch):
    checkpoint = {"experiment_type": "fusion_audio_auxiliary_imboll", "checkpoint_type": checkpoint_type, "epoch": epoch, "question_index": question_index, "question_number": question_index + 1, "question_name": PHQ8_QUESTION_COLUMNS[question_index], "seed": seed, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "train_metrics": train_metrics, "val_metrics": val_metrics, "class_counts": class_counts, "imboll_weights": imboll_weights, "text_checkpoint_path": str(text_checkpoint_path), "audio_checkpoint_path": str(audio_checkpoint_path), "encoder_checkpoint_type": "best_ccc", "lambda_audio_aux": lambda_audio_aux, "best_val_fusion_loss": best_val_fusion_loss, "best_val_fusion_ccc": best_val_fusion_ccc, "best_val_audio_aux_ccc": best_val_audio_aux_ccc, "best_fusion_loss_epoch": best_fusion_loss_epoch, "best_fusion_ccc_epoch": best_fusion_ccc_epoch, "best_audio_aux_ccc_epoch": best_audio_aux_ccc_epoch}
    return checkpoint

In [298]:
def train_fusion_question_audio_aux(question_index, seed, lambda_audio_aux=0.1):
    set_seed(seed)
    question_number = question_index + 1
    question_name = PHQ8_QUESTION_COLUMNS[question_index]
    output_dir = AUDIO_AUX_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_number}"
    output_dir.mkdir(parents=True, exist_ok=True)
    best_fusion_loss_path = output_dir / "best_fusion_imboll.pt"
    best_fusion_ccc_path = output_dir / "best_fusion_ccc.pt"
    best_audio_aux_ccc_path = output_dir / "best_audio_aux_ccc.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.csv"
    train_loader, val_loader = create_fusion_dataloaders(seed)
    train_labels = train_fusion_dataset.labels[:, question_index]
    class_counts, inverse_frequency_weights, imboll_weights = calculate_imboll_weights(train_labels, BETA)
    imboll_weights_device = imboll_weights.to(device)
    text_encoder, audio_encoder, _, _, text_checkpoint_path, audio_checkpoint_path = load_pretrained_text_audio_encoders(question_index=question_index, seed=seed, checkpoint_type="best_ccc")
    model = TextAudioQuestMFAudioAux(text_encoder=text_encoder, audio_encoder=audio_encoder).to(device)
    optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    best_val_fusion_loss = float("inf")
    best_val_fusion_ccc = float("-inf")
    best_val_audio_aux_ccc = float("-inf")
    best_fusion_loss_epoch = None
    best_fusion_ccc_epoch = None
    best_audio_aux_ccc_epoch = None
    history_rows = []
    print("=" * 100)
    print(f"开始Audio辅助监督训练：Seed={seed}，Q{question_number}，{question_name}")
    print(f"Lambda Audio Aux：{lambda_audio_aux}")
    print("单模态初始化：best_ccc")
    print("类别数量：", class_counts.tolist())
    print("=" * 100)
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.time()
        train_metrics = run_fusion_epoch_audio_aux(model, train_loader, question_index, imboll_weights_device, lambda_audio_aux, optimizer)
        val_metrics = run_fusion_epoch_audio_aux(model, val_loader, question_index, imboll_weights_device, lambda_audio_aux, optimizer=None)
        epoch_seconds = time.time() - epoch_start_time
        history_row = {"Epoch": epoch, "Train_Total_Loss": train_metrics["total_loss"], "Train_Fusion_ImbOLL": train_metrics["fusion_imboll_loss"], "Train_Audio_Aux_ImbOLL": train_metrics["audio_aux_imboll_loss"], "Train_Fusion_CCC": train_metrics["ccc"], "Train_Audio_Aux_CCC": train_metrics["audio_aux_ccc"], "Val_Total_Loss": val_metrics["total_loss"], "Val_Fusion_ImbOLL": val_metrics["fusion_imboll_loss"], "Val_Audio_Aux_ImbOLL": val_metrics["audio_aux_imboll_loss"], "Val_Fusion_Accuracy": val_metrics["accuracy"], "Val_Fusion_Macro_F1": val_metrics["macro_f1"], "Val_Fusion_CCC": val_metrics["ccc"], "Val_Audio_Aux_Accuracy": val_metrics["audio_aux_accuracy"], "Val_Audio_Aux_Macro_F1": val_metrics["audio_aux_macro_f1"], "Val_Audio_Aux_CCC": val_metrics["audio_aux_ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Seconds": epoch_seconds}
        history_rows.append(history_row)
        if not np.isfinite(val_metrics["fusion_imboll_loss"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限Fusion ImbOLL")
        if not np.isfinite(val_metrics["ccc"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限Fusion CCC")
        if not np.isfinite(val_metrics["audio_aux_ccc"]):
            raise RuntimeError(f"Seed={seed}，Q{question_number}在Epoch={epoch}出现非有限Audio Aux CCC")
        improved_fusion_loss = val_metrics["fusion_imboll_loss"] < best_val_fusion_loss
        improved_fusion_ccc = val_metrics["ccc"] > best_val_fusion_ccc
        improved_audio_aux_ccc = val_metrics["audio_aux_ccc"] > best_val_audio_aux_ccc
        if improved_fusion_loss:
            best_val_fusion_loss = val_metrics["fusion_imboll_loss"]
            best_fusion_loss_epoch = epoch
        if improved_fusion_ccc:
            best_val_fusion_ccc = val_metrics["ccc"]
            best_fusion_ccc_epoch = epoch
        if improved_audio_aux_ccc:
            best_val_audio_aux_ccc = val_metrics["audio_aux_ccc"]
            best_audio_aux_ccc_epoch = epoch
        if improved_fusion_loss:
            checkpoint = create_fusion_audio_aux_checkpoint(model, optimizer, "best_fusion_imboll", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, lambda_audio_aux, best_val_fusion_loss, best_val_fusion_ccc, best_val_audio_aux_ccc, best_fusion_loss_epoch, best_fusion_ccc_epoch, best_audio_aux_ccc_epoch)
            torch.save(checkpoint, best_fusion_loss_path)
        if improved_fusion_ccc:
            checkpoint = create_fusion_audio_aux_checkpoint(model, optimizer, "best_fusion_ccc", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, lambda_audio_aux, best_val_fusion_loss, best_val_fusion_ccc, best_val_audio_aux_ccc, best_fusion_loss_epoch, best_fusion_ccc_epoch, best_audio_aux_ccc_epoch)
            torch.save(checkpoint, best_fusion_ccc_path)
        if improved_audio_aux_ccc:
            checkpoint = create_fusion_audio_aux_checkpoint(model, optimizer, "best_audio_aux_ccc", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, lambda_audio_aux, best_val_fusion_loss, best_val_fusion_ccc, best_val_audio_aux_ccc, best_fusion_loss_epoch, best_fusion_ccc_epoch, best_audio_aux_ccc_epoch)
            torch.save(checkpoint, best_audio_aux_ccc_path)
        last_checkpoint = create_fusion_audio_aux_checkpoint(model, optimizer, "last", epoch, question_index, seed, train_metrics, val_metrics, class_counts, imboll_weights, text_checkpoint_path, audio_checkpoint_path, lambda_audio_aux, best_val_fusion_loss, best_val_fusion_ccc, best_val_audio_aux_ccc, best_fusion_loss_epoch, best_fusion_ccc_epoch, best_audio_aux_ccc_epoch)
        torch.save(last_checkpoint, last_path)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Fusion CCC {train_metrics['ccc']:.6f} | Train Audio CCC {train_metrics['audio_aux_ccc']:.6f} | Val Fusion ImbOLL {val_metrics['fusion_imboll_loss']:.6f} | Val Fusion CCC {val_metrics['ccc']:.6f} | Val Audio CCC {val_metrics['audio_aux_ccc']:.6f} | {epoch_seconds:.2f}s")
    result = {"seed": seed, "question_index": question_index, "question_number": question_number, "question_name": question_name, "lambda_audio_aux": lambda_audio_aux, "best_val_fusion_loss": best_val_fusion_loss, "best_val_fusion_ccc": best_val_fusion_ccc, "best_val_audio_aux_ccc": best_val_audio_aux_ccc, "best_fusion_loss_epoch": best_fusion_loss_epoch, "best_fusion_ccc_epoch": best_fusion_ccc_epoch, "best_audio_aux_ccc_epoch": best_audio_aux_ccc_epoch, "best_fusion_loss_path": best_fusion_loss_path, "best_fusion_ccc_path": best_fusion_ccc_path, "best_audio_aux_ccc_path": best_audio_aux_ccc_path, "last_path": last_path, "history_path": history_path}
    del model
    del optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("=" * 100)
    print(f"Audio辅助监督训练完成：Seed={seed}，Q{question_number}")
    print(f"最佳Val Fusion ImbOLL：{best_val_fusion_loss:.6f}，Epoch={best_fusion_loss_epoch}")
    print(f"最佳Val Fusion CCC：{best_val_fusion_ccc:.6f}，Epoch={best_fusion_ccc_epoch}")
    print(f"最佳Val Audio Aux CCC：{best_val_audio_aux_ccc:.6f}，Epoch={best_audio_aux_ccc_epoch}")
    print("=" * 100)
    return result

In [299]:
q8_seed42_audio_aux_result = train_fusion_question_audio_aux(question_index=7, seed=42, lambda_audio_aux=0.1)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_42/best_ccc.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_8/best_ccc.pt
文本保存Epoch： 12
语音保存Epoch： 44
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始Audio辅助监督训练：Seed=42，Q8，PHQ_8Moving
Lambda Audio Aux：0.1
单模态初始化：best_ccc
类别数量： [128, 20, 9, 6]
Epoch 01/20 | Train Fusion CCC 0.177927 | Train Audio CCC 0.019987 | Val Fusion ImbOLL 2.012310 | Val Fusion CCC 0.134101 | Val Audio CCC 0.142991 | 1.16s
Epoch 02/20 | Train Fusion CCC 0.575613 | Train Audio CCC 0.158087 | Val Fusion ImbOLL 2.701980 | Val Fusion CCC 0.355560 | Val Audio CCC 0.079207 | 0.98s
Epoch 03/20 | Train Fusion CCC 0.658504 | Train Audio CCC 0.178318 | Val Fusion ImbOLL 2.081773 | Val Fusion CCC 0.240340 | Val Audio CCC 0.146697 | 1.18s
Epoch 04/20 | Train Fusion CCC 0.645635 | Train Audio CCC 0.085594 | Val Fusion ImbOLL 2.353838 | Val Fusion CCC 0.344000 | Val Audio CCC 0.101218 | 1.23s
Epoch 05/20 | Train

In [300]:
q8_seed100_audio_aux_result = train_fusion_question_audio_aux(question_index=7, seed=100, lambda_audio_aux=0.1)
q8_seed1234_audio_aux_result = train_fusion_question_audio_aux(question_index=7, seed=1234, lambda_audio_aux=0.1)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q8_PHQ_8Moving/seed_100/best_ccc.pt
语音Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_100/question_8/best_ccc.pt
文本保存Epoch： 14
语音保存Epoch： 20
文本缺少参数： []
文本多余参数： []
语音缺少参数： []
语音多余参数： []
开始Audio辅助监督训练：Seed=100，Q8，PHQ_8Moving
Lambda Audio Aux：0.1
单模态初始化：best_ccc
类别数量： [128, 20, 9, 6]
Epoch 01/20 | Train Fusion CCC 0.377254 | Train Audio CCC -0.075053 | Val Fusion ImbOLL 2.757789 | Val Fusion CCC 0.180810 | Val Audio CCC 0.111695 | 1.63s
Epoch 02/20 | Train Fusion CCC 0.636810 | Train Audio CCC 0.007309 | Val Fusion ImbOLL 4.306805 | Val Fusion CCC 0.124419 | Val Audio CCC 0.076744 | 1.21s
Epoch 03/20 | Train Fusion CCC 0.655113 | Train Audio CCC -0.061999 | Val Fusion ImbOLL 3.157558 | Val Fusion CCC 0.198397 | Val Audio CCC 0.079207 | 1.42s
Epoch 04/20 | Train Fusion CCC 0.670604 | Train Audio CCC 0.095040 | Val Fusion ImbOLL 4.154633 | Val Fusion CCC 0.326618 | Val Audio CCC 0.079207 | 1.06s
Epoch 05/20 | 

In [302]:
q8_best_ccc_baselines = {42: 0.3880239725112915, 100: 0.3146892488002777, 1234: 0.29118987917900085}
q8_audio_aux_results = [q8_seed42_audio_aux_result, q8_seed100_audio_aux_result, q8_seed1234_audio_aux_result]
q8_audio_aux_comparison_rows = []
for result in q8_audio_aux_results:
    seed = result["seed"]
    baseline_ccc = q8_best_ccc_baselines[seed]
    audio_aux_fusion_ccc = result["best_val_fusion_ccc"]
    q8_audio_aux_comparison_rows.append({"Seed": seed, "Baseline_CCC": baseline_ccc, "Audio_Aux_Fusion_CCC": audio_aux_fusion_ccc, "CCC_Change": audio_aux_fusion_ccc - baseline_ccc, "Best_Audio_Aux_CCC": result["best_val_audio_aux_ccc"], "Fusion_Epoch": result["best_fusion_ccc_epoch"], "Audio_Aux_Epoch": result["best_audio_aux_ccc_epoch"]})
q8_audio_aux_comparison = pd.DataFrame(q8_audio_aux_comparison_rows)
display(q8_audio_aux_comparison)
print("平均基线CCC：", q8_audio_aux_comparison["Baseline_CCC"].mean())
print("平均Audio辅助Fusion CCC：", q8_audio_aux_comparison["Audio_Aux_Fusion_CCC"].mean())
print("平均CCC变化：", q8_audio_aux_comparison["CCC_Change"].mean())

,Seed,Baseline_CCC,Audio_Aux_Fusion_CCC,CCC_Change,Best_Audio_Aux_CCC,Fusion_Epoch,Audio_Aux_Epoch
0,42,0.388024,0.444857,0.056833,0.171727,12,15
1,100,0.314689,0.326618,0.011929,0.128254,4,11
2,1234,0.291190,0.295822,0.004632,0.179163,20,18


平均基线CCC： 0.3313010334968567
平均Audio辅助Fusion CCC： 0.35576552152633667
平均CCC变化： 0.02446448802947998
